In [ ]:
# Kaggle bootstrap: when only this notebook is uploaded, fetch config.py
# from the project repository without embedding any API keys in the notebook.
import os as _bootstrap_os
from pathlib import Path as _BootstrapPath
import subprocess as _bootstrap_subprocess
import sys as _bootstrap_sys

# Deterministic CUDA/cuBLAS settings must be defined before importing torch.
_bootstrap_os.environ.setdefault(
    "CUBLAS_WORKSPACE_CONFIG",
    ":16:8",
)
_bootstrap_os.environ.setdefault(
    "NVIDIA_TF32_OVERRIDE",
    "0",
)

_IS_KAGGLE_BOOTSTRAP = bool(
    _bootstrap_os.environ.get(
        "KAGGLE_KERNEL_RUN_TYPE"
    )
    or _BootstrapPath(
        "/kaggle/working"
    ).is_dir()
)

if (
    _IS_KAGGLE_BOOTSTRAP
    and not _BootstrapPath(
        "config.py"
    ).is_file()
):
    _project_dir = _BootstrapPath(
        "/kaggle/working/"
        "uav_search_network"
    )

    if not _project_dir.is_dir():
        _bootstrap_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                (
                    "https://github.com/"
                    "truongdaoanhduy/"
                    "uav_search_network.git"
                ),
                str(_project_dir),
            ],
            check=True,
        )

    _bootstrap_os.chdir(
        _project_dir
    )

    if str(_project_dir) not in (
        _bootstrap_sys.path
    ):
        _bootstrap_sys.path.insert(
            0,
            str(_project_dir),
        )


In [16]:
from abc import ABC, abstractmethod
import copy
import csv
import importlib
from itertools import pairwise
import json
import os
import secrets
from pathlib import Path
import sys
import subprocess
import tempfile
from typing import ClassVar

import gymnasium as gym
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from torch.distributions import Normal

from config import CONFIG


In [17]:
import random


def set_seed(seed=44):
    seed = int(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True
    )

    if hasattr(
        torch.backends,
        "cudnn",
    ):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

        if hasattr(
            torch.backends.cudnn,
            "allow_tf32",
        ):
            torch.backends.cudnn.allow_tf32 = False

    if (
        hasattr(
            torch.backends,
            "cuda",
        )
        and hasattr(
            torch.backends.cuda,
            "matmul",
        )
    ):
        torch.backends.cuda.matmul.allow_tf32 = False

set_seed(44)

In [ ]:
from dataclasses import dataclass


@dataclass
class UAV:
    id: int
    position: np.ndarray
    velocity: np.ndarray
    battery_j: float

    active: bool = True


@dataclass
class Target:
    id: int
    position: np.ndarray
    confirmed: bool = False


@dataclass
class Obstacle:
    position: np.ndarray
    radius: float
    height: float

    def __post_init__(self):
        self.position = np.asarray(
            self.position,
            dtype=np.float64,
        )
        self.radius = float(self.radius)
        self.height = float(self.height)

        if self.position.shape != (2,):
            raise ValueError(
                "Obstacle.position must be the XY center with shape (2,)"
            )

        if not np.all(np.isfinite(self.position)):
            raise ValueError(
                "Obstacle.position must contain only finite values"
            )

        if not np.isfinite(self.radius) or self.radius <= 0.0:
            raise ValueError(
                "Obstacle.radius must be finite and > 0"
            )

        if not np.isfinite(self.height) or self.height <= 0.0:
            raise ValueError(
                "Obstacle.height must be finite and > 0"
            )


@dataclass
class Report:
    target_id: int
    source_uav: int
    created_step: int
    size_bytes: int
    ttl_s: float
    delivered_bytes: int = 0

    def __post_init__(self):
        for name, value in (
            ("target_id", self.target_id),
            ("source_uav", self.source_uav),
            ("created_step", self.created_step),
            ("size_bytes", self.size_bytes),
            ("delivered_bytes", self.delivered_bytes),
        ):
            if (
                isinstance(value, (bool, np.bool_))
                or not isinstance(value, (int, np.integer))
            ):
                raise TypeError(
                    f"{name} must be an integer"
                )

        self.target_id = int(self.target_id)
        self.source_uav = int(self.source_uav)
        self.created_step = int(self.created_step)
        self.size_bytes = int(self.size_bytes)
        self.delivered_bytes = int(self.delivered_bytes)
        self.ttl_s = float(self.ttl_s)

        if self.target_id < 0:
            raise ValueError(
                "target_id must be >= 0"
            )

        if self.source_uav < 0:
            raise ValueError(
                "source_uav must be >= 0"
            )

        if self.created_step < 0:
            raise ValueError(
                "created_step must be >= 0"
            )

        if self.size_bytes <= 0:
            raise ValueError(
                "size_bytes must be > 0"
            )

        if (
            not np.isfinite(self.ttl_s)
            or self.ttl_s <= 0.0
        ):
            raise ValueError(
                "ttl_s must be finite and > 0"
            )

        if not (
            0
            <= self.delivered_bytes
            <= self.size_bytes
        ):
            raise ValueError(
                "delivered_bytes must be in "
                "[0, size_bytes]"
            )


In [19]:
def create_uav(rng):
    if not isinstance(rng, np.random.Generator):
        raise TypeError("rng must be numpy.random.Generator")

    num_uavs = int(CONFIG["num_uavs"])
    gcs = np.asarray(CONFIG["gcs_position"], dtype=np.float64)
    launch_radius = float(CONFIG["launch_radius_m"])
    safety_distance = float(CONFIG["safety_distance"])
    launch_min_spacing = float(CONFIG["launch_min_spacing_m"])
    map_size = float(CONFIG["map_size"])
    altitude = float(CONFIG["altitude_min"])

    if num_uavs < 1:
        raise ValueError("num_uavs must be >= 1")

    if gcs.shape != (3,) or not np.all(np.isfinite(gcs)):
        raise ValueError("gcs_position must be a finite 3D position")

    if not (
        0.0 <= gcs[0] <= map_size
        and 0.0 <= gcs[1] <= map_size
    ):
        raise ValueError("gcs_position must lie inside the map")

    if not np.isfinite(launch_radius) or launch_radius <= 0.0:
        raise ValueError("launch_radius_m must be finite and > 0")

    if not np.isfinite(safety_distance) or safety_distance < 0.0:
        raise ValueError("safety_distance must be finite and >= 0")

    if (
        not np.isfinite(launch_min_spacing)
        or launch_min_spacing < safety_distance
    ):
        raise ValueError(
            "launch_min_spacing_m must be finite and >= safety_distance"
        )

    if not np.isfinite(map_size) or map_size <= 0.0:
        raise ValueError("map_size must be finite and > 0")

    if not np.isfinite(altitude):
        raise ValueError("altitude_min must be finite")

    uavs = []
    max_attempts = max(10_000, 1_000 * num_uavs)
    attempts = 0
    launch_radius_sq = launch_radius * launch_radius

    while len(uavs) < num_uavs:
        attempts += 1

        if attempts > max_attempts:
            raise RuntimeError(
                "Could not place all UAVs inside the launch region while "
                "respecting map bounds and launch_min_spacing_m. "
                "Increase launch_radius_m or reduce num_uavs/launch_min_spacing_m."
            )

        offset_xy = rng.uniform(
            -launch_radius,
            launch_radius,
            size=2,
        ).astype(np.float64)

        if float(offset_xy @ offset_xy) > launch_radius_sq:
            continue

        candidate_xy = gcs[:2] + offset_xy

        if not (
            0.0 <= candidate_xy[0] <= map_size
            and 0.0 <= candidate_xy[1] <= map_size
        ):
            continue

        too_close = any(
            np.linalg.norm(candidate_xy - uav.position[:2])
            < launch_min_spacing
            for uav in uavs
        )

        if too_close:
            continue

        position = np.array(
            [candidate_xy[0], candidate_xy[1], altitude],
            dtype=np.float64,
        )

        uavs.append(
            UAV(
                id=len(uavs),
                position=position,
                velocity=np.zeros(3, dtype=np.float64),
                battery_j=CONFIG["battery_j"],
                active=True,
            )
        )

    return uavs


In [ ]:
def point_inside_obstacle(point, obstacles, margin=0.0):
    point = np.asarray(point, dtype=np.float64)
    margin = float(margin)

    if not np.isfinite(margin) or margin < 0.0:
        raise ValueError(
            "margin must be finite and >= 0"
        )

    if point.shape == (2,):
        point_xy = point
        z = 0.0
    elif point.shape == (3,):
        point_xy = point[:2]
        z = float(point[2])
    else:
        raise ValueError(
            "point must have shape (2,) or (3,)"
        )

    if not np.all(np.isfinite(point)):
        raise ValueError(
            "point must contain only finite values"
        )

    for obs in obstacles:
        horizontal_distance = np.linalg.norm(
            point_xy - obs.position
        )

        inside_horizontal = (
            horizontal_distance
            <= obs.radius + margin
        )
        inside_vertical = (
            0.0 <= z <= obs.height + margin
        )

        if inside_horizontal and inside_vertical:
            return True

    return False


In [21]:
def create_obstacle(rng, uavs):
    obstacles = []

    map_size = float(CONFIG["map_size"])
    r_min = float(CONFIG["obstacle_radius_min_m"])
    r_max = float(CONFIG["obstacle_radius_max_m"])
    h_min = float(CONFIG["obstacle_height_min_m"])
    h_max = float(CONFIG["obstacle_height_max_m"])

    safety_margin = float(CONFIG["safety_distance"])
    gcs_xy = np.asarray(CONFIG["gcs_position"][:2], dtype=np.float64)
    gcs_exclusion = float(CONFIG["gcs_exclusion_radius_m"])

    if r_min <= 0.0 or r_max < r_min:
        raise ValueError("obstacle radius range is invalid")
    if 2.0 * r_max > map_size:
        raise ValueError(
            "obstacle_radius_max_m must be <= map_size / 2"
        )

    max_attempts = 100000
    attempts = 0

    while len(obstacles) < CONFIG["num_obstacles"]:
        attempts += 1

        if attempts > max_attempts:
            raise RuntimeError(
                "Could not place all obstacles with the current constraints."
            )

        radius = float(rng.uniform(r_min, r_max))
        height = float(rng.uniform(h_min, h_max))

        x = float(rng.uniform(radius, map_size - radius))
        y = float(rng.uniform(radius, map_size - radius))
        center = np.array([x, y], dtype=np.float64)

        distance_to_gcs = np.linalg.norm(center - gcs_xy)
        if distance_to_gcs <= gcs_exclusion + radius:
            continue

        overlaps_launch = any(
            np.linalg.norm(center - uav.position[:2])
            <= radius + safety_margin
            for uav in uavs
        )
        if overlaps_launch:
            continue

        overlaps_obstacle = any(
            np.linalg.norm(center - other.position)
            <= radius + other.radius
            for other in obstacles
        )
        if overlaps_obstacle:
            continue

        obstacles.append(
            Obstacle(
                position=center,
                radius=radius,
                height=height,
            )
        )

    return obstacles


In [22]:
def create_target(rng, obstacles):
    targets = []
    occupied_cells = set()

    map_size = float(CONFIG["map_size"])
    cell_size = float(CONFIG["grid_cell_m"])
    gcs_xy = np.asarray(CONFIG["gcs_position"][:2], dtype=np.float64)
    target_exclusion = float(CONFIG["target_exclusion_radius_m"])

    max_attempts = 100000
    attempts = 0

    while len(targets) < CONFIG["num_targets"]:
        attempts += 1

        if attempts > max_attempts:
            raise RuntimeError(
                "Could not place all targets with the current constraints."
            )

        position = rng.uniform(0.0,map_size,size=2,).astype(np.float64)

        if point_inside_obstacle(position, obstacles):
            continue

        if np.linalg.norm(position - gcs_xy) <= target_exclusion:
            continue

        gx = int(np.floor(position[0] / cell_size))
        gy = int(np.floor(position[1] / cell_size))
        cell = (gx, gy)

        if cell in occupied_cells:
            continue

        targets.append(
            Target(
                id=len(targets),
                position=position,
                confirmed=False,
            )
        )
        occupied_cells.add(cell)

    return targets


In [23]:
def create_world(seed):
    rng = np.random.default_rng(seed)
    uavs = create_uav(rng)
    obstacles = create_obstacle(rng, uavs)
    targets = create_target(rng, obstacles)

    return rng, uavs, targets, obstacles


In [ ]:
def segment_intersects_obstacle(
    start_position,
    end_position,
    obstacles,
    margin=0.0,
):
    start = np.asarray(
        start_position,
        dtype=np.float64,
    )

    end = np.asarray(
        end_position,
        dtype=np.float64,
    )

    if start.shape != (3,) or end.shape != (3,):
        raise ValueError(
            "start_position and end_position must have shape (3,)"
        )

    if (
        not np.all(np.isfinite(start))
        or not np.all(np.isfinite(end))
    ):
        raise ValueError(
            "start_position and end_position "
            "must contain only finite values"
        )

    margin = float(margin)

    if not np.isfinite(margin) or margin < 0.0:
        raise ValueError(
            "margin must be finite and >= 0"
        )

    direction = end - start
    eps = 1e-12

    for obs in obstacles:

        center = np.asarray(obs.position,dtype=np.float64,)

        radius = float(obs.radius) + float(margin)
        height = float(obs.height) + float(margin)

        dz = float(direction[2])

        if abs(dz) <= eps:

            if not (
                0.0 <= start[2] <= height
            ):
                continue

            z_enter = 0.0
            z_exit = 1.0

        else:

            t_ground = (0.0 - start[2]) / dz
            t_top = (height - start[2]) / dz

            z_enter = max(0.0,min(t_ground, t_top))
            z_exit = min(1.0,max(t_ground, t_top))

            if z_enter > z_exit:
                continue

        relative_xy = start[:2] - center[:2]
        direction_xy = direction[:2]

        a = direction_xy@direction_xy
        c = (relative_xy@relative_xy)- radius * radius

        if a <= eps:

            if c > 0:
                continue

            xy_enter = 0
            xy_exit = 1

        else:

            b = 2 * relative_xy@direction_xy
            discriminant = (b * b- 4.0 * a * c)

            if discriminant < 0.0:
                continue

            root = np.sqrt(max(discriminant, 0.0))

            t1 = (-b - root) / (2.0 * a)
            t2 = (-b + root) / (2.0 * a)

            xy_enter = max(0.0,min(t1, t2))
            xy_exit = min(1.0,max(t1, t2))

            if xy_enter > xy_exit:
                continue

        enter = max(z_enter,xy_enter)
        exit_ = min(z_exit,xy_exit)

        if enter <= exit_:
            return True

    return False


In [ ]:
def compute_motion_candidate(
    uav,
    action,
    dt=None,
):
    if dt is None:
        dt = CONFIG["dt"]

    dt = float(dt)

    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(
            "dt must be finite and > 0"
        )

    action = np.asarray(
        action,
        dtype=np.float64,
    )

    if action.shape != (3,):
        raise ValueError("action must have shape (3,)")

    if not np.all(np.isfinite(action)):
        raise ValueError("action must contain only finite values")

    if not uav.active:
        return (
            uav.position.copy(),
            np.zeros(3, dtype=np.float64),
            False,
        )

    action = np.clip(
        action,
        -1.0,
        1.0,
    )

    action_norm = np.linalg.norm(action)

    if action_norm > 1.0:
        action = action / action_norm

    acceleration = action * CONFIG["max_accel"]

    candidate_velocity = (
        uav.velocity
        + acceleration * dt
    )

    speed = np.linalg.norm(candidate_velocity)

    if speed > CONFIG["max_speed"]:
        candidate_velocity = (candidate_velocity/ speed* CONFIG["max_speed"])

    old_position = uav.position.copy()

    raw_candidate_position = old_position+ candidate_velocity * dt

    candidate_position = raw_candidate_position.copy()

    candidate_position[0] = np.clip(
        candidate_position[0],
        0.0,
        CONFIG["map_size"],
    )

    candidate_position[1] = np.clip(
        candidate_position[1],
        0.0,
        CONFIG["map_size"],
    )

    candidate_position[2] = np.clip(
        candidate_position[2],
        CONFIG["altitude_min"],
        CONFIG["altitude_max"],
    )

    boundary_clipped = not np.allclose(
        raw_candidate_position,
        candidate_position,
    )

    actual_velocity = (
        candidate_position
        - old_position
    ) / dt

    return (
        candidate_position,
        actual_velocity,
        boundary_clipped,
    )


In [26]:
def minimum_distance_during_motion(
    start_a,
    end_a,
    start_b,
    end_b,
):
    start_a = np.asarray(start_a,dtype=np.float64,)
    end_a = np.asarray(end_a,dtype=np.float64,)
    start_b = np.asarray(start_b,dtype=np.float64,)
    end_b = np.asarray(end_b,dtype=np.float64,)
    for point in (
        start_a,
        end_a,
        start_b,
        end_b,
    ):
        if point.shape != (3,):
            raise ValueError(
                "all positions must have shape (3,)"
            )

        if not np.all(np.isfinite(point)):
            raise ValueError(
                "all positions must contain only finite values"
            )
    relative_start = start_a - start_b
    displacement_a = end_a - start_a
    displacement_b = end_b - start_b

    relative_motion = displacement_a - displacement_b

    denominator = relative_motion@relative_motion
    if denominator <= 1e-12:
        return np.linalg.norm(
                relative_start
            )
        

    t_closest = -(relative_start@relative_motion)/ denominator
    t_closest = np.clip(t_closest,0.0,1.0,)

    relative_at_closest = (
        relative_start
        + t_closest * relative_motion
    )

    return float(
        np.linalg.norm(
            relative_at_closest
        )
    )

In [27]:
def apply_swarm_motion(
    uavs,
    actions,
    obstacles=None,
    dt=None,
):
    if obstacles is None:
        obstacles = []

    if dt is None:
        dt = CONFIG["dt"]
    dt = float(dt)


    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(
            "dt must be finite and > 0"
        )

    num_uavs = len(uavs)

    if num_uavs == 0:
        raise ValueError(
            "uavs must not be empty"
        )

    actions = np.asarray(actions,dtype=np.float64)

    if actions.shape != (num_uavs, 3):
        raise ValueError(
            f"actions must have shape "
            f"({num_uavs}, 3)"
        )

    if not np.all(np.isfinite(actions)):
        raise ValueError(
            "actions must contain only finite values"
        )

    old_positions = np.stack([
        uav.position.copy()
        for uav in uavs
    ])

    safety_distance = float(CONFIG["safety_distance"])

    for i in range(num_uavs):
        if not uavs[i].active:
            continue

        for j in range(i + 1,num_uavs):
            if not uavs[j].active:
                continue

            distance = np.linalg.norm(old_positions[i]- old_positions[j])

            if distance < safety_distance:
                raise ValueError(
                    "initial active UAV positions "
                    "violate safety_distance"
                )

    candidate_positions = []
    candidate_velocities = []
    boundary_clipped = np.zeros(
        num_uavs,
        dtype=bool,
    )

    for i, (uav, action) in enumerate(zip(uavs, actions)):
        
        position,velocity,clipped= compute_motion_candidate(
            uav,
            action,
            dt=dt,
        )

        candidate_positions.append(position)
        candidate_velocities.append(velocity)

        boundary_clipped[i] = clipped

    candidate_positions = np.stack(candidate_positions)
    candidate_velocities = np.stack(candidate_velocities)

    blocked_by_obstacle = np.zeros(
        num_uavs,
        dtype=bool,
    )

    for i, uav in enumerate(uavs):
        if not uav.active:
            continue

        if segment_intersects_obstacle(
            old_positions[i],
            candidate_positions[i],
            obstacles,
        ):
            blocked_by_obstacle[i] = True

    blocked_by_peer = np.zeros(num_uavs,dtype=bool)
    blocked = blocked_by_obstacle.copy()
    

    while True:
        effective_positions = candidate_positions.copy()
        effective_positions[blocked] = old_positions[blocked]

        next_blocked = blocked.copy()

        for i in range(num_uavs):
            if not uavs[i].active:
                continue

            for j in range(i + 1,num_uavs):
                if not uavs[j].active:
                    continue

                distance = minimum_distance_during_motion(
                        old_positions[i],
                        effective_positions[i],
                        old_positions[j],
                        effective_positions[j],
                    )
                
                if distance < safety_distance:
                    next_blocked[i] = True
                    next_blocked[j] = True

                    blocked_by_peer[i] = True
                    blocked_by_peer[j] = True

        if np.array_equal(next_blocked,blocked):
            break

        blocked = next_blocked

    for i, uav in enumerate(uavs):
        if not uav.active:
            uav.velocity = np.zeros(3, dtype=np.float64)
            continue

        if blocked[i]:
            uav.position = old_positions[i].copy()
            uav.velocity = np.zeros(3,dtype=np.float64)

        else:
            uav.position = candidate_positions[i].copy()
            uav.velocity = candidate_velocities[i].copy()

    return {
        "blocked": blocked,
        "blocked_by_obstacle":blocked_by_obstacle,
        "blocked_by_peer":blocked_by_peer,
        "boundary_clipped":boundary_clipped,
    }


In [ ]:
def sensing_profile(altitude):
    altitude = float(altitude)

    if not np.isfinite(altitude):
        raise ValueError("altitude must be finite")

    altitude = np.clip(altitude,CONFIG["altitude_min"],CONFIG["altitude_max"])
    altitude_anchors = np.asarray(CONFIG["altitude_anchors"])

    pd_anchors = np.asarray(CONFIG["pd"])
    pf_anchors = np.asarray(CONFIG["pf"])

    pd = np.interp(altitude,altitude_anchors,pd_anchors)
    pf = np.interp(altitude,altitude_anchors,pf_anchors)

    full_fov = CONFIG["camera_full_fov_deg"]
    half_fov_rad = np.deg2rad(full_fov / 2.0)
    fov_radius = altitude* np.tan(half_fov_rad)

    return (
        float(pd),
        float(pf),
        float(fov_radius)
    )

In [29]:
def create_belief_maps():
    grid_n = int(np.ceil(CONFIG["map_size"] / CONFIG["grid_cell_m"]))
    belief_maps = np.full((CONFIG["num_uavs"],grid_n,grid_n),CONFIG["belief_prior"])
    return belief_maps

In [ ]:
def world_to_grid(position_xy):
    position_xy = np.asarray(position_xy, dtype=np.float64)

    if position_xy.ndim != 1 or position_xy.size < 2:
        raise ValueError(
            "position_xy must be a 1D array containing at least x and y"
        )

    x = float(position_xy[0])
    y = float(position_xy[1])

    if not np.isfinite(x) or not np.isfinite(y):
        raise ValueError("position must be finite")

    map_size = float(CONFIG["map_size"])
    cell_size = float(CONFIG["grid_cell_m"])

    if not np.isfinite(map_size) or map_size <= 0.0:
        raise ValueError("map_size must be finite and > 0")

    if not np.isfinite(cell_size) or cell_size <= 0.0:
        raise ValueError("grid_cell_m must be finite and > 0")

    if not (
        0.0 <= x <= map_size
        and 0.0 <= y <= map_size
    ):
        raise ValueError("position is outside the map")

    grid_n = int(np.ceil(map_size / cell_size))

    gx = int(np.floor(x / cell_size))
    gy = int(np.floor(y / cell_size))

    gx = min(gx, grid_n - 1)
    gy = min(gy, grid_n - 1)

    return gx, gy


In [31]:
def cell_intersects_fov(
    gx,
    gy,
    uav_xy,
    fov_radius,
):
    cell_size = float(CONFIG["grid_cell_m"])
    map_size = float(CONFIG["map_size"])

    x_min = gx * cell_size
    x_max = min((gx + 1) * cell_size,map_size)

    y_min = gy * cell_size
    y_max = min((gy + 1) * cell_size,map_size)

    closest_x = np.clip(uav_xy[0],x_min,x_max)
    closest_y = np.clip(uav_xy[1],y_min,y_max)

    dx = float(uav_xy[0] - closest_x)
    dy = float(uav_xy[1] - closest_y)

    return (
        dx * dx + dy * dy <= fov_radius * fov_radius
    )

In [32]:
import math


def _circle_sqrt_integral(x, radius):
    """Antiderivative of sqrt(radius^2 - x^2) on [-radius, radius]."""
    x = max(-radius, min(radius, float(x)))
    root = math.sqrt(max(0.0, radius * radius - x * x))
    return 0.5 * (
        x * root
        + radius * radius * math.asin(x / radius)
    )


def _circle_rectangle_intersection_area(
    circle_x,
    circle_y,
    radius,
    x_min,
    x_max,
    y_min,
    y_max,
):
    """Exact area of a circle intersected with an axis-aligned rectangle."""
    radius = float(radius)
    if radius <= 0.0 or x_max <= x_min or y_max <= y_min:
        return 0.0

    # Translate the circle center to the origin.
    x_min = float(x_min) - float(circle_x)
    x_max = float(x_max) - float(circle_x)
    y_min = float(y_min) - float(circle_y)
    y_max = float(y_max) - float(circle_y)

    left = max(x_min, -radius)
    right = min(x_max, radius)

    if right <= left or y_max <= -radius or y_min >= radius:
        return 0.0

    # The integrand changes form only where a horizontal rectangle edge
    # intersects the circle. Split at those x values, then integrate the
    # circle arc analytically on each interval.
    cuts = [left, right]
    for y_edge in (y_min, y_max):
        if abs(y_edge) < radius:
            x_cross = math.sqrt(
                max(0.0, radius * radius - y_edge * y_edge)
            )
            for cut in (-x_cross, x_cross):
                if left < cut < right:
                    cuts.append(cut)

    cuts = sorted(set(cuts))
    area = 0.0

    for a, b in pairwise(cuts):
        if b <= a:
            continue

        midpoint = 0.5 * (a + b)
        half_height = math.sqrt(
            max(0.0, radius * radius - midpoint * midpoint)
        )

        upper = min(y_max, half_height)
        lower = max(y_min, -half_height)
        if upper <= lower:
            continue

        arc_integral = (
            _circle_sqrt_integral(b, radius)
            - _circle_sqrt_integral(a, radius)
        )

        if y_max < half_height:
            upper_integral = y_max * (b - a)
        else:
            upper_integral = arc_integral

        if y_min > -half_height:
            lower_integral = y_min * (b - a)
        else:
            lower_integral = -arc_integral

        area += upper_integral - lower_integral

    return max(0.0, float(area))


def cell_fov_coverage_fraction(gx, gy, uav_xy, fov_radius):
    """Return the fraction of a belief cell covered by the circular FOV.

    Geometry depends only on UAV/FOV and grid cell; target ground truth never
    changes measurement support. The circle/rectangle area is analytic, which
    is both deterministic and much cheaper than per-cell numerical quadrature.
    """
    if isinstance(gx, (bool, np.bool_)) or not isinstance(gx, (int, np.integer)):
        raise TypeError("gx must be an integer")
    if isinstance(gy, (bool, np.bool_)) or not isinstance(gy, (int, np.integer)):
        raise TypeError("gy must be an integer")

    gx = int(gx)
    gy = int(gy)
    uav_xy = np.asarray(uav_xy, dtype=np.float64)
    fov_radius = float(fov_radius)

    if uav_xy.shape != (2,) or not np.all(np.isfinite(uav_xy)):
        raise ValueError("uav_xy must be a finite shape-(2,) position")
    if not np.isfinite(fov_radius) or fov_radius < 0.0:
        raise ValueError("fov_radius must be finite and >= 0")
    if fov_radius == 0.0:
        return 0.0

    cell_size = float(CONFIG["grid_cell_m"])
    map_size = float(CONFIG["map_size"])
    grid_n = int(np.ceil(map_size / cell_size))

    if not (0 <= gx < grid_n and 0 <= gy < grid_n):
        raise ValueError("grid cell index out of range")

    x_min = gx * cell_size
    x_max = min((gx + 1) * cell_size, map_size)
    y_min = gy * cell_size
    y_max = min((gy + 1) * cell_size, map_size)

    if not cell_intersects_fov(gx, gy, uav_xy, fov_radius):
        return 0.0

    intersection_area = _circle_rectangle_intersection_area(
        circle_x=float(uav_xy[0]),
        circle_y=float(uav_xy[1]),
        radius=fov_radius,
        x_min=x_min,
        x_max=x_max,
        y_min=y_min,
        y_max=y_max,
    )
    cell_area = (x_max - x_min) * (y_max - y_min)

    if cell_area <= 0.0:
        return 0.0

    return float(np.clip(intersection_area / cell_area, 0.0, 1.0))

def cells_with_fov_coverage(uav):
    if not uav.active:
        return []

    altitude = float(uav.position[2])

    if altitude <= 0.0:
        return []

    _, _, fov_radius = sensing_profile(altitude)
    cell_size = float(CONFIG["grid_cell_m"])
    grid_n = int(np.ceil(CONFIG["map_size"] / cell_size))
    uav_xy = np.asarray(uav.position[:2], dtype=np.float64)
    uav_x = float(uav_xy[0])
    uav_y = float(uav_xy[1])

    gx_min = max(0, int(np.floor((uav_x - fov_radius) / cell_size)))
    gx_max = min(grid_n - 1, int(np.floor((uav_x + fov_radius) / cell_size)))
    gy_min = max(0, int(np.floor((uav_y - fov_radius) / cell_size)))
    gy_max = min(grid_n - 1, int(np.floor((uav_y + fov_radius) / cell_size)))

    visible = []
    for gy in range(gy_min, gy_max + 1):
        for gx in range(gx_min, gx_max + 1):
            coverage = cell_fov_coverage_fraction(
                gx,
                gy,
                uav_xy,
                fov_radius,
            )
            if coverage > 0.0:
                visible.append((gx, gy, coverage))

    return visible


def cells_in_fov(uav):
    return [
        (gx, gy)
        for gx, gy, _ in cells_with_fov_coverage(uav)
    ]

In [33]:
def get_target_cells(targets):
    target_cells = {}

    for target in targets:
        gx, gy = world_to_grid(target.position)
        target_cells.setdefault((gx, gy), []).append(target.id)

    return target_cells


In [34]:
def sample_sensor_measurement(has_target,pd,pf,rng):
    if has_target:
        positive_probability = pd

    else:
        positive_probability = pf
    observation = (rng.random()< positive_probability)

    return int(observation)

In [ ]:
def bayes_update(prior, observation, pd, pf, eps=1e-8):
    eps = float(eps)

    if (
        not np.isfinite(eps)
        or not 0.0 < eps < 0.5
    ):
        raise ValueError(
            "eps must be finite and in (0, 0.5)"
        )

    prior = float(prior)
    pd = float(pd)
    pf = float(pf)

    for name, value in (
        ("prior", prior),
        ("pd", pd),
        ("pf", pf),
    ):
        if not np.isfinite(value):
            raise ValueError(
                f"{name} must be finite"
            )

        if not 0.0 <= value <= 1.0:
            raise ValueError(
                f"{name} must be in [0, 1]"
            )

    prior = np.clip(prior,eps,1.0 - eps)

    if observation == 1:
        numerator = pd * prior
        denominator = pd * prior + pf * (1.0 - prior)
    elif observation == 0:
        numerator = (1.0 - pd) * prior
        denominator = (
            (1.0 - pd) * prior
            + (1.0 - pf) * (1.0 - prior)
        )
    else:
        raise ValueError("observation must be 0 or 1")

    denominator = max(float(denominator), eps)
    posterior = numerator / denominator

    return float(np.clip(posterior, eps, 1.0 - eps))


In [36]:
def sense_and_update(uav, belief_map, targets, rng):
    if not uav.active:
        return []

    altitude = float(uav.position[2])

    if altitude <= 0.0:
        return []

    base_pd, base_pf, fov_radius = sensing_profile(altitude)
    visible_cells = cells_with_fov_coverage(uav)
    target_cells = get_target_cells(targets)
    targets_by_id = {target.id: target for target in targets}

    sensing_log = []

    for gx, gy, coverage_fraction in visible_cells:
        candidate_target_ids = target_cells.get((gx, gy), [])
        target_ids_in_fov = [
            target_id
            for target_id in candidate_target_ids
            if np.linalg.norm(
                targets_by_id[target_id].position - uav.position[:2]
            ) <= fov_radius
        ]

        has_target = len(target_ids_in_fov) > 0

        # Scale cell-level evidence by the actually observed cell fraction.
        # At zero coverage the measurement carries no information; at full
        # coverage this reduces exactly to the original Pd/Pf model.
        effective_pf = float(
            1.0 - (1.0 - base_pf) ** coverage_fraction
        )
        effective_pd = float(
            coverage_fraction * base_pd
            + (1.0 - coverage_fraction) * effective_pf
        )

        if has_target:
            # The simulator knows the target's true continuous position is
            # inside the footprint, so the physical detector uses base Pd.
            positive_probability = base_pd
        else:
            # False alarms scale with the observed fraction of the cell.
            positive_probability = effective_pf

        observation = int(rng.random() < positive_probability)

        prior = float(belief_map[gy, gx])
        posterior = bayes_update(
            prior,
            observation,
            effective_pd,
            effective_pf,
        )
        belief_map[gy, gx] = posterior

        sensing_log.append({
            "gx": gx,
            "gy": gy,
            "target_ids": target_ids_in_fov,
            "has_target": has_target,
            "observation": observation,
            "prior": prior,
            "posterior": posterior,
            "pd": effective_pd,
            "pf": effective_pf,
            "base_pd": base_pd,
            "base_pf": base_pf,
            "coverage_fraction": float(coverage_fraction),
        })

    return sensing_log

In [37]:
def check_confirmation(uav, sensing_record):
    if not uav.active:
        return "none", []

    posterior = float(sensing_record["posterior"])
    observation = int(sensing_record["observation"])
    target_ids = list(sensing_record["target_ids"])

    if posterior < CONFIG["confirmation_threshold"]:
        return "none", []

    if observation != 1:
        return "none", []

    if target_ids:
        return "true_confirmation", target_ids

    return "false_confirmation", []

In [38]:
@dataclass
class ConfirmationEvent:
    target_id: int | None
    uav_id: int
    step: int

    cell: tuple
    uav_position: np.ndarray

    altitude: float
    belief: float
    observation: int

    confirmation_type: str

In [39]:
def create_confirmation_events(
    uav,
    sensing_record,
    step
):
    confirmation_type, target_ids = check_confirmation(
        uav,
        sensing_record
    )

    if confirmation_type == "none":
        return []

    common = {
        "uav_id": int(uav.id),
        "step": int(step),
        "cell": (
            int(sensing_record["gx"]),
            int(sensing_record["gy"]),
        ),
        "uav_position": uav.position.copy(),
        "altitude": float(uav.position[2]),
        "belief": float(sensing_record["posterior"]),
        "observation": int(sensing_record["observation"]),
        "confirmation_type": confirmation_type,
    }

    if confirmation_type == "false_confirmation":
        return [ConfirmationEvent(target_id=None,**common,)]

    return [ConfirmationEvent(target_id=int(target_id),**common,) for target_id in target_ids]

In [40]:
def process_confirmation_events(
    events,
    targets,
    report_buffers,
    pending_reports,
    gcs_received_target_ids,
):
    reports = []
    event_log = []

    if not isinstance(report_buffers, list):
        raise TypeError("report_buffers must be a list")
    if not isinstance(pending_reports, list):
        raise TypeError("pending_reports must be a list")
    if not isinstance(gcs_received_target_ids, set):
        raise TypeError("gcs_received_target_ids must be a set")

    targets_by_id = {
        target.id: target
        for target in targets
    }

    generated_target_ids = set()

    for event in events:
        event_log.append(event)

        if event.confirmation_type != "true_confirmation":
            continue

        if event.target_id not in targets_by_id:
            raise ValueError("confirmation event target_id not found")

        target = targets_by_id[event.target_id]
        target.confirmed = True

        # Expired network state must not suppress a fresh observation.
        # Clean stale buffered/pending copies using the observation step
        # before duplicate checks, so correctness does not depend on caller
        # cleanup order.
        current_step = int(event.step)
        for buffer in report_buffers:
            remove_expired_reports(buffer, current_step)
        remove_expired_reports(pending_reports, current_step)

        if target.id in gcs_received_target_ids:
            continue

        report_already_buffered = any(
            report.target_id == target.id
            for buffer in report_buffers
            for report in buffer
        )
        report_already_pending = any(
            report.target_id == target.id
            for report in pending_reports
        )

        if (
            report_already_buffered
            or report_already_pending
            or target.id in generated_target_ids
        ):
            continue

        source_uav = int(event.uav_id)
        if not 0 <= source_uav < len(report_buffers):
            raise ValueError("confirmation event uav_id out of range")

        report = Report(
            target_id=target.id,
            source_uav=source_uav,
            created_step=event.step,
            size_bytes=CONFIG["report_bytes"],
            ttl_s=CONFIG["report_ttl"],
        )

        enqueued, reason = enqueue_report(
            report_buffers[source_uav],
            report,
        )

        if not enqueued:
            if reason == "buffer_full":
                pending_reports.append(report)
            elif reason != "duplicate":
                raise RuntimeError(f"unexpected enqueue result: {reason}")

        reports.append(report)
        generated_target_ids.add(target.id)

    return reports, event_log

In [41]:
def create_report_buffers():
    return [[] for _ in range(CONFIG["num_uavs"])]


def create_pending_reports():
    return []


def create_gcs_received_target_ids():
    return set()

In [42]:
def report_remaining_bytes(report):
    remaining = report.size_bytes - report.delivered_bytes
    return max(0, int(remaining))


def report_is_complete(report):
    return report_remaining_bytes(report) == 0


def buffer_used_bytes(buffer):
    return sum(report.size_bytes for report in buffer)

In [43]:
def report_exists(buffer, target_id):
    return any(
        report.target_id == target_id for report in buffer
    )


def report_exists_in_buffers(report_buffers, target_id):
    return any(
        report_exists(buffer, target_id)
        for buffer in report_buffers
    )


def pending_report_exists(pending_reports, target_id):
    return any(
        report.target_id == target_id
        for report in pending_reports
    )

In [44]:
def enqueue_report(buffer, report):
    if not isinstance(buffer, list):
        raise TypeError("buffer must be a list")
    if not isinstance(report, Report):
        raise TypeError("report must be a Report")

    # A report whose bytes have all reached the GCS is retained until
    # mark_target_delivered_to_gcs() records mission-level delivery and
    # removes the report. This prevents losing delivery state between steps.
    if report_exists(buffer, report.target_id):
        return False, "duplicate"

    used_bytes = buffer_used_bytes(buffer)
    new_used_bytes = used_bytes + report.size_bytes

    if new_used_bytes > CONFIG["buffer_bytes"]:
        return False, "buffer_full"

    buffer.append(report)
    return True, "enqueued"

In [ ]:
def report_age_s(report, current_step):
    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(current_step, (int, np.integer))
    ):
        raise TypeError("current_step must be an integer")

    created_step = report.created_step

    if (
        isinstance(created_step, (bool, np.bool_))
        or not isinstance(created_step, (int, np.integer))
    ):
        raise TypeError("report.created_step must be an integer")

    current_step = int(current_step)
    created_step = int(created_step)

    if created_step < 0:
        raise ValueError("report.created_step must be >= 0")

    if current_step < created_step:
        raise ValueError(
            "current_step must be >= report.created_step"
        )

    dt = float(CONFIG["dt"])

    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError("CONFIG['dt'] must be finite and > 0")

    return float(
        (current_step - created_step) * dt
    )


In [46]:
def report_is_expired(report, current_step):
    age_s = report_age_s(report,current_step)

    return age_s >= report.ttl_s

In [47]:
def remove_expired_reports(
    buffer,
    current_step,
):
    kept_reports = []
    expired_reports = []

    for report in buffer:
        if report_is_expired(report, current_step):
            expired_reports.append(report)
        else:
            kept_reports.append(report)

    buffer[:] = kept_reports
    return expired_reports


def remove_completed_reports(buffer):
    completed_reports = [
        report
        for report in buffer
        if report_is_complete(report)
    ]
    buffer[:] = [
        report
        for report in buffer
        if not report_is_complete(report)
    ]
    return completed_reports


def cleanup_report_buffer(buffer, current_step):
    expired = remove_expired_reports(buffer, current_step)
    completed = [
        report
        for report in buffer
        if report_is_complete(report)
    ]
    # Completed reports are intentionally retained until the GCS-delivery
    # state is marked. FIFO selection skips them without deleting them.
    return {
        "expired": expired,
        "completed": completed,
    }


def flush_pending_reports(
    pending_reports,
    report_buffers,
    current_step,
    gcs_received_target_ids,
    uavs,
):
    if not isinstance(pending_reports, list):
        raise TypeError("pending_reports must be a list")
    if not isinstance(report_buffers, list):
        raise TypeError("report_buffers must be a list")
    if not isinstance(gcs_received_target_ids, set):
        raise TypeError("gcs_received_target_ids must be a set")
    if not isinstance(uavs, list):
        raise TypeError("uavs must be a list")

    uavs_by_id = {}
    for uav in uavs:
        if not isinstance(uav, UAV):
            raise TypeError("uavs must contain only UAV objects")
        uav_id = int(uav.id)
        if uav_id in uavs_by_id:
            raise ValueError("UAV ids must be unique")
        uavs_by_id[uav_id] = uav

    for buffer in report_buffers:
        cleanup_report_buffer(buffer, current_step)

    kept_pending = []
    enqueued_target_ids = []
    expired_target_ids = []
    discarded_target_ids = []
    deferred_inactive_target_ids = []

    for report in pending_reports:
        if report.target_id in gcs_received_target_ids:
            discarded_target_ids.append(report.target_id)
            continue

        if report_is_expired(report, current_step):
            expired_target_ids.append(report.target_id)
            continue

        if report_exists_in_buffers(report_buffers, report.target_id):
            discarded_target_ids.append(report.target_id)
            continue

        source_uav = int(report.source_uav)
        if not 0 <= source_uav < len(report_buffers):
            raise ValueError("pending report source_uav out of range")
        if source_uav not in uavs_by_id:
            raise ValueError("pending report source_uav not found")

        if not uavs_by_id[source_uav].active:
            kept_pending.append(report)
            deferred_inactive_target_ids.append(report.target_id)
            continue

        enqueued, reason = enqueue_report(
            report_buffers[source_uav],
            report,
        )

        if enqueued:
            enqueued_target_ids.append(report.target_id)
        elif reason == "buffer_full":
            kept_pending.append(report)
        elif reason == "duplicate":
            discarded_target_ids.append(report.target_id)
        else:
            raise RuntimeError(f"unexpected enqueue result: {reason}")

    pending_reports[:] = kept_pending

    return {
        "enqueued_target_ids": enqueued_target_ids,
        "expired_target_ids": expired_target_ids,
        "discarded_target_ids": discarded_target_ids,
        "deferred_inactive_target_ids": deferred_inactive_target_ids,
    }

def mark_target_delivered_to_gcs(
    target_id,
    gcs_received_target_ids,
    report_buffers,
    pending_reports,
):
    if isinstance(target_id, (bool, np.bool_)) or not isinstance(
        target_id,
        (int, np.integer),
    ):
        raise TypeError("target_id must be an integer")

    target_id = int(target_id)
    if target_id < 0:
        raise ValueError("target_id must be >= 0")
    if not isinstance(gcs_received_target_ids, set):
        raise TypeError("gcs_received_target_ids must be a set")

    gcs_received_target_ids.add(target_id)

    removed_from_buffers = 0
    for buffer in report_buffers:
        before = len(buffer)
        buffer[:] = [
            report
            for report in buffer
            if report.target_id != target_id
        ]
        removed_from_buffers += before - len(buffer)

    before_pending = len(pending_reports)
    pending_reports[:] = [
        report
        for report in pending_reports
        if report.target_id != target_id
    ]

    return {
        "removed_from_buffers": removed_from_buffers,
        "removed_from_pending": before_pending - len(pending_reports),
    }

In [48]:
def distance_3d(position_a, position_b):
    position_a = np.asarray(position_a,dtype=np.float64)
    position_b = np.asarray(position_b, dtype=np.float64)

    if position_a.shape != (3,):
        raise ValueError("position_a must have shape (3,)")

    if position_b.shape != (3,):
        raise ValueError("position_b must have shape (3,)")

    if not np.all(np.isfinite(position_a)):
        raise ValueError("position_a must contain only finite values")

    if not np.all(np.isfinite(position_b)):
        raise ValueError("position_b must contain only finite values")

    return np.linalg.norm(position_a - position_b)
    

In [49]:
def positions_can_communicate(position_a,position_b,max_range_m):
    max_range_m = float(max_range_m)

    if (
        not np.isfinite(max_range_m)
        or max_range_m <= 0.0
    ):
        raise ValueError(
            "max_range_m must be finite and > 0"
        )

    return (distance_3d(position_a,position_b)<= max_range_m)

In [50]:
def uavs_can_communicate(uav_a,uav_b):
    if uav_a.id == uav_b.id:
        return False

    if not uav_a.active:
        return False

    if not uav_b.active:
        return False

    return positions_can_communicate(
        uav_a.position,
        uav_b.position,
        CONFIG["peer_contact_range_m"]
    )

In [51]:
def uav_can_reach_gcs(uav):
    if not uav.active:
        return False

    return positions_can_communicate(
        uav.position,
        CONFIG["gcs_position"],
        CONFIG["gcs_contact_range_m"],
    )

In [52]:
def get_uav_neighbors(uav,uavs):
    neighbors = []

    for other in uavs:
        if uavs_can_communicate(uav,other):
            neighbors.append(other.id)

    return neighbors

In [53]:
GCS_NODE = -1
SILENT_DESTINATION = None

In [54]:
def communication_choices(sender_id,num_uavs=None):
    if num_uavs is None:
        num_uavs = CONFIG["num_uavs"]

    if (isinstance(sender_id, bool) or not isinstance(sender_id,(int, np.integer))):
        raise TypeError("sender_id must be an integer")

    if (isinstance(num_uavs, bool) or not isinstance(num_uavs,(int, np.integer))):
        raise TypeError("num_uavs must be an integer")

    sender_id = int(sender_id)
    num_uavs = int(num_uavs)

    if num_uavs < 1:
        raise ValueError("num_uavs must be >= 1")

    if not 0 <= sender_id < num_uavs:
        raise ValueError("sender_id out of range")

    peers = tuple(
        uav_id
        for uav_id in range(num_uavs)
        if uav_id != sender_id
    )

    return (SILENT_DESTINATION,*peers,GCS_NODE)

In [ ]:
def decode_destination(
    sender_id,
    destination_index,
    num_uavs=None,
):
    if num_uavs is None:
        num_uavs = CONFIG["num_uavs"]

    if (
        isinstance(
            destination_index,
            (bool, np.bool_),
        )
        or not isinstance(
            destination_index,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "destination_index must be an integer"
        )

    destination_index = int(destination_index)

    choices = communication_choices(
        sender_id,
        num_uavs,
    )

    if not 0<= destination_index< len(choices):
        raise ValueError(
            "destination_index out of range"
        )

    return choices[destination_index]

In [ ]:
def decode_tx_power_w(power_action):
    power_array = np.asarray(power_action)

    if power_array.shape not in [(), (1,)]:
        raise ValueError(
            "power_action must be a scalar or have shape (1,)"
        )

    if (
        np.issubdtype(power_array.dtype, np.bool_)
        or not np.issubdtype(power_array.dtype, np.number)
    ):
        raise TypeError(
            "power_action must be numeric and not boolean"
        )

    power_value = float(power_array.item())

    if not np.isfinite(power_value):
        raise ValueError(
            "power_action must be finite"
        )

    if not -1.0 <= power_value <= 1.0:
        raise ValueError(
            "power_action must be within [-1, 1]"
        )

    power_min = float(CONFIG["tx_power_min_w"])
    power_max = float(CONFIG["tx_power_max_w"])

    if (
        not np.isfinite(power_min)
        or not np.isfinite(power_max)
        or power_min <= 0.0
        or power_max < power_min
    ):
        raise ValueError(
            "invalid TX power range"
        )

    normalized = (power_value + 1.0) / 2.0

    return float(
        power_min
        + normalized
        * (power_max - power_min)
    )


In [60]:
@dataclass
class HybridAction:
    movement: np.ndarray
    destination: int | None
    tx_power_w: float

In [ ]:
def decode_hybrid_action(
    sender_id,
    action,
    num_uavs=None,
):
    if num_uavs is None:
        num_uavs = CONFIG["num_uavs"]

    if not isinstance(action, dict):
        raise TypeError(
            "action must be a dict"
        )

    required_keys = {
        "motion",
        "destination",
        "power",
    }

    if set(action.keys()) != required_keys:
        raise ValueError(
            "action must contain exactly "
            "motion, destination, power"
        )

    motion_object = np.asarray(
        action["motion"],
        dtype=object,
    )

    if motion_object.shape != (3,):
        raise ValueError(
            "motion must have shape (3,)"
        )

    for value in motion_object:
        if isinstance(value, (bool, np.bool_)):
            raise TypeError(
                "motion must be numeric and not boolean"
            )

        if not isinstance(
            value,
            (int, float, np.integer, np.floating),
        ):
            raise TypeError(
                "motion must be numeric and not boolean"
            )

    motion = motion_object.astype(
        np.float64,
    )

    if not np.all(np.isfinite(motion)):
        raise ValueError(
            "motion must contain only finite values"
        )

    if (
        np.any(motion < -1.0)
        or np.any(motion > 1.0)
    ):
        raise ValueError(
            "motion must be within [-1, 1]"
        )

    destination = decode_destination(
        sender_id,
        action["destination"],
        num_uavs,
    )

    decoded_power_w = decode_tx_power_w(action["power"])

    if destination is SILENT_DESTINATION:
        tx_power_w = 0.0
    else:
        tx_power_w = decoded_power_w

    return HybridAction(
        movement=motion.copy(),
        destination=destination,
        tx_power_w=tx_power_w,
    )


In [ ]:
def select_report_for_transmission(
    buffer,
    current_step,
):
    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(current_step, (int, np.integer))
    ):
        raise TypeError("current_step must be an integer")

    current_step = int(current_step)

    if not isinstance(buffer, list):
        raise TypeError("buffer must be a list")

    for report in buffer:
        if not isinstance(report, Report):
            raise TypeError("buffer must contain only Report objects")

    cleanup_report_buffer(buffer, current_step)

    for report in buffer:
        if not report_is_complete(report):
            return report

    return None

In [ ]:
@dataclass(frozen=True)
class TransmissionIntent:
    sender: int
    recipient: int
    target_id: int
    requested_bytes: int
    tx_power_w: float

    def __post_init__(self):
        for name, value in (
            ("sender", self.sender),
            ("recipient", self.recipient),
            ("target_id", self.target_id),
            ("requested_bytes", self.requested_bytes),
        ):
            if (
                isinstance(value, (bool, np.bool_))
                or not isinstance(
                    value,
                    (int, np.integer),
                )
            ):
                raise TypeError(
                    f"{name} must be an integer"
                )

        sender = int(self.sender)
        recipient = int(self.recipient)
        target_id = int(self.target_id)
        requested_bytes = int(
            self.requested_bytes
        )
        tx_power_w = float(
            self.tx_power_w
        )

        num_uavs = int(
            CONFIG["num_uavs"]
        )

        if not 0 <= sender < num_uavs:
            raise ValueError(
                "sender out of range"
            )

        if (
            recipient != GCS_NODE
            and not 0 <= recipient < num_uavs
        ):
            raise ValueError(
                "recipient must be GCS_NODE "
                "or a valid UAV id"
            )

        if recipient == sender:
            raise ValueError(
                "sender and recipient "
                "must be different"
            )

        if target_id < 0:
            raise ValueError(
                "target_id must be >= 0"
            )

        if requested_bytes <= 0:
            raise ValueError(
                "requested_bytes must be > 0"
            )

        power_min = float(
            CONFIG["tx_power_min_w"]
        )
        power_max = float(
            CONFIG["tx_power_max_w"]
        )

        if (
            not np.isfinite(tx_power_w)
            or not power_min
            <= tx_power_w
            <= power_max
        ):
            raise ValueError(
                "tx_power_w outside "
                "configured range"
            )

        object.__setattr__(
            self,
            "sender",
            sender,
        )
        object.__setattr__(
            self,
            "recipient",
            recipient,
        )
        object.__setattr__(
            self,
            "target_id",
            target_id,
        )
        object.__setattr__(
            self,
            "requested_bytes",
            requested_bytes,
        )
        object.__setattr__(
            self,
            "tx_power_w",
            tx_power_w,
        )

In [ ]:
def build_transmission_intent(
    sender_id,
    hybrid_action,
    buffer,
    current_step,
    peer_transfer_states=None,
):
    if (
        isinstance(
            sender_id,
            (bool, np.bool_),
        )
        or not isinstance(
            sender_id,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "sender_id must be an integer"
        )

    sender_id = int(sender_id)

    num_uavs = int(
        CONFIG["num_uavs"]
    )

    if not 0 <= sender_id < num_uavs:
        raise ValueError(
            "sender_id out of range"
        )

    if not isinstance(
        hybrid_action,
        HybridAction,
    ):
        raise TypeError(
            "hybrid_action must be HybridAction"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(current_step)

    if not isinstance(
        buffer,
        list,
    ):
        raise TypeError(
            "buffer must be a list"
        )

    for report in buffer:
        if not isinstance(
            report,
            Report,
        ):
            raise TypeError(
                "buffer must contain only "
                "Report objects"
            )

    if (
        peer_transfer_states is not None
        and not isinstance(
            peer_transfer_states,
            dict,
        )
    ):
        raise TypeError(
            "peer_transfer_states must be "
            "a dict or None"
        )

    if (
        hybrid_action.destination
        is SILENT_DESTINATION
    ):
        return None

    cleanup_report_buffer(
        buffer,
        current_step,
    )

    destination = (
        hybrid_action.destination
    )

    if destination == GCS_NODE:
        for report in buffer:
            if report_is_complete(
                report
            ):
                continue

            requested_bytes = (
                report_remaining_bytes(
                    report
                )
            )

            if requested_bytes <= 0:
                continue

            return TransmissionIntent(
                sender=sender_id,
                recipient=destination,
                target_id=report.target_id,
                requested_bytes=requested_bytes,
                tx_power_w=(
                    hybrid_action.tx_power_w
                ),
            )

        return None

    # Peer destination: preserve FIFO order, but skip any report
    # that this exact peer already received completely.
    for report in buffer:
        if report_is_complete(
            report
        ):
            continue

        requested_bytes = int(
            report.size_bytes
        )

        if peer_transfer_states is not None:
            key = peer_transfer_key(
                sender_id,
                destination,
                report.target_id,
            )

            state = (
                peer_transfer_states.get(
                    key
                )
            )

            if state is not None:
                if not isinstance(
                    state,
                    PeerTransferState,
                ):
                    raise TypeError(
                        "peer_transfer_states "
                        "values must be "
                        "PeerTransferState"
                    )

                if peer_transfer_is_expired(
                    state,
                    current_step,
                ):
                    del peer_transfer_states[
                        key
                    ]
                    state = None

            if state is not None:
                if (
                    state.source_uav
                    != report.source_uav
                    or state.created_step
                    != report.created_step
                    or state.size_bytes
                    != report.size_bytes
                    or not np.isclose(
                        state.ttl_s,
                        report.ttl_s,
                    )
                ):
                    raise ValueError(
                        "peer transfer state "
                        "does not match "
                        "sender report"
                    )

                requested_bytes = (
                    peer_transfer_remaining_bytes(
                        state
                    )
                )

                if requested_bytes <= 0:
                    # This peer already has the full report.
                    # Continue FIFO search so later reports are
                    # not head-of-line blocked.
                    continue

        return TransmissionIntent(
            sender=sender_id,
            recipient=destination,
            target_id=report.target_id,
            requested_bytes=(
                requested_bytes
            ),
            tx_power_w=(
                hybrid_action.tx_power_w
            ),
        )

    return None


In [ ]:
def transmission_is_feasible(
    intent,
    uavs,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if not isinstance(uavs, list):
        raise TypeError(
            "uavs must be a list"
        )

    num_uavs = int(
        CONFIG["num_uavs"]
    )

    if len(uavs) != num_uavs:
        raise ValueError(
            "uavs length does not match "
            "CONFIG['num_uavs']"
        )

    for uav in uavs:
        if not isinstance(uav, UAV):
            raise TypeError(
                "uavs must contain only UAV objects"
            )

        if (
            isinstance(uav.id, (bool, np.bool_))
            or not isinstance(
                uav.id,
                (int, np.integer),
            )
        ):
            raise TypeError(
                "each UAV id must be an integer"
            )

    ids = [
        int(uav.id)
        for uav in uavs
    ]

    if len(set(ids)) != num_uavs:
        raise ValueError(
            "UAV ids must be unique"
        )

    if set(ids) != set(range(num_uavs)):
        raise ValueError(
            "UAV ids must be exactly "
            "0..num_uavs-1"
        )

    uavs_by_id = {
        int(uav.id): uav
        for uav in uavs
    }

    sender = uavs_by_id[
        intent.sender
    ]

    if not sender.active:
        return False

    if intent.recipient == GCS_NODE:
        return bool(
            uav_can_reach_gcs(
                sender
            )
        )

    recipient = uavs_by_id[
        intent.recipient
    ]

    return bool(
        uavs_can_communicate(
            sender,
            recipient,
        )
    )


In [ ]:
LOS_LINK = "los"
NLOS_LINK = "nlos"


def positions_have_los(
    position_a,
    position_b,
    obstacles,
):
    if obstacles is None:
        obstacles = []

    if not isinstance(obstacles, list):
        raise TypeError("obstacles must be a list")

    for obstacle in obstacles:
        if not isinstance(obstacle, Obstacle):
            raise TypeError(
                "obstacles must contain only Obstacle objects"
            )

    return not segment_intersects_obstacle(
        position_a,
        position_b,
        obstacles,
    )


def classify_link_state(
    position_a,
    position_b,
    obstacles,
):
    if positions_have_los(
        position_a,
        position_b,
        obstacles,
    ):
        return LOS_LINK

    return NLOS_LINK


In [ ]:
def db_to_linear(db_value):
    db_value = float(db_value)

    if not np.isfinite(db_value):
        raise ValueError(
            "db_value must be finite"
        )

    return float(
        10.0 ** (db_value / 10.0)
    )


def dbm_to_w(dbm_value):
    dbm_value = float(dbm_value)

    if not np.isfinite(dbm_value):
        raise ValueError(
            "dbm_value must be finite"
        )

    return float(
        10.0 ** (
            (dbm_value - 30.0) / 10.0
        )
    )


def calculate_link_snr(
    tx_power_w,
    distance_m,
    additional_loss_db=0.0,
):
    tx_power_w = float(tx_power_w)
    distance_m = float(distance_m)
    additional_loss_db = float(additional_loss_db)

    if (
        not np.isfinite(tx_power_w)
        or tx_power_w <= 0.0
    ):
        raise ValueError(
            "tx_power_w must be finite and > 0"
        )

    if (
        not np.isfinite(distance_m)
        or distance_m < 0.0
    ):
        raise ValueError(
            "distance_m must be finite and >= 0"
        )

    if (
        not np.isfinite(additional_loss_db)
        or additional_loss_db < 0.0
    ):
        raise ValueError(
            "additional_loss_db must be finite and >= 0"
        )

    power_min = float(
        CONFIG["tx_power_min_w"]
    )
    power_max = float(
        CONFIG["tx_power_max_w"]
    )

    if not (
        power_min
        <= tx_power_w
        <= power_max
    ):
        raise ValueError(
            "tx_power_w outside configured range"
        )

    reference_distance_m = float(
        CONFIG["comm_reference_distance_m"]
    )
    path_loss_exponent = float(
        CONFIG["comm_path_loss_exponent"]
    )

    if (
        not np.isfinite(reference_distance_m)
        or reference_distance_m <= 0.0
    ):
        raise ValueError(
            "comm_reference_distance_m "
            "must be finite and > 0"
        )

    if (
        not np.isfinite(path_loss_exponent)
        or path_loss_exponent <= 0.0
    ):
        raise ValueError(
            "comm_path_loss_exponent "
            "must be finite and > 0"
        )

    reference_gain_linear = db_to_linear(
        CONFIG["comm_reference_gain_db"]
    )

    noise_power_w = dbm_to_w(
        CONFIG["comm_noise_power_dbm"]
    )

    if (
        reference_gain_linear <= 0.0
        or noise_power_w <= 0.0
    ):
        raise ValueError(
            "invalid communication model"
        )

    effective_distance_m = max(
        distance_m,
        reference_distance_m,
    )

    channel_power_gain = (
        reference_gain_linear
        * (
            reference_distance_m
            / effective_distance_m
        )
        ** path_loss_exponent
    )

    additional_gain = db_to_linear(
        -additional_loss_db
    )

    received_power_w = (
        tx_power_w
        * channel_power_gain
        * additional_gain
    )

    snr_linear = (
        received_power_w
        / noise_power_w
    )

    return float(snr_linear)


def snr_linear_to_db(snr_linear):
    snr_linear = float(snr_linear)

    if not np.isfinite(snr_linear) or snr_linear < 0.0:
        raise ValueError(
            "snr_linear must be finite and >= 0"
        )

    if snr_linear == 0.0:
        return float("-inf")

    return float(
        10.0 * np.log10(snr_linear)
    )


In [ ]:
def transmission_distance_m(
    intent,
    uavs,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if not isinstance(uavs, list):
        raise TypeError(
            "uavs must be a list"
        )

    uavs_by_id = {
        int(uav.id): uav
        for uav in uavs
        if isinstance(uav, UAV)
    }

    if intent.sender not in uavs_by_id:
        raise ValueError(
            "sender UAV not found"
        )

    sender = uavs_by_id[
        intent.sender
    ]

    if intent.recipient == GCS_NODE:
        return distance_3d(
            sender.position,
            CONFIG["gcs_position"],
        )

    if intent.recipient not in uavs_by_id:
        raise ValueError(
            "recipient UAV not found"
        )

    recipient = uavs_by_id[
        intent.recipient
    ]

    return distance_3d(
        sender.position,
        recipient.position,
    )


def calculate_intent_snr(
    intent,
    uavs,
    obstacles=None,
):
    if obstacles is None:
        obstacles = []

    if not transmission_is_feasible(
        intent,
        uavs,
    ):
        return 0.0

    distance_m = transmission_distance_m(
        intent,
        uavs,
    )

    uavs_by_id = {
        int(uav.id): uav
        for uav in uavs
        if isinstance(uav, UAV)
    }

    sender_position = uavs_by_id[
        intent.sender
    ].position

    if intent.recipient == GCS_NODE:
        recipient_position = np.asarray(
            CONFIG["gcs_position"],
            dtype=np.float64,
        )
    else:
        recipient_position = uavs_by_id[
            intent.recipient
        ].position

    link_state = classify_link_state(
        sender_position,
        recipient_position,
        obstacles,
    )

    if link_state == LOS_LINK:
        additional_loss_db = 0.0
    else:
        additional_loss_db = float(
            CONFIG["comm_nlos_additional_loss_db"]
        )

    return calculate_link_snr(
        intent.tx_power_w,
        distance_m,
        additional_loss_db=additional_loss_db,
    )


In [ ]:
def calculate_link_rate_bps(snr_linear):
    snr_linear = float(snr_linear)

    if not np.isfinite(snr_linear) or snr_linear < 0.0:
        raise ValueError(
            "snr_linear must be finite and >= 0"
        )

    bandwidth_hz = float(
        CONFIG["comm_bandwidth_hz"]
    )

    if (
        not np.isfinite(bandwidth_hz)
        or bandwidth_hz <= 0.0
    ):
        raise ValueError(
            "comm_bandwidth_hz must be finite and > 0"
        )

    if snr_linear == 0.0:
        return 0.0

    return float(
        bandwidth_hz
        * np.log1p(snr_linear)
        / np.log(2.0)
    )


def calculate_intent_rate_bps(
    intent,
    uavs,
    obstacles=None,
):
    snr_linear = calculate_intent_snr(
        intent,
        uavs,
        obstacles=obstacles,
    )

    return calculate_link_rate_bps(
        snr_linear
    )


def calculate_intent_tx_bytes(
    intent,
    uavs,
    obstacles=None,
    dt=None,
):
    if dt is None:
        dt = CONFIG["dt"]

    dt = float(dt)

    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(
            "dt must be finite and > 0"
        )

    rate_bps = calculate_intent_rate_bps(
        intent,
        uavs,
        obstacles=obstacles,
    )

    capacity_bytes = (
        rate_bps
        * dt
        / 8.0
    )

    transferable_bytes = int(
        np.floor(capacity_bytes)
    )

    return min(
        int(intent.requested_bytes),
        transferable_bytes,
    )


In [56]:
@dataclass
class PeerTransferState:
    sender: int
    recipient: int
    target_id: int
    source_uav: int
    created_step: int
    size_bytes: int
    ttl_s: float
    received_bytes: int = 0

    def __post_init__(self):
        for name, value in (
            ("sender", self.sender),
            ("recipient", self.recipient),
            ("target_id", self.target_id),
            ("source_uav", self.source_uav),
            ("created_step", self.created_step),
            ("size_bytes", self.size_bytes),
            ("received_bytes", self.received_bytes),
        ):
            if (
                isinstance(value, (bool, np.bool_))
                or not isinstance(value, (int, np.integer))
            ):
                raise TypeError(
                    f"{name} must be an integer"
                )

        self.sender = int(self.sender)
        self.recipient = int(self.recipient)
        self.target_id = int(self.target_id)
        self.source_uav = int(self.source_uav)
        self.created_step = int(self.created_step)
        self.size_bytes = int(self.size_bytes)
        self.received_bytes = int(self.received_bytes)
        self.ttl_s = float(self.ttl_s)

        num_uavs = int(CONFIG["num_uavs"])

        if not 0 <= self.sender < num_uavs:
            raise ValueError("sender out of range")

        if not 0 <= self.recipient < num_uavs:
            raise ValueError("recipient out of range")

        if self.sender == self.recipient:
            raise ValueError(
                "sender and recipient must be different"
            )

        if self.target_id < 0:
            raise ValueError(
                "target_id must be >= 0"
            )

        if not 0 <= self.source_uav < num_uavs:
            raise ValueError(
                "source_uav out of range"
            )

        if self.created_step < 0:
            raise ValueError(
                "created_step must be >= 0"
            )

        if self.size_bytes <= 0:
            raise ValueError(
                "size_bytes must be > 0"
            )

        if (
            not np.isfinite(self.ttl_s)
            or self.ttl_s <= 0.0
        ):
            raise ValueError(
                "ttl_s must be finite and > 0"
            )

        if not (
            0
            <= self.received_bytes
            <= self.size_bytes
        ):
            raise ValueError(
                "received_bytes must be in "
                "[0, size_bytes]"
            )



In [ ]:
def create_peer_transfer_states():
    return {}


def peer_transfer_key(
    sender,
    recipient,
    target_id,
):
    for name, value in (
        ("sender", sender),
        ("recipient", recipient),
        ("target_id", target_id),
    ):
        if (
            isinstance(value, (bool, np.bool_))
            or not isinstance(value, (int, np.integer))
        ):
            raise TypeError(
                f"{name} must be an integer"
            )

    sender = int(sender)
    recipient = int(recipient)
    target_id = int(target_id)

    num_uavs = int(CONFIG["num_uavs"])

    if not 0 <= sender < num_uavs:
        raise ValueError(
            "sender out of range"
        )

    if not 0 <= recipient < num_uavs:
        raise ValueError(
            "recipient out of range"
        )

    if sender == recipient:
        raise ValueError(
            "sender and recipient must be different"
        )

    if target_id < 0:
        raise ValueError(
            "target_id must be >= 0"
        )

    return (
        sender,
        recipient,
        target_id,
    )

In [ ]:
def peer_transfer_remaining_bytes(state):
    if not isinstance(
        state,
        PeerTransferState,
    ):
        raise TypeError(
            "state must be PeerTransferState"
        )

    return max(
        0,
        int(
            state.size_bytes
            - state.received_bytes
        ),
    )


def peer_transfer_is_complete(state):
    return (peer_transfer_remaining_bytes(state)== 0)


def peer_transfer_age_s(
    state,
    current_step,
):
    if not isinstance(
        state,
        PeerTransferState,
    ):
        raise TypeError(
            "state must be PeerTransferState"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(current_step)

    if current_step < state.created_step:
        raise ValueError(
            "current_step must be >= "
            "state.created_step"
        )

    dt = float(CONFIG["dt"])

    if (
        not np.isfinite(dt)
        or dt <= 0.0
    ):
        raise ValueError(
            "CONFIG['dt'] must be finite and > 0"
        )

    return float((current_step - state.created_step)* dt)

In [ ]:
def peer_transfer_is_expired(
    state,
    current_step,
):
    return peer_transfer_age_s(state,current_step)>= state.ttl_s


def find_report_in_buffer(
    buffer,
    target_id,
):
    if not isinstance(buffer, list):
        raise TypeError("buffer must be a list")

    if (
        isinstance(target_id, (bool, np.bool_))
        or not isinstance(
            target_id,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "target_id must be an integer"
        )

    target_id = int(target_id)

    for report in buffer:
        if not isinstance(
            report,
            Report,
        ):
            raise TypeError(
                "buffer must contain only "
                "Report objects"
            )

        if report.target_id == target_id:
            return report

    return None





In [ ]:
def get_or_create_peer_transfer_state(
    intent,
    sender_buffer,
    peer_transfer_states,
    current_step,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if intent.recipient == GCS_NODE:
        raise ValueError(
            "peer transfer state is only "
            "for UAV-to-UAV transmissions"
        )

    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    report = find_report_in_buffer(
        sender_buffer,
        intent.target_id,
    )

    if report is None:
        raise ValueError(
            "intent target report not found "
            "in sender buffer"
        )

    if report_is_expired(
        report,
        current_step,
    ):
        raise ValueError(
            "cannot create transfer state "
            "for an expired report"
        )

    key = peer_transfer_key(
        intent.sender,
        intent.recipient,
        intent.target_id,
    )

    state = peer_transfer_states.get(
        key
    )

    if state is not None:
        if not isinstance(
            state,
            PeerTransferState,
        ):
            raise TypeError(
                "peer_transfer_states values "
                "must be PeerTransferState"
            )

        # A stale partial copy must not block a fresh report with the
        # same target_id after the original report lifetime expires.
        if peer_transfer_is_expired(
            state,
            current_step,
        ):
            del peer_transfer_states[key]
            state = None

    if state is not None:
        if (
            state.source_uav
            != report.source_uav
            or state.created_step
            != report.created_step
            or state.size_bytes
            != report.size_bytes
            or not np.isclose(
                state.ttl_s,
                report.ttl_s,
            )
        ):
            raise ValueError(
                "existing peer transfer state "
                "does not match sender report"
            )

        return state

    state = PeerTransferState(
        sender=int(intent.sender),
        recipient=int(intent.recipient),
        target_id=int(intent.target_id),
        source_uav=int(report.source_uav),
        created_step=int(report.created_step),
        size_bytes=int(report.size_bytes),
        ttl_s=float(report.ttl_s),
        received_bytes=0,
    )

    peer_transfer_states[key] = state

    return state

In [57]:
def commit_peer_transfer_bytes(
    intent,
    tx_bytes,
    sender_buffer,
    peer_transfer_states,
    current_step,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if (
        isinstance(tx_bytes, (bool, np.bool_))
        or not isinstance(
            tx_bytes,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "tx_bytes must be an integer"
        )

    tx_bytes = int(tx_bytes)

    if tx_bytes < 0:
        raise ValueError(
            "tx_bytes must be >= 0"
        )

    if tx_bytes > intent.requested_bytes:
        raise ValueError(
            "tx_bytes cannot exceed "
            "intent.requested_bytes"
        )

    state = get_or_create_peer_transfer_state(
        intent,
        sender_buffer,
        peer_transfer_states,
        current_step,
    )

    if peer_transfer_is_expired(
        state,
        current_step,
    ):
        raise ValueError(
            "cannot commit bytes to an "
            "expired peer transfer"
        )

    remaining = (
        peer_transfer_remaining_bytes(
            state
        )
    )

    committed_bytes = min(
        tx_bytes,
        remaining,
    )

    state.received_bytes += (
        committed_bytes
    )

    return {
        "state": state,
        "committed_bytes": (
            committed_bytes
        ),
        "remaining_bytes": (
            peer_transfer_remaining_bytes(
                state
            )
        ),
        "complete": (
            peer_transfer_is_complete(
                state
            )
        ),
    }

In [ ]:
def cleanup_peer_transfer_states(
    peer_transfer_states,
    current_step,
):
    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    expired_keys = []

    for key, state in list(
        peer_transfer_states.items()
    ):
        if not isinstance(
            state,
            PeerTransferState,
        ):
            raise TypeError(
                "peer_transfer_states values "
                "must be PeerTransferState"
            )

        if peer_transfer_is_expired(
            state,
            current_step,
        ):
            expired_keys.append(key)
            del peer_transfer_states[key]

    return expired_keys




def known_gcs_delivered_bytes_for_transfer(
    state,
    report_buffers,
):
    if not isinstance(
        state,
        PeerTransferState,
    ):
        raise TypeError(
            "state must be PeerTransferState"
        )

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    known_bytes = 0

    for buffer in report_buffers:
        if not isinstance(
            buffer,
            list,
        ):
            raise TypeError(
                "each report buffer must be a list"
            )

        for report in buffer:
            if not isinstance(
                report,
                Report,
            ):
                raise TypeError(
                    "report buffers must contain "
                    "only Report objects"
                )

            if report.target_id != state.target_id:
                continue

            # Only merge progress from the same report generation.
            if (
                report.source_uav
                != state.source_uav
                or report.created_step
                != state.created_step
                or report.size_bytes
                != state.size_bytes
                or not np.isclose(
                    report.ttl_s,
                    state.ttl_s,
                )
            ):
                continue

            known_bytes = max(
                known_bytes,
                int(
                    report.delivered_bytes
                ),
            )

    return min(
        known_bytes,
        int(state.size_bytes),
    )

def commit_completed_peer_transfer_to_receiver(
    state_key,
    peer_transfer_states,
    report_buffers,
    current_step,
    gcs_received_target_ids=None,
):
    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    num_uavs = int(
        CONFIG["num_uavs"]
    )

    if len(report_buffers) != num_uavs:
        raise ValueError(
            "report_buffers length does not match "
            "CONFIG['num_uavs']"
        )

    for buffer in report_buffers:
        if not isinstance(buffer, list):
            raise TypeError(
                "each report buffer must be a list"
            )

    if gcs_received_target_ids is None:
        gcs_received_target_ids = set()

    if not isinstance(
        gcs_received_target_ids,
        set,
    ):
        raise TypeError(
            "gcs_received_target_ids must be a set"
        )

    if (
        not isinstance(state_key, tuple)
        or len(state_key) != 3
    ):
        raise TypeError(
            "state_key must be a "
            "(sender, recipient, target_id) tuple"
        )

    key = peer_transfer_key(
        state_key[0],
        state_key[1],
        state_key[2],
    )

    state = peer_transfer_states.get(
        key
    )

    if state is None:
        raise ValueError(
            "peer transfer state not found"
        )

    if not isinstance(
        state,
        PeerTransferState,
    ):
        raise TypeError(
            "peer_transfer_states values must "
            "be PeerTransferState"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(current_step)

    if not peer_transfer_is_complete(
        state
    ):
        return {
            "status": "incomplete",
            "report": None,
            "remaining_bytes": (
                peer_transfer_remaining_bytes(
                    state
                )
            ),
        }

    if peer_transfer_is_expired(
        state,
        current_step,
    ):
        del peer_transfer_states[key]

        return {
            "status": "expired",
            "report": None,
            "remaining_bytes": 0,
        }

    if (
        state.target_id
        in gcs_received_target_ids
    ):
        del peer_transfer_states[key]

        return {
            "status": "already_delivered",
            "report": None,
            "remaining_bytes": 0,
        }

    receiver_buffer = report_buffers[
        state.recipient
    ]

    cleanup_report_buffer(
        receiver_buffer,
        current_step,
    )

    known_gcs_delivered_bytes = (
        known_gcs_delivered_bytes_for_transfer(
            state,
            report_buffers,
        )
    )

    existing_report = find_report_in_buffer(
        receiver_buffer,
        state.target_id,
    )

    if existing_report is not None:
        existing_report.delivered_bytes = max(
            int(existing_report.delivered_bytes),
            int(known_gcs_delivered_bytes),
        )
        # The receiver already has a full copy. Keep the completed
        # transfer state so this sender does not retransmit it.
        return {
            "status": "already_present",
            "report": existing_report,
            "remaining_bytes": 0,
        }

    received_report = Report(
        target_id=state.target_id,
        source_uav=state.source_uav,
        created_step=state.created_step,
        size_bytes=state.size_bytes,
        ttl_s=state.ttl_s,
        delivered_bytes=(
            known_gcs_delivered_bytes
        ),
    )

    enqueued, reason = enqueue_report(
        receiver_buffer,
        received_report,
    )

    if enqueued:
        # Keep the completed state as a lightweight receipt that this
        # sender already transferred the full report to this recipient.
        # build_transmission_intent() will therefore return None for the
        # same hop instead of sending the same report again.
        return {
            "status": "enqueued",
            "report": received_report,
            "remaining_bytes": 0,
        }

    if reason == "buffer_full":
        return {
            "status": "buffer_full",
            "report": None,
            "remaining_bytes": 0,
        }

    if reason == "duplicate":
        existing_report = find_report_in_buffer(
            receiver_buffer,
            state.target_id,
        )

        return {
            "status": "already_present",
            "report": existing_report,
            "remaining_bytes": 0,
        }

    raise RuntimeError(
        f"unexpected enqueue result: {reason}"
    )

In [ ]:
def reports_are_same_generation(
    report_a,
    report_b,
):
    if not isinstance(
        report_a,
        Report,
    ):
        raise TypeError(
            "report_a must be Report"
        )

    if not isinstance(
        report_b,
        Report,
    ):
        raise TypeError(
            "report_b must be Report"
        )

    return bool(
        report_a.target_id
        == report_b.target_id
        and report_a.source_uav
        == report_b.source_uav
        and report_a.created_step
        == report_b.created_step
        and report_a.size_bytes
        == report_b.size_bytes
        and np.isclose(
            report_a.ttl_s,
            report_b.ttl_s,
        )
    )


def remove_peer_transfer_states_for_target(
    peer_transfer_states,
    target_id,
):
    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    if (
        isinstance(target_id, (bool, np.bool_))
        or not isinstance(
            target_id,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "target_id must be an integer"
        )

    target_id = int(target_id)

    if target_id < 0:
        raise ValueError(
            "target_id must be >= 0"
        )

    removed_keys = []

    for key, state in list(
        peer_transfer_states.items()
    ):
        if not isinstance(
            state,
            PeerTransferState,
        ):
            raise TypeError(
                "peer_transfer_states values "
                "must be PeerTransferState"
            )

        if state.target_id == target_id:
            removed_keys.append(key)
            del peer_transfer_states[key]

    return removed_keys


def commit_gcs_transfer_bytes(
    intent,
    tx_bytes,
    report_buffers,
    pending_reports,
    gcs_received_target_ids,
    peer_transfer_states,
    current_step,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if intent.recipient != GCS_NODE:
        raise ValueError(
            "commit_gcs_transfer_bytes "
            "requires a GCS intent"
        )

    if (
        isinstance(tx_bytes, (bool, np.bool_))
        or not isinstance(
            tx_bytes,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "tx_bytes must be an integer"
        )

    tx_bytes = int(tx_bytes)

    if tx_bytes < 0:
        raise ValueError(
            "tx_bytes must be >= 0"
        )

    if tx_bytes > intent.requested_bytes:
        raise ValueError(
            "tx_bytes cannot exceed "
            "intent.requested_bytes"
        )

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    if len(report_buffers) != int(
        CONFIG["num_uavs"]
    ):
        raise ValueError(
            "report_buffers length does not "
            "match CONFIG['num_uavs']"
        )

    for buffer in report_buffers:
        if not isinstance(
            buffer,
            list,
        ):
            raise TypeError(
                "each report buffer "
                "must be a list"
            )

        for report in buffer:
            if not isinstance(
                report,
                Report,
            ):
                raise TypeError(
                    "report buffers must contain "
                    "only Report objects"
                )

    if not isinstance(
        pending_reports,
        list,
    ):
        raise TypeError(
            "pending_reports must be a list"
        )

    for report in pending_reports:
        if not isinstance(
            report,
            Report,
        ):
            raise TypeError(
                "pending_reports must contain "
                "only Report objects"
            )

    if not isinstance(
        gcs_received_target_ids,
        set,
    ):
        raise TypeError(
            "gcs_received_target_ids "
            "must be a set"
        )

    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(
        current_step
    )

    if current_step < 0:
        raise ValueError(
            "current_step must be >= 0"
        )

    if (
        intent.target_id
        in gcs_received_target_ids
    ):
        # Make this path idempotent: if stale network copies remain for
        # any reason, calling the commit again restores the invariant
        # that a GCS-delivered target has no buffered/pending copies.
        cleanup_result = (
            mark_target_delivered_to_gcs(
                intent.target_id,
                gcs_received_target_ids,
                report_buffers,
                pending_reports,
            )
        )

        removed_peer_states = (
            remove_peer_transfer_states_for_target(
                peer_transfer_states,
                intent.target_id,
            )
        )

        return {
            "status": "already_delivered",
            "committed_bytes": 0,
            "delivered_bytes": 0,
            "remaining_bytes": 0,
            "removed_peer_states": (
                removed_peer_states
            ),
            "cleanup_result": (
                cleanup_result
            ),
        }

    sender_buffer = report_buffers[
        intent.sender
    ]

    sender_report = (
        find_report_in_buffer(
            sender_buffer,
            intent.target_id,
        )
    )

    if sender_report is None:
        raise ValueError(
            "intent target report "
            "not found in sender buffer"
        )

    if report_is_expired(
        sender_report,
        current_step,
    ):
        raise ValueError(
            "cannot commit an "
            "expired report to GCS"
        )

    report_copies = []

    for buffer in report_buffers:
        for report in buffer:
            if reports_are_same_generation(
                report,
                sender_report,
            ):
                report_copies.append(
                    report
                )

    if not report_copies:
        raise RuntimeError(
            "no matching report copies found"
        )

    known_delivered_bytes = max(
        int(report.delivered_bytes)
        for report in report_copies
    )

    remaining_bytes = max(
        0,
        int(sender_report.size_bytes)
        - known_delivered_bytes,
    )

    committed_bytes = min(
        tx_bytes,
        remaining_bytes,
    )

    new_delivered_bytes = (
        known_delivered_bytes
        + committed_bytes
    )

    for report in report_copies:
        report.delivered_bytes = (
            new_delivered_bytes
        )

    complete = (
        new_delivered_bytes
        >= sender_report.size_bytes
    )

    removed_peer_states = []

    if complete:
        mark_target_delivered_to_gcs(
            intent.target_id,
            gcs_received_target_ids,
            report_buffers,
            pending_reports,
        )

        removed_peer_states = (
            remove_peer_transfer_states_for_target(
                peer_transfer_states,
                intent.target_id,
            )
        )

    return {
        "status": (
            "delivered"
            if complete
            else "partial"
        ),
        "committed_bytes": (
            committed_bytes
        ),
        "delivered_bytes": (
            new_delivered_bytes
        ),
        "remaining_bytes": max(
            0,
            int(sender_report.size_bytes)
            - new_delivered_bytes,
        ),
        "removed_peer_states": (
            removed_peer_states
        ),
    }


In [ ]:
def retry_completed_peer_transfers(
    peer_transfer_states,
    report_buffers,
    current_step,
    gcs_received_target_ids,
):
    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    if len(report_buffers) != int(
        CONFIG["num_uavs"]
    ):
        raise ValueError(
            "report_buffers length does not "
            "match CONFIG['num_uavs']"
        )

    for buffer in report_buffers:
        if not isinstance(buffer, list):
            raise TypeError(
                "each report buffer must be a list"
            )

    if not isinstance(
        gcs_received_target_ids,
        set,
    ):
        raise TypeError(
            "gcs_received_target_ids "
            "must be a set"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(current_step)

    if current_step < 0:
        raise ValueError(
            "current_step must be >= 0"
        )

    results = []

    for key, state in list(
        peer_transfer_states.items()
    ):
        if not isinstance(
            state,
            PeerTransferState,
        ):
            raise TypeError(
                "peer_transfer_states values "
                "must be PeerTransferState"
            )

        if not peer_transfer_is_complete(
            state
        ):
            continue

        receiver_buffer = report_buffers[
            state.recipient
        ]

        # Remove stale receiver copies before deciding whether the
        # completed state can act as a receipt.
        cleanup_report_buffer(
            receiver_buffer,
            current_step,
        )

        # Expiration and global GCS delivery take precedence over receipt
        # retention. commit_completed_peer_transfer_to_receiver() removes
        # the completed state in both cases.
        if (
            peer_transfer_is_expired(
                state,
                current_step,
            )
            or state.target_id
            in gcs_received_target_ids
        ):
            result = (
                commit_completed_peer_transfer_to_receiver(
                    key,
                    peer_transfer_states,
                    report_buffers,
                    current_step,
                    gcs_received_target_ids,
                )
            )

            results.append(
                {
                    "key": key,
                    "status": result["status"],
                    "result": result,
                }
            )
            continue

        # A non-expired completed state is also used as a receipt after
        # the receiver has a full copy. Do not repeatedly re-commit it.
        existing_report = find_report_in_buffer(
            receiver_buffer,
            state.target_id,
        )

        if (
            existing_report is not None
            and reports_are_same_generation(
                existing_report,
                Report(
                    target_id=state.target_id,
                    source_uav=state.source_uav,
                    created_step=state.created_step,
                    size_bytes=state.size_bytes,
                    ttl_s=state.ttl_s,
                    delivered_bytes=0,
                ),
            )
        ):
            results.append(
                {
                    "key": key,
                    "status": "receipt_present",
                }
            )
            continue

        result = (
            commit_completed_peer_transfer_to_receiver(
                key,
                peer_transfer_states,
                report_buffers,
                current_step,
                gcs_received_target_ids,
            )
        )

        results.append(
            {
                "key": key,
                "status": result["status"],
                "result": result,
            }
        )

    return results


def execute_transmission_intent(
    intent,
    uavs,
    report_buffers,
    pending_reports,
    gcs_received_target_ids,
    peer_transfer_states,
    current_step,
    obstacles=None,
    dt=None,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if not isinstance(uavs, list):
        raise TypeError(
            "uavs must be a list"
        )

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    if len(report_buffers) != int(
        CONFIG["num_uavs"]
    ):
        raise ValueError(
            "report_buffers length does not "
            "match CONFIG['num_uavs']"
        )

    if not isinstance(
        pending_reports,
        list,
    ):
        raise TypeError(
            "pending_reports must be a list"
        )

    if not isinstance(
        gcs_received_target_ids,
        set,
    ):
        raise TypeError(
            "gcs_received_target_ids "
            "must be a set"
        )

    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(current_step)

    if current_step < 0:
        raise ValueError(
            "current_step must be >= 0"
        )

    if obstacles is None:
        obstacles = []

    if not isinstance(obstacles, list):
        raise TypeError(
            "obstacles must be a list"
        )

    retry_results = (
        retry_completed_peer_transfers(
            peer_transfer_states,
            report_buffers,
            current_step,
            gcs_received_target_ids,
        )
    )

    # A stale intent can survive after another copy reaches GCS.
    # Restore the global-delivery invariant before doing any radio work.
    if (
        intent.target_id
        in gcs_received_target_ids
    ):
        cleanup_result = (
            mark_target_delivered_to_gcs(
                intent.target_id,
                gcs_received_target_ids,
                report_buffers,
                pending_reports,
            )
        )

        removed_peer_states = (
            remove_peer_transfer_states_for_target(
                peer_transfer_states,
                intent.target_id,
            )
        )

        return {
            "status": "already_delivered",
            "tx_bytes": 0,
            "retry_results": retry_results,
            "cleanup_result": cleanup_result,
            "removed_peer_states": (
                removed_peer_states
            ),
        }

    tx_bytes = calculate_intent_tx_bytes(
        intent,
        uavs,
        obstacles=obstacles,
        dt=dt,
    )

    if tx_bytes <= 0:
        return {
            "status": "no_transfer",
            "tx_bytes": 0,
            "retry_results": retry_results,
        }

    if intent.recipient == GCS_NODE:
        commit_result = (
            commit_gcs_transfer_bytes(
                intent,
                tx_bytes,
                report_buffers,
                pending_reports,
                gcs_received_target_ids,
                peer_transfer_states,
                current_step,
            )
        )

        return {
            "status": (
                "gcs_"
                + commit_result["status"]
            ),
            "tx_bytes": tx_bytes,
            "retry_results": retry_results,
            "commit_result": commit_result,
        }

    sender_buffer = report_buffers[
        intent.sender
    ]

    peer_result = (
        commit_peer_transfer_bytes(
            intent,
            tx_bytes,
            sender_buffer,
            peer_transfer_states,
            current_step,
        )
    )

    custody_result = None

    if peer_result["complete"]:
        key = peer_transfer_key(
            intent.sender,
            intent.recipient,
            intent.target_id,
        )

        custody_result = (
            commit_completed_peer_transfer_to_receiver(
                key,
                peer_transfer_states,
                report_buffers,
                current_step,
                gcs_received_target_ids,
            )
        )

    if custody_result is None:
        status = "peer_partial"
    else:
        status = (
            "peer_"
            + custody_result["status"]
        )

    return {
        "status": status,
        "tx_bytes": tx_bytes,
        "retry_results": retry_results,
        "peer_result": peer_result,
        "custody_result": custody_result,
    }


In [ ]:
class NetworkBackend(ABC):
    """Backend-neutral interface for mission-level network simulation."""

    @abstractmethod
    def reset(
        self,
        *,
        uavs,
        gcs_position,
        obstacles,
        report_buffers,
        pending_reports,
        gcs_received_target_ids,
    ):
        """Reset backend state and bind the current mission state."""
        raise NotImplementedError

    @abstractmethod
    def sync_positions(self, uavs):
        """Synchronize backend node positions with mission state."""
        raise NotImplementedError

    @abstractmethod
    def step(
        self,
        requests,
        dt,
        current_step,
    ):
        """Execute one mission-step worth of network requests."""
        raise NotImplementedError

    @abstractmethod
    def metrics(self):
        """Return backend metrics without mutating backend state."""
        raise NotImplementedError

    @abstractmethod
    def last_step_communication_energy_by_uav(self):
        """Return communication energy used by each mission UAV last step."""
        raise NotImplementedError


In [ ]:
class SimpleNetworkBackend(NetworkBackend):
    """Thin backend wrapper around the existing simple-network primitives."""

    def __init__(self):
        self._initialized = False
        self._uavs = None
        self._gcs_position = None
        self._obstacles = None
        self._report_buffers = None
        self._pending_reports = None
        self._gcs_received_target_ids = None
        self._peer_transfer_states = create_peer_transfer_states()
        self._metric_state = self._new_metric_state()
        self._last_step_comm_energy_by_uav = np.zeros(
            int(CONFIG["num_uavs"]),
            dtype=np.float64,
        )

    @staticmethod
    def _new_metric_state():
        return {
            "steps": 0,
            "requests": 0,
            "tx_bytes": 0,
            "communication_energy_j": 0.0,
            "expired_peer_states": 0,
            "custody_retry_events": 0,
            "status_counts": {},
        }

    @staticmethod
    def _validate_uavs(uavs):
        if not isinstance(uavs, list):
            raise TypeError("uavs must be a list")

        num_uavs = int(CONFIG["num_uavs"])

        if len(uavs) != num_uavs:
            raise ValueError(
                "uavs length does not match CONFIG['num_uavs']"
            )

        for expected_id, uav in enumerate(uavs):
            if not isinstance(uav, UAV):
                raise TypeError(
                    "uavs must contain only UAV objects"
                )

            if int(uav.id) != expected_id:
                raise ValueError(
                    "uavs must be ordered by contiguous UAV id"
                )

    @staticmethod
    def _validate_obstacles(obstacles):
        if not isinstance(obstacles, list):
            raise TypeError("obstacles must be a list")

        for obstacle in obstacles:
            if not isinstance(obstacle, Obstacle):
                raise TypeError(
                    "obstacles must contain only Obstacle objects"
                )

    @staticmethod
    def _validate_report_state(
        report_buffers,
        pending_reports,
        gcs_received_target_ids,
    ):
        if not isinstance(report_buffers, list):
            raise TypeError(
                "report_buffers must be a list"
            )

        if len(report_buffers) != int(
            CONFIG["num_uavs"]
        ):
            raise ValueError(
                "report_buffers length does not match "
                "CONFIG['num_uavs']"
            )

        for buffer in report_buffers:
            if not isinstance(buffer, list):
                raise TypeError(
                    "each report buffer must be a list"
                )

            for report in buffer:
                if not isinstance(report, Report):
                    raise TypeError(
                        "report buffers must contain "
                        "only Report objects"
                    )

        if not isinstance(pending_reports, list):
            raise TypeError(
                "pending_reports must be a list"
            )

        for report in pending_reports:
            if not isinstance(report, Report):
                raise TypeError(
                    "pending_reports must contain "
                    "only Report objects"
                )

        if not isinstance(
            gcs_received_target_ids,
            set,
        ):
            raise TypeError(
                "gcs_received_target_ids must be a set"
            )

    @staticmethod
    def _validated_gcs_position(gcs_position):
        position = np.asarray(
            gcs_position,
            dtype=np.float64,
        )

        if position.shape != (3,):
            raise ValueError(
                "gcs_position must have shape (3,)"
            )

        if not np.all(np.isfinite(position)):
            raise ValueError(
                "gcs_position must contain only finite values"
            )

        configured_position = np.asarray(
            CONFIG["gcs_position"],
            dtype=np.float64,
        )

        # Existing simple-network primitives read GCS position from
        # CONFIG directly. Reject a mismatch instead of silently using
        # two different GCS locations.
        if not np.allclose(
            position,
            configured_position,
            rtol=0.0,
            atol=1e-12,
        ):
            raise ValueError(
                "SimpleNetworkBackend gcs_position must match "
                "CONFIG['gcs_position']"
            )

        return position.copy()

    def reset(
        self,
        *,
        uavs,
        gcs_position,
        obstacles,
        report_buffers,
        pending_reports,
        gcs_received_target_ids,
    ):
        self._validate_uavs(uavs)
        self._validate_obstacles(obstacles)
        self._validate_report_state(
            report_buffers,
            pending_reports,
            gcs_received_target_ids,
        )

        self._uavs = uavs
        self._gcs_position = (
            self._validated_gcs_position(
                gcs_position
            )
        )
        self._obstacles = obstacles
        self._report_buffers = report_buffers
        self._pending_reports = pending_reports
        self._gcs_received_target_ids = (
            gcs_received_target_ids
        )
        self._peer_transfer_states = (
            create_peer_transfer_states()
        )
        self._metric_state = (
            self._new_metric_state()
        )
        self._last_step_comm_energy_by_uav = np.zeros(
            int(CONFIG["num_uavs"]),
            dtype=np.float64,
        )
        self._initialized = True

    def sync_positions(self, uavs):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before sync_positions"
            )

        self._validate_uavs(uavs)
        self._uavs = uavs

    def step(
        self,
        requests,
        dt,
        current_step,
    ):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before step"
            )

        if (
            isinstance(current_step, (bool, np.bool_))
            or not isinstance(
                current_step,
                (int, np.integer),
            )
        ):
            raise TypeError(
                "current_step must be an integer"
            )

        current_step = int(current_step)

        if current_step < 0:
            raise ValueError(
                "current_step must be >= 0"
            )

        if isinstance(dt, (bool, np.bool_)):
            raise TypeError(
                "dt must be numeric and not boolean"
            )

        dt = float(dt)

        if not np.isfinite(dt) or dt <= 0.0:
            raise ValueError(
                "dt must be finite and > 0"
            )

        # Report/peer TTL helpers currently use CONFIG['dt'].
        # Enforce one mission clock until those helpers are made
        # backend-neutral.
        if not np.isclose(
            dt,
            float(CONFIG["dt"]),
            rtol=0.0,
            atol=1e-12,
        ):
            raise ValueError(
                "SimpleNetworkBackend dt must match CONFIG['dt']"
            )

        if requests is None:
            request_list = []
        elif isinstance(
            requests,
            TransmissionIntent,
        ):
            request_list = [requests]
        elif isinstance(requests, (list, tuple)):
            request_list = list(requests)
        else:
            raise TypeError(
                "requests must be a TransmissionIntent, "
                "a list/tuple of intents, or None"
            )

        for request in request_list:
            if not isinstance(
                request,
                TransmissionIntent,
            ):
                raise TypeError(
                    "requests must contain only "
                    "TransmissionIntent objects"
                )

        sender_ids = [
            int(request.sender)
            for request in request_list
        ]

        if len(sender_ids) != len(
            set(sender_ids)
        ):
            raise ValueError(
                "each sender may submit at most one "
                "transmission request per network step"
            )

        expired_peer_states = (
            cleanup_peer_transfer_states(
                self._peer_transfer_states,
                current_step,
            )
        )
        custody_retry_results = (
            retry_completed_peer_transfers(
                self._peer_transfer_states,
                self._report_buffers,
                current_step,
                self._gcs_received_target_ids,
            )
        )

        self._metric_state[
            "expired_peer_states"
        ] += len(expired_peer_states)
        self._metric_state[
            "custody_retry_events"
        ] += len(custody_retry_results)

        self._last_step_comm_energy_by_uav.fill(
            0.0
        )

        results = []

        for request in request_list:
            result = execute_transmission_intent(
                request,
                self._uavs,
                self._report_buffers,
                self._pending_reports,
                self._gcs_received_target_ids,
                self._peer_transfer_states,
                current_step,
                obstacles=self._obstacles,
                dt=dt,
            )

            results.append(result)

            status = str(
                result.get(
                    "status",
                    "unknown",
                )
            )
            status_counts = self._metric_state[
                "status_counts"
            ]
            status_counts[status] = (
                status_counts.get(status, 0)
                + 1
            )
            tx_bytes = int(
                result.get(
                    "tx_bytes",
                    0,
                )
            )
            self._metric_state["tx_bytes"] += tx_bytes

            if (
                tx_bytes > 0
                and request.tx_power_w > 0.0
            ):
                rate_bps = calculate_intent_rate_bps(
                    request,
                    self._uavs,
                    obstacles=self._obstacles,
                )

                if rate_bps > 0.0:
                    duration_s = min(
                        dt,
                        (
                            tx_bytes
                            * 8.0
                            / rate_bps
                        ),
                    )
                    energy_j = (
                        float(request.tx_power_w)
                        * duration_s
                    )

                    self._last_step_comm_energy_by_uav[
                        request.sender
                    ] += energy_j
                    self._metric_state[
                        "communication_energy_j"
                    ] += energy_j

        self._metric_state["steps"] += 1
        self._metric_state["requests"] += len(
            request_list
        )

        return results

    @property
    def peer_transfer_states(self):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before accessing "
                "peer_transfer_states"
            )

        return self._peer_transfer_states

    def metrics(self):
        status_counts = dict(
            self._metric_state[
                "status_counts"
            ]
        )

        return {
            "steps": int(
                self._metric_state["steps"]
            ),
            "requests": int(
                self._metric_state["requests"]
            ),
            "tx_bytes": int(
                self._metric_state["tx_bytes"]
            ),
            "communication_energy_j": float(
                self._metric_state[
                    "communication_energy_j"
                ]
            ),
            "expired_peer_states": int(
                self._metric_state[
                    "expired_peer_states"
                ]
            ),
            "custody_retry_events": int(
                self._metric_state[
                    "custody_retry_events"
                ]
            ),
            "status_counts": status_counts,
            "active_peer_transfers": len(
                self._peer_transfer_states
            ),
        }

    def last_step_communication_energy_by_uav(self):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before "
                "reading communication energy"
            )

        return (
            self._last_step_comm_energy_by_uav
            .copy()
        )


In [ ]:
def resolve_uavnetsim_repo_path(repo_path=None):
    candidates = [
        repo_path,
        CONFIG.get("uavnetsim_path"),
        os.environ.get("UAVNETSIM_PATH"),
    ]

    checked = []

    for candidate in candidates:
        if candidate in (None, ""):
            continue

        path = Path(candidate).expanduser().resolve()
        checked.append(str(path))

        required = (
            path / "simulator" / "simulator.py",
            path / "entities" / "packet.py",
            path / "utils" / "config.py",
        )

        if all(item.is_file() for item in required):
            return path

    suffix = (
        f" Checked: {checked}."
        if checked
        else ""
    )

    raise FileNotFoundError(
        "UavNetSim repository was not found. "
        "Pass repo_path=... to UavNetSimBackend, set "
        "CONFIG['uavnetsim_path'], or set UAVNETSIM_PATH."
        + suffix
    )


def uavnetsim_available(repo_path=None):
    try:
        resolve_uavnetsim_repo_path(
            repo_path
        )
    except FileNotFoundError:
        return False

    return True


def _uavnetsim_scene_feature(
    obstacle,
    obstacle_id,
):
    if not isinstance(obstacle, Obstacle):
        raise TypeError(
            "obstacle must be an Obstacle"
        )

    sides = int(
        CONFIG[
            "uavnetsim_obstacle_polygon_sides"
        ]
    )

    if sides < 8:
        raise ValueError(
            "uavnetsim_obstacle_polygon_sides "
            "must be >= 8"
        )

    cx = float(obstacle.position[0])
    cy = float(obstacle.position[1])
    radius = float(obstacle.radius)

    footprint = []

    for index in range(sides):
        angle = (
            2.0
            * np.pi
            * index
            / sides
        )

        footprint.append(
            {
                "x": float(
                    cx
                    + radius
                    * np.cos(angle)
                ),
                "y": float(
                    cy
                    + radius
                    * np.sin(angle)
                ),
                "z": 0.0,
            }
        )

    return {
        "id": f"mission-obstacle-{obstacle_id}",
        "category": "building",
        "height": float(obstacle.height),
        "material": "itu_concrete",
        "source": "uav_search_target",
        "footprint": footprint,
    }


def build_uavnetsim_scene_payload(
    obstacles,
):
    if not isinstance(obstacles, list):
        raise TypeError(
            "obstacles must be a list"
        )

    features = [
        _uavnetsim_scene_feature(
            obstacle,
            obstacle_id,
        )
        for obstacle_id, obstacle
        in enumerate(obstacles)
    ]

    map_size = float(
        CONFIG["map_size"]
    )

    if (
        not np.isfinite(map_size)
        or map_size <= 0.0
    ):
        raise ValueError(
            "CONFIG['map_size'] must be "
            "finite and > 0"
        )

    return {
        "schema_version": 1,
        "name": "uav-search-target",
        "anchor": {
            "latitude": 0.0,
            "longitude": 0.0,
        },
        "size_x": map_size,
        "size_y": map_size,
        "features": features,
    }


def write_uavnetsim_scene(
    obstacles,
    directory,
):
    directory = Path(directory)
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    scene_path = (
        directory / "scene.json"
    )

    payload = (
        build_uavnetsim_scene_payload(
            obstacles
        )
    )

    scene_path.write_text(
        json.dumps(
            payload,
            indent=2,
        ),
        encoding="utf-8",
    )

    return scene_path


In [ ]:
class MarlSelectedNextHopRouting:
    """Routing shim that preserves the MARL-selected mission next hop."""

    def __init__(
        self,
        simulator,
        my_drone,
    ):
        self.simulator = simulator
        self.my_drone = my_drone

    def next_hop_selection(
        self,
        packet,
    ):
        enquire = False

        next_hop_id = getattr(
            packet,
            "forced_next_hop_id",
            packet.dst_drone.identifier,
        )

        try:
            next_hop_id = int(
                next_hop_id
            )
        except (TypeError, ValueError):
            return (
                False,
                packet,
                enquire,
            )

        if not (
            0
            <= next_hop_id
            < len(self.simulator.drones)
        ):
            return (
                False,
                packet,
                enquire,
            )

        if (
            next_hop_id
            == self.my_drone.identifier
        ):
            return (
                False,
                packet,
                enquire,
            )

        packet.next_hop_id = (
            next_hop_id
        )

        if (
            self.my_drone.identifier
            not in packet.intermediate_drones
        ):
            packet.intermediate_drones.append(
                self.my_drone.identifier
            )

        return (
            True,
            packet,
            enquire,
        )

    def packet_reception(
        self,
        packet,
        src_drone_id,
    ):
        import copy

        from entities.packet import (
            AckPacket,
            DataPacket,
        )
        from utils import config as uav_config

        if isinstance(
            packet,
            DataPacket,
        ):
            packet_copy = copy.copy(
                packet
            )

            destination_id = (
                packet_copy
                .dst_drone
                .identifier
            )

            if (
                destination_id
                == self.my_drone.identifier
            ):
                if (
                    packet_copy.packet_id
                    not in (
                        self.simulator
                        .metrics
                        .datapacket_arrived
                    )
                ):
                    self.simulator.metrics.calculate_metrics(
                        packet_copy
                    )

                uav_config.GL_ID_ACK_PACKET += 1

                ack_packet = AckPacket(
                    src_drone=self.my_drone,
                    dst_drone=(
                        self.simulator
                        .drones[src_drone_id]
                    ),
                    ack_packet_id=(
                        uav_config
                        .GL_ID_ACK_PACKET
                    ),
                    ack_packet_length=(
                        uav_config
                        .ACK_PACKET_LENGTH
                    ),
                    ack_packet=packet_copy,
                    simulator=self.simulator,
                    channel_id=(
                        packet_copy.channel_id
                    ),
                )

                yield self.simulator.env.timeout(
                    uav_config.SIFS_DURATION
                )

                ack_packet.increase_ttl()

                self.my_drone.mac_protocol.phy.unicast(
                    ack_packet,
                    src_drone_id,
                )

                yield self.simulator.env.timeout(
                    ack_packet.packet_length
                    / uav_config.BIT_RATE
                    * 1e6
                )

                return

            if (
                self.my_drone
                .transmitting_queue
                .qsize()
                < self.my_drone.max_queue_size
            ):
                self.my_drone.transmitting_queue.put(
                    packet_copy
                )

            return

        if isinstance(
            packet,
            AckPacket,
        ):
            data_packet = (
                packet.ack_packet
            )

            if (
                getattr(
                    data_packet,
                    "first_attempt_time",
                    None,
                )
                is not None
            ):
                self.simulator.metrics.mac_delay.append(
                    (
                        self.simulator.env.now
                        - data_packet.first_attempt_time
                    )
                    / 1e3
                )

            self.my_drone.remove_from_queue(
                data_packet
            )

            key = (
                f"wait_ack"
                f"{self.my_drone.identifier}"
                f"_{data_packet.packet_id}"
            )

            finish_state = (
                self.my_drone
                .mac_protocol
                .wait_ack_process_finish
                .get(
                    key,
                    1,
                )
            )

            if finish_state == 0:
                process = (
                    self.my_drone
                    .mac_protocol
                    .wait_ack_process_dict
                    .get(key)
                )

                if (
                    process is not None
                    and not process.triggered
                ):
                    self.my_drone.mac_protocol.wait_ack_process_finish[
                        key
                    ] = 1

                    process.interrupt()

            return

    def penalize(
        self,
        packet,
    ):
        return None


In [ ]:
class UavNetSimBackend(NetworkBackend):
    """Adapter from mission reports to UavNetSim packet-level networking."""

    def __init__(
        self,
        repo_path=None,
        seed=None,
    ):
        self.repo_path = repo_path
        self.seed = (
            int(CONFIG.get("seed", 44))
            if seed is None
            else int(seed)
        )

        self._initialized = False
        self._uavs = None
        self._gcs_position = None
        self._obstacles = None
        self._report_buffers = None
        self._pending_reports = None
        self._gcs_received_target_ids = None
        self._peer_transfer_states = (
            create_peer_transfer_states()
        )

        self._repo = None
        self._simpy = None
        self._simulator_class = None
        self._data_packet_class = None
        self._uav_config = None

        self._scene_directory = None
        self._environment = None
        self._simulator = None
        self._gcs_node_id = int(
            CONFIG["num_uavs"]
        )

        self._packet_records = {}
        self._metric_state = (
            self._new_metric_state()
        )
        self._last_step_comm_energy_by_uav = np.zeros(
            int(CONFIG["num_uavs"]),
            dtype=np.float64,
        )

    @staticmethod
    def _new_metric_state():
        return {
            "steps": 0,
            "requests": 0,
            "packets_injected": 0,
            "packets_delivered": 0,
            "packets_failed": 0,
            "queue_limit_events": 0,
            "payload_bytes_deferred": 0,
            "payload_bytes_injected": 0,
            "payload_bytes_delivered": 0,
            "communication_energy_j": 0.0,
            "status_counts": {},
        }

    @staticmethod
    def _validate_backend_config():
        payload_bytes = int(
            CONFIG[
                "uavnetsim_payload_bytes"
            ]
        )

        if payload_bytes <= 0:
            raise ValueError(
                "uavnetsim_payload_bytes "
                "must be > 0"
            )

        packet_lifetime_s = float(
            CONFIG[
                "uavnetsim_packet_lifetime_s"
            ]
        )

        if (
            not np.isfinite(
                packet_lifetime_s
            )
            or packet_lifetime_s <= 0.0
        ):
            raise ValueError(
                "uavnetsim_packet_lifetime_s "
                "must be finite and > 0"
            )

        max_queue_size = int(
            CONFIG[
                "uavnetsim_max_queue_size"
            ]
        )

        if max_queue_size <= 0:
            raise ValueError(
                "uavnetsim_max_queue_size "
                "must be > 0"
            )

    def _load_uavnetsim(self):
        self._repo = (
            resolve_uavnetsim_repo_path(
                self.repo_path
            )
        )

        repo_string = str(
            self._repo
        )

        if repo_string not in sys.path:
            sys.path.insert(
                0,
                repo_string,
            )

        # UavNetSim's simulator.log configures a relative
        # running_log.log file at import time when the root logger has
        # no handlers. Add a temporary NullHandler so importing the
        # external simulator never writes files into this project.
        import logging

        root_logger = logging.getLogger()
        temporary_handler = None

        if not root_logger.handlers:
            temporary_handler = (
                logging.NullHandler()
            )
            root_logger.addHandler(
                temporary_handler
            )

        try:
            self._simpy = (
                importlib.import_module(
                    "simpy"
                )
            )
            simulator_module = (
                importlib.import_module(
                    "simulator.simulator"
                )
            )
            packet_module = (
                importlib.import_module(
                    "entities.packet"
                )
            )
            self._uav_config = (
                importlib.import_module(
                    "utils.config"
                )
            )
        finally:
            if (
                temporary_handler
                is not None
            ):
                root_logger.removeHandler(
                    temporary_handler
                )

        simulator_file = Path(
            simulator_module.__file__
        ).resolve()

        if (
            self._repo
            not in simulator_file.parents
        ):
            raise RuntimeError(
                "A different simulator package "
                "named 'simulator' is already "
                "loaded in this Python process"
            )

        self._simulator_class = (
            simulator_module.Simulator
        )
        self._data_packet_class = (
            packet_module.DataPacket
        )

    def _configure_uavnetsim(
        self,
        scene_directory,
    ):
        uav_config = self._uav_config

        num_network_nodes = (
            int(CONFIG["num_uavs"])
            + 1
        )

        uav_config.NUMBER_OF_DRONES = (
            num_network_nodes
        )
        uav_config.MAX_TTL = (
            num_network_nodes + 1
        )

        uav_config.ROUTING_PROTOCOL = (
            "DRL"
        )
        uav_config.DRL_ROUTING_PROTOCOL_CLASS = (
            MarlSelectedNextHopRouting
        )
        uav_config.DRL_HELLO_PACKET_CLASS = (
            None
        )
        uav_config.DRL_NEIGHBOR_TABLE_CLASS = (
            None
        )

        uav_config.MAC_PROTOCOL = str(
            CONFIG[
                "uavnetsim_mac_protocol"
            ]
        )
        uav_config.MOBILITY_MODEL = (
            "GaussMarkov3D"
        )

        uav_config.CHANNEL_MODE = str(
            CONFIG[
                "uavnetsim_channel_mode"
            ]
        )
        uav_config.LOS_A2A_MODEL = str(
            CONFIG[
                "uavnetsim_los_model"
            ]
        )
        uav_config.NLOS_A2A_MODEL = str(
            CONFIG[
                "uavnetsim_nlos_model"
            ]
        )

        uav_config.STATIC_CASE = 1
        uav_config.HETEROGENEOUS = 0

        uav_config.TRAFFIC_PATTERN = (
            "UNIFORM"
        )
        uav_config.PACKET_ARRIVAL_RATE = (
            1e-12
        )

        uav_config.MAP_LENGTH = float(
            CONFIG["map_size"]
        )
        uav_config.MAP_WIDTH = float(
            CONFIG["map_size"]
        )
        uav_config.MAP_HEIGHT = float(
            CONFIG["altitude_max"]
        )
        uav_config.UAV_MIN_ALTITUDE = (
            float(
                CONFIG[
                    "altitude_min"
                ]
            )
        )
        uav_config.UAV_MAX_ALTITUDE = (
            float(
                CONFIG[
                    "altitude_max"
                ]
            )
        )
        uav_config.UAV_BOUNDARY_CLEARANCE = (
            0.0
        )
        uav_config.UAV_BUILDING_CLEARANCE = (
            0.0
        )

        uav_config.MAX_QUEUE_SIZE = int(
            CONFIG[
                "uavnetsim_max_queue_size"
            ]
        )

        uav_config.AVERAGE_PAYLOAD_LENGTH = (
            int(
                CONFIG[
                    "uavnetsim_payload_bytes"
                ]
            )
            * 8
        )
        uav_config.VARIABLE_PAYLOAD_LENGTH = (
            0
        )
        uav_config.PACKET_LIFETIME = (
            float(
                CONFIG[
                    "uavnetsim_packet_lifetime_s"
                ]
            )
            * 1e6
        )

        uav_config.INITIAL_ENERGY = max(
            float(CONFIG["battery_j"]),
            10_000.0,
        )
        uav_config.ENERGY_THRESHOLD = (
            0.0
        )

        uav_config.TRANSMITTING_POWER = (
            float(
                CONFIG[
                    "tx_power_min_w"
                ]
            )
        )

        uav_config.SIONNA_SCENE_PATH = str(
            Path(scene_directory)
            / "scene.xml"
        )

    def _patch_dynamic_tx_power(self):
        original_transmit = (
            self._simulator
            .channel
            .transmit
        )
        uav_config = self._uav_config

        def transmit_with_packet_power(
            packet,
            transmitter_id,
            receiver_ids,
        ):
            previous_power = (
                uav_config
                .TRANSMITTING_POWER
            )

            packet_power = float(
                getattr(
                    packet,
                    "mission_tx_power_w",
                    previous_power,
                )
            )

            uav_config.TRANSMITTING_POWER = (
                packet_power
            )

            try:
                return original_transmit(
                    packet,
                    transmitter_id,
                    receiver_ids,
                )
            finally:
                uav_config.TRANSMITTING_POWER = (
                    previous_power
                )

        self._simulator.channel.transmit = (
            transmit_with_packet_power
        )

        for drone in self._simulator.drones:
            phy = (
                drone
                .mac_protocol
                .phy
            )

            def consume_energy(
                packet,
                phy=phy,
            ):
                power_w = float(
                    getattr(
                        packet,
                        "mission_tx_power_w",
                        uav_config
                        .TRANSMITTING_POWER,
                    )
                )

                duration_s = (
                    packet.packet_length
                    / uav_config.BIT_RATE
                )

                phy.my_drone.residual_energy = max(
                    0.0,
                    (
                        phy.my_drone
                        .residual_energy
                        - duration_s
                        * power_w
                    ),
                )

            phy._consume_transmit_energy = (
                consume_energy
            )

    def _freeze_external_mobility(self):
        total_us = (
            (
                int(CONFIG["max_steps"])
                + 2
            )
            * float(CONFIG["dt"])
            * 1e6
        )

        freeze_interval = (
            total_us + 1e6
        )

        for drone in self._simulator.drones:
            drone.speed = 0.0
            drone.velocity = [
                0.0,
                0.0,
                0.0,
            ]
            drone.velocity_mean = 0.0

            model = getattr(
                drone,
                "mobility_model",
                None,
            )

            if model is not None:
                if hasattr(
                    model,
                    "position_update_interval",
                ):
                    model.position_update_interval = (
                        freeze_interval
                    )

                if hasattr(
                    model,
                    "direction_update_interval",
                ):
                    model.direction_update_interval = (
                        freeze_interval
                    )

    def reset(
        self,
        *,
        uavs,
        gcs_position,
        obstacles,
        report_buffers,
        pending_reports,
        gcs_received_target_ids,
    ):
        self._validate_backend_config()

        SimpleNetworkBackend._validate_uavs(
            uavs
        )
        SimpleNetworkBackend._validate_obstacles(
            obstacles
        )
        SimpleNetworkBackend._validate_report_state(
            report_buffers,
            pending_reports,
            gcs_received_target_ids,
        )

        validated_gcs = (
            SimpleNetworkBackend
            ._validated_gcs_position(
                gcs_position
            )
        )

        if self._initialized:
            self.close()

        self._load_uavnetsim()

        self._scene_directory = (
            tempfile.TemporaryDirectory(
                prefix="uav_search_uavnetsim_"
            )
        )

        write_uavnetsim_scene(
            obstacles,
            self._scene_directory.name,
        )

        self._configure_uavnetsim(
            self._scene_directory.name
        )

        self._environment = (
            self._simpy.Environment()
        )

        total_us = (
            (
                int(CONFIG["max_steps"])
                + 2
            )
            * float(CONFIG["dt"])
            * 1e6
        )

        self._simulator = (
            self._simulator_class(
                seed=self.seed,
                env=self._environment,
                n_drones=(
                    int(
                        CONFIG[
                            "num_uavs"
                        ]
                    )
                    + 1
                ),
                total_simulation_time=(
                    total_us
                ),
                drone_speed=0.0,
            )
        )

        self._freeze_external_mobility()
        self._patch_dynamic_tx_power()

        self._uavs = uavs
        self._gcs_position = (
            validated_gcs
        )
        self._obstacles = obstacles
        self._report_buffers = (
            report_buffers
        )
        self._pending_reports = (
            pending_reports
        )
        self._gcs_received_target_ids = (
            gcs_received_target_ids
        )

        self._peer_transfer_states = (
            create_peer_transfer_states()
        )
        self._packet_records = {}
        self._metric_state = (
            self._new_metric_state()
        )
        self._last_step_comm_energy_by_uav = np.zeros(
            int(CONFIG["num_uavs"]),
            dtype=np.float64,
        )

        self._initialized = True

        self.sync_positions(
            uavs
        )

        # Execute all processes scheduled at t=0 once.
        self._environment.run(
            until=1.0
        )

        # UavNetSim mobility is deliberately frozen; re-apply the
        # mission positions after its t=0 initialization.
        self.sync_positions(
            uavs
        )

    def sync_positions(
        self,
        uavs,
    ):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before "
                "sync_positions"
            )

        SimpleNetworkBackend._validate_uavs(
            uavs
        )

        map_size = float(
            CONFIG["map_size"]
        )
        altitude_min = float(
            CONFIG["altitude_min"]
        )
        altitude_max = float(
            CONFIG["altitude_max"]
        )

        for mission_uav in uavs:
            position = np.asarray(
                mission_uav.position,
                dtype=np.float64,
            )
            velocity = np.asarray(
                mission_uav.velocity,
                dtype=np.float64,
            )

            if (
                position.shape != (3,)
                or velocity.shape != (3,)
            ):
                raise ValueError(
                    "UAV position and velocity "
                    "must have shape (3,)"
                )

            if (
                not np.all(
                    np.isfinite(position)
                )
                or not np.all(
                    np.isfinite(velocity)
                )
            ):
                raise ValueError(
                    "UAV position and velocity "
                    "must be finite"
                )

            if not (
                0.0
                <= position[0]
                <= map_size
                and 0.0
                <= position[1]
                <= map_size
                and altitude_min
                <= position[2]
                <= altitude_max
            ):
                raise ValueError(
                    "UAV position is outside "
                    "mission bounds"
                )

            external_drone = (
                self._simulator
                .drones[mission_uav.id]
            )

            # Mission motion remains the source of truth. Directly
            # synchronize coordinates instead of invoking UavNetSim
            # mobility/collision resolution.
            external_drone._coords = [
                float(value)
                for value in position
            ]
            external_drone.velocity = [
                float(value)
                for value in velocity
            ]
            external_drone.sleep = bool(
                not mission_uav.active
            )

        gcs_drone = (
            self._simulator
            .drones[
                self._gcs_node_id
            ]
        )
        gcs_drone._coords = [
            float(value)
            for value
            in self._gcs_position
        ]
        gcs_drone.velocity = [
            0.0,
            0.0,
            0.0,
        ]

        self._uavs = uavs

    @staticmethod
    def _validate_step_clock(
        dt,
        current_step,
    ):
        if (
            isinstance(
                current_step,
                (bool, np.bool_),
            )
            or not isinstance(
                current_step,
                (int, np.integer),
            )
        ):
            raise TypeError(
                "current_step must be an integer"
            )

        current_step = int(
            current_step
        )

        if current_step < 0:
            raise ValueError(
                "current_step must be >= 0"
            )

        if isinstance(
            dt,
            (bool, np.bool_),
        ):
            raise TypeError(
                "dt must be numeric and not boolean"
            )

        dt = float(dt)

        if (
            not np.isfinite(dt)
            or dt <= 0.0
        ):
            raise ValueError(
                "dt must be finite and > 0"
            )

        if not np.isclose(
            dt,
            float(CONFIG["dt"]),
            rtol=0.0,
            atol=1e-12,
        ):
            raise ValueError(
                "UavNetSimBackend dt must match "
                "CONFIG['dt']"
            )

        return dt, current_step

    @staticmethod
    def _normalize_requests(
        requests,
    ):
        if requests is None:
            request_list = []
        elif isinstance(
            requests,
            TransmissionIntent,
        ):
            request_list = [
                requests
            ]
        elif isinstance(
            requests,
            (list, tuple),
        ):
            request_list = list(
                requests
            )
        else:
            raise TypeError(
                "requests must be a "
                "TransmissionIntent, a list/"
                "tuple of intents, or None"
            )

        for request in request_list:
            if not isinstance(
                request,
                TransmissionIntent,
            ):
                raise TypeError(
                    "requests must contain only "
                    "TransmissionIntent objects"
                )

        sender_ids = [
            int(request.sender)
            for request in request_list
        ]

        if len(sender_ids) != len(
            set(sender_ids)
        ):
            raise ValueError(
                "each sender may submit at most "
                "one transmission request per "
                "network step"
            )

        return request_list

    def _report_generation_key(
        self,
        intent,
        report,
    ):
        return (
            int(intent.sender),
            int(intent.recipient),
            int(report.target_id),
            int(report.source_uav),
            int(report.created_step),
            int(report.size_bytes),
            float(report.ttl_s),
        )

    def _report_for_intent(
        self,
        intent,
        current_step,
    ):
        if (
            intent.target_id
            in self._gcs_received_target_ids
        ):
            return None

        sender_buffer = (
            self._report_buffers[
                intent.sender
            ]
        )

        cleanup_report_buffer(
            sender_buffer,
            current_step,
        )

        report = find_report_in_buffer(
            sender_buffer,
            intent.target_id,
        )

        if report is None:
            return None

        if report_is_expired(
            report,
            current_step,
        ):
            return None

        return report

    def _inflight_bytes(
        self,
        transfer_key,
    ):
        return sum(
            int(record["payload_bytes"])
            for record
            in self._packet_records.values()
            if (
                record["status"]
                == "in_flight"
                and record["transfer_key"]
                == transfer_key
            )
        )

    def _recipient_node_id(
        self,
        recipient,
    ):
        if recipient == GCS_NODE:
            return self._gcs_node_id

        return int(recipient)

    def _request_is_active(
        self,
        intent,
    ):
        if not self._uavs[
            intent.sender
        ].active:
            return False

        if intent.recipient == GCS_NODE:
            return True

        return bool(
            self._uavs[
                intent.recipient
            ].active
        )

    def _request_is_within_contact_range(
        self,
        intent,
    ):
        sender_position = np.asarray(
            self._uavs[
                intent.sender
            ].position,
            dtype=np.float64,
        )

        if intent.recipient == GCS_NODE:
            recipient_position = np.asarray(
                self._gcs_position,
                dtype=np.float64,
            )
            max_range = float(
                CONFIG[
                    "gcs_contact_range_m"
                ]
            )
        else:
            recipient_position = np.asarray(
                self._uavs[
                    intent.recipient
                ].position,
                dtype=np.float64,
            )
            max_range = float(
                CONFIG[
                    "peer_contact_range_m"
                ]
            )

        distance = float(
            np.linalg.norm(
                sender_position
                - recipient_position
            )
        )

        return bool(
            distance <= max_range
        )

    def _inject_request(
        self,
        intent,
        current_step,
    ):
        if not self._request_is_active(
            intent
        ):
            return {
                "transfer_key": None,
                "injected_bytes": 0,
                "injected_packets": 0,
                "reason": "inactive_node",
            }

        if not self._request_is_within_contact_range(
            intent
        ):
            return {
                "transfer_key": None,
                "injected_bytes": 0,
                "injected_packets": 0,
                "reason": "out_of_contact_range",
            }

        report = self._report_for_intent(
            intent,
            current_step,
        )

        if report is None:
            return {
                "transfer_key": None,
                "injected_bytes": 0,
                "injected_packets": 0,
                "reason": "no_report",
            }

        transfer_key = (
            self._report_generation_key(
                intent,
                report,
            )
        )

        inflight_bytes = (
            self._inflight_bytes(
                transfer_key
            )
        )

        bytes_to_inject = max(
            0,
            int(intent.requested_bytes)
            - inflight_bytes,
        )

        if bytes_to_inject <= 0:
            return {
                "transfer_key": transfer_key,
                "injected_bytes": 0,
                "injected_packets": 0,
                "reason": "already_in_flight",
            }

        recipient_node_id = (
            self._recipient_node_id(
                intent.recipient
            )
        )

        source_drone = (
            self._simulator
            .drones[intent.sender]
        )
        destination_drone = (
            self._simulator
            .drones[recipient_node_id]
        )

        payload_limit = int(
            CONFIG[
                "uavnetsim_payload_bytes"
            ]
        )

        injected_bytes = 0
        injected_packets = 0

        remaining = (
            bytes_to_inject
        )

        while remaining > 0:
            payload_bytes = min(
                payload_limit,
                remaining,
            )

            (
                self._uav_config
                .GL_ID_DATA_PACKET
            ) += 1

            packet_id = int(
                self._uav_config
                .GL_ID_DATA_PACKET
            )

            packet_length_bits = int(
                self._uav_config
                .IP_HEADER_LENGTH
                + self._uav_config
                .MAC_HEADER_LENGTH
                + self._uav_config
                .PHY_HEADER_LENGTH
                + payload_bytes
                * 8
            )

            channel_id = (
                source_drone
                .channel_assigner
                .channel_assign()
            )

            packet = (
                self._data_packet_class(
                    src_drone=source_drone,
                    dst_drone=destination_drone,
                    creation_time=(
                        self._environment.now
                    ),
                    data_packet_id=(
                        packet_id
                    ),
                    data_packet_length=(
                        packet_length_bits
                    ),
                    simulator=(
                        self._simulator
                    ),
                    channel_id=(
                        channel_id
                    ),
                )
            )

            packet.transmission_mode = 0
            packet.forced_next_hop_id = (
                recipient_node_id
            )
            packet.mission_tx_power_w = (
                float(
                    intent.tx_power_w
                )
            )

            self._simulator.metrics.record_generated(
                packet
            )

            packet.waiting_start_time = (
                self._environment.now
            )

            if (
                source_drone
                .transmitting_queue
                .qsize()
                >= source_drone.max_queue_size
            ):
                self._metric_state[
                    "queue_limit_events"
                ] += 1
                self._metric_state[
                    "payload_bytes_deferred"
                ] += int(remaining)
                break

            source_drone.transmitting_queue.put(
                packet
            )

            self._packet_records[
                packet_id
            ] = {
                "packet": packet,
                "transfer_key": transfer_key,
                "sender": int(
                    intent.sender
                ),
                "recipient": int(
                    intent.recipient
                ),
                "target_id": int(
                    intent.target_id
                ),
                "source_uav": int(
                    report.source_uav
                ),
                "created_step": int(
                    report.created_step
                ),
                "size_bytes": int(
                    report.size_bytes
                ),
                "ttl_s": float(
                    report.ttl_s
                ),
                "payload_bytes": int(
                    payload_bytes
                ),
                "tx_power_w": float(
                    intent.tx_power_w
                ),
                "status": "in_flight",
            }

            injected_bytes += int(
                payload_bytes
            )
            injected_packets += 1
            remaining -= int(
                payload_bytes
            )

        self._metric_state[
            "packets_injected"
        ] += injected_packets
        self._metric_state[
            "payload_bytes_injected"
        ] += injected_bytes

        deferred_bytes = int(
            remaining
        )

        return {
            "transfer_key": transfer_key,
            "injected_bytes": (
                injected_bytes
            ),
            "injected_packets": (
                injected_packets
            ),
            "deferred_bytes": (
                deferred_bytes
            ),
            "reason": (
                "queue_limited"
                if deferred_bytes > 0
                else "injected"
            ),
        }

    def _collect_packet_outcomes(self):
        delivered_ids = (
            self._simulator
            .metrics
            .datapacket_arrived
        )

        delivered_by_key = {}
        failed_by_key = {}

        max_attempts = int(
            self._uav_config
            .MAX_RETRANSMISSION_ATTEMPT
        )

        for (
            packet_id,
            record,
        ) in self._packet_records.items():
            if (
                record["status"]
                != "in_flight"
            ):
                continue

            packet = record["packet"]
            transfer_key = (
                record["transfer_key"]
            )

            if packet_id in delivered_ids:
                recipient = int(
                    record["recipient"]
                )

                if (
                    recipient != GCS_NODE
                    and not self._uavs[
                        recipient
                    ].active
                ):
                    record["status"] = (
                        "failed"
                    )
                    failed_by_key[
                        transfer_key
                    ] = (
                        failed_by_key.get(
                            transfer_key,
                            0,
                        )
                        + int(
                            record[
                                "payload_bytes"
                            ]
                        )
                    )
                    self._metric_state[
                        "packets_failed"
                    ] += 1
                    continue

                record["status"] = (
                    "delivered"
                )

                delivered_by_key[
                    transfer_key
                ] = (
                    delivered_by_key.get(
                        transfer_key,
                        0,
                    )
                    + int(
                        record[
                            "payload_bytes"
                        ]
                    )
                )

                self._metric_state[
                    "packets_delivered"
                ] += 1
                self._metric_state[
                    "payload_bytes_delivered"
                ] += int(
                    record[
                        "payload_bytes"
                    ]
                )

                continue

            attempts = int(
                packet
                .number_retransmission_attempt
                .get(
                    record["sender"],
                    0,
                )
            )

            expired = bool(
                self._environment.now
                >= (
                    packet.creation_time
                    + packet.deadline
                )
            )

            if (
                attempts
                >= max_attempts
                or expired
            ):
                record["status"] = (
                    "failed"
                )
                failed_by_key[
                    transfer_key
                ] = (
                    failed_by_key.get(
                        transfer_key,
                        0,
                    )
                    + int(
                        record[
                            "payload_bytes"
                        ]
                    )
                )
                self._metric_state[
                    "packets_failed"
                ] += 1

        return (
            delivered_by_key,
            failed_by_key,
        )

    def _generation_matches_record(
        self,
        report,
        record,
    ):
        return bool(
            report.target_id
            == record["target_id"]
            and report.source_uav
            == record["source_uav"]
            and report.created_step
            == record["created_step"]
            and report.size_bytes
            == record["size_bytes"]
            and np.isclose(
                report.ttl_s,
                record["ttl_s"],
            )
        )

    def _commit_delivered_bytes(
        self,
        transfer_key,
        delivered_bytes,
        current_step,
    ):
        delivered_bytes = int(
            delivered_bytes
        )

        if delivered_bytes <= 0:
            return None

        matching_records = [
            record
            for record
            in self._packet_records.values()
            if (
                record["transfer_key"]
                == transfer_key
            )
        ]

        if not matching_records:
            return None

        record = (
            matching_records[0]
        )

        target_id = int(
            record["target_id"]
        )

        if (
            target_id
            in self._gcs_received_target_ids
        ):
            return {
                "status": "already_delivered",
                "committed_bytes": 0,
            }

        sender = int(
            record["sender"]
        )

        sender_report = find_report_in_buffer(
            self._report_buffers[
                sender
            ],
            target_id,
        )

        if (
            sender_report is None
            or not self._generation_matches_record(
                sender_report,
                record,
            )
        ):
            return {
                "status": "stale_generation",
                "committed_bytes": 0,
            }

        intent = TransmissionIntent(
            sender=sender,
            recipient=int(
                record["recipient"]
            ),
            target_id=target_id,
            requested_bytes=(
                delivered_bytes
            ),
            tx_power_w=float(
                record["tx_power_w"]
            ),
        )

        if (
            intent.recipient
            == GCS_NODE
        ):
            result = (
                commit_gcs_transfer_bytes(
                    intent,
                    delivered_bytes,
                    self._report_buffers,
                    self._pending_reports,
                    self._gcs_received_target_ids,
                    self._peer_transfer_states,
                    current_step,
                )
            )

            return {
                "status": (
                    "gcs_delivered"
                    if (
                        result["status"]
                        == "delivered"
                    )
                    else (
                        "gcs_partial"
                        if (
                            result["status"]
                            == "partial"
                        )
                        else result[
                            "status"
                        ]
                    )
                ),
                "committed_bytes": int(
                    result.get(
                        "committed_bytes",
                        0,
                    )
                ),
                "commit_result": result,
            }

        peer_commit = commit_peer_transfer_bytes(
            intent,
            delivered_bytes,
            self._report_buffers[
                sender
            ],
            self._peer_transfer_states,
            current_step,
        )

        state = peer_commit["state"]
        committed_bytes = int(
            peer_commit[
                "committed_bytes"
            ]
        )

        if not peer_commit["complete"]:
            return {
                "status": "peer_partial",
                "committed_bytes": (
                    committed_bytes
                ),
                "peer_state": state,
                "peer_commit_result": (
                    peer_commit
                ),
            }

        key = peer_transfer_key(
            sender,
            intent.recipient,
            target_id,
        )

        custody_result = (
            commit_completed_peer_transfer_to_receiver(
                key,
                self._peer_transfer_states,
                self._report_buffers,
                current_step,
                self._gcs_received_target_ids,
            )
        )

        status_map = {
            "enqueued": "peer_enqueued",
            "buffer_full": (
                "peer_buffer_full"
            ),
            "already_present": (
                "peer_already_present"
            ),
            "expired": "expired",
            "already_delivered": (
                "already_delivered"
            ),
        }

        return {
            "status": status_map.get(
                custody_result["status"],
                custody_result["status"],
            ),
            "committed_bytes": (
                committed_bytes
            ),
            "peer_state": state,
            "peer_commit_result": (
                peer_commit
            ),
            "custody_result": (
                custody_result
            ),
        }

    def _has_inflight_for_key(
        self,
        transfer_key,
    ):
        return any(
            (
                record["status"]
                == "in_flight"
                and record[
                    "transfer_key"
                ]
                == transfer_key
            )
            for record
            in self._packet_records.values()
        )

    def _cleanup_terminal_packet_records(
        self,
    ):
        for (
            packet_id,
            record,
        ) in list(
            self._packet_records.items()
        ):
            if (
                record["status"]
                == "in_flight"
            ):
                continue

            del self._packet_records[
                packet_id
            ]

    def step(
        self,
        requests,
        dt,
        current_step,
    ):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before step"
            )

        dt, current_step = (
            self._validate_step_clock(
                dt,
                current_step,
            )
        )

        request_list = (
            self._normalize_requests(
                requests
            )
        )

        self.sync_positions(
            self._uavs
        )

        cleanup_peer_transfer_states(
            self._peer_transfer_states,
            current_step,
        )

        retry_completed_peer_transfers(
            self._peer_transfer_states,
            self._report_buffers,
            current_step,
            self._gcs_received_target_ids,
        )

        request_contexts = [
            (
                request,
                self._inject_request(
                    request,
                    current_step,
                ),
            )
            for request
            in request_list
        ]

        energy_before = [
            float(
                drone.residual_energy
            )
            for drone
            in self._simulator.drones[
                : int(
                    CONFIG["num_uavs"]
                )
            ]
        ]

        target_time = (
            self._environment.now
            + dt
            * 1e6
        )

        self._environment.run(
            until=target_time
        )

        energy_after = [
            float(
                drone.residual_energy
            )
            for drone
            in self._simulator.drones[
                : int(
                    CONFIG["num_uavs"]
                )
            ]
        ]

        self._last_step_comm_energy_by_uav = np.asarray(
            [
                max(
                    0.0,
                    before - after,
                )
                for before, after
                in zip(
                    energy_before,
                    energy_after,
                )
            ],
            dtype=np.float64,
        )

        communication_energy = float(
            np.sum(
                self._last_step_comm_energy_by_uav
            )
        )

        self._metric_state[
            "communication_energy_j"
        ] += communication_energy

        (
            delivered_by_key,
            failed_by_key,
        ) = self._collect_packet_outcomes()

        commit_results = {}

        for (
            transfer_key,
            delivered_bytes,
        ) in delivered_by_key.items():
            commit_results[
                transfer_key
            ] = (
                self._commit_delivered_bytes(
                    transfer_key,
                    delivered_bytes,
                    current_step,
                )
            )

        results = []

        for (
            request,
            context,
        ) in request_contexts:
            transfer_key = (
                context["transfer_key"]
            )

            if transfer_key is None:
                result = {
                    "status": "no_transfer",
                    "tx_bytes": 0,
                    "injected_bytes": 0,
                    "injected_packets": 0,
                    "deferred_bytes": int(
                        context.get(
                            "deferred_bytes",
                            0,
                        )
                    ),
                    "reason": (
                        context["reason"]
                    ),
                }
            else:
                delivered_bytes = int(
                    delivered_by_key.get(
                        transfer_key,
                        0,
                    )
                )
                failed_bytes = int(
                    failed_by_key.get(
                        transfer_key,
                        0,
                    )
                )
                commit_result = (
                    commit_results.get(
                        transfer_key
                    )
                )

                if commit_result is not None:
                    status = (
                        commit_result[
                            "status"
                        ]
                    )
                    committed_bytes = int(
                        commit_result.get(
                            "committed_bytes",
                            0,
                        )
                    )
                elif self._has_inflight_for_key(
                    transfer_key
                ):
                    status = "in_flight"
                    committed_bytes = 0
                elif failed_bytes > 0:
                    status = "packet_failed"
                    committed_bytes = 0
                else:
                    status = "no_transfer"
                    committed_bytes = 0

                result = {
                    "status": status,
                    "tx_bytes": (
                        committed_bytes
                    ),
                    "delivered_payload_bytes": (
                        delivered_bytes
                    ),
                    "failed_payload_bytes": (
                        failed_bytes
                    ),
                    "injected_bytes": int(
                        context[
                            "injected_bytes"
                        ]
                    ),
                    "injected_packets": int(
                        context[
                            "injected_packets"
                        ]
                    ),
                    "deferred_bytes": int(
                        context.get(
                            "deferred_bytes",
                            0,
                        )
                    ),
                    "reason": (
                        context["reason"]
                    ),
                    "commit_result": (
                        commit_result
                    ),
                }

            results.append(
                result
            )

            status = str(
                result["status"]
            )

            status_counts = (
                self._metric_state[
                    "status_counts"
                ]
            )

            status_counts[status] = (
                status_counts.get(
                    status,
                    0,
                )
                + 1
            )

        self._metric_state[
            "steps"
        ] += 1
        self._metric_state[
            "requests"
        ] += len(
            request_list
        )

        self._cleanup_terminal_packet_records()

        return results

    @property
    def peer_transfer_states(self):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before "
                "accessing peer_transfer_states"
            )

        return self._peer_transfer_states

    def metrics(self):
        network_metrics = {}

        if self._simulator is not None:
            network_metrics = (
                self._simulator
                .metrics
                .snapshot()
            )

        return {
            "backend": "uavnetsim",
            "steps": int(
                self._metric_state[
                    "steps"
                ]
            ),
            "requests": int(
                self._metric_state[
                    "requests"
                ]
            ),
            "packets_injected": int(
                self._metric_state[
                    "packets_injected"
                ]
            ),
            "packets_delivered": int(
                self._metric_state[
                    "packets_delivered"
                ]
            ),
            "packets_failed": int(
                self._metric_state[
                    "packets_failed"
                ]
            ),
            "queue_limit_events": int(
                self._metric_state[
                    "queue_limit_events"
                ]
            ),
            "payload_bytes_deferred": int(
                self._metric_state[
                    "payload_bytes_deferred"
                ]
            ),
            "payload_bytes_injected": int(
                self._metric_state[
                    "payload_bytes_injected"
                ]
            ),
            "payload_bytes_delivered": int(
                self._metric_state[
                    "payload_bytes_delivered"
                ]
            ),
            "communication_energy_j": float(
                self._metric_state[
                    "communication_energy_j"
                ]
            ),
            "in_flight_packets": sum(
                record["status"]
                == "in_flight"
                for record
                in self._packet_records.values()
            ),
            "active_peer_transfers": len(
                self._peer_transfer_states
            ),
            "status_counts": dict(
                self._metric_state[
                    "status_counts"
                ]
            ),
            "network_metrics": dict(
                network_metrics
            ),
        }

    def last_step_communication_energy_by_uav(self):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before "
                "reading communication energy"
            )

        return (
            self._last_step_comm_energy_by_uav
            .copy()
        )

    def close(self):
        if self._simulator is not None:
            self._simulator.close()

        if (
            self._scene_directory
            is not None
        ):
            self._scene_directory.cleanup()

        self._scene_directory = None
        self._environment = None
        self._simulator = None
        self._packet_records = {}
        self._initialized = False


In [ ]:
def create_network_backend(
    backend_name=None,
    **kwargs,
):
    if backend_name is None:
        backend_name = CONFIG.get(
            "network_backend",
            "simple",
        )

    normalized = str(
        backend_name
    ).strip().lower()

    if normalized == "simple":
        if kwargs:
            unexpected = ", ".join(
                sorted(kwargs)
            )
            raise TypeError(
                "SimpleNetworkBackend does not "
                f"accept: {unexpected}"
            )

        return SimpleNetworkBackend()

    if normalized in {
        "uavnetsim",
        "uav_net_sim",
    }:
        return UavNetSimBackend(
            **kwargs
        )

    raise ValueError(
        "network backend must be "
        "'simple' or 'uavnetsim'"
    )


In [ ]:
def binary_entropy_bits(probability):
    probability = float(
        np.clip(
            probability,
            1e-12,
            1.0 - 1e-12,
        )
    )

    return float(
        -probability
        * np.log2(probability)
        - (1.0 - probability)
        * np.log2(
            1.0 - probability
        )
    )


def sensing_information_gain_bits(
    sensing_records,
):
    gain = 0.0

    for record in sensing_records:
        prior_entropy = (
            binary_entropy_bits(
                record["prior"]
            )
        )
        posterior_entropy = (
            binary_entropy_bits(
                record["posterior"]
            )
        )

        # Use realized positive entropy reduction as a stable
        # cooperative shaping term.
        gain += max(
            0.0,
            prior_entropy
            - posterior_entropy,
        )

    return float(gain)


def propulsion_energy_j(
    uav,
    hybrid_action,
    dt=None,
):
    if not isinstance(uav, UAV):
        raise TypeError(
            "uav must be UAV"
        )

    if not isinstance(
        hybrid_action,
        HybridAction,
    ):
        raise TypeError(
            "hybrid_action must be HybridAction"
        )

    if dt is None:
        dt = CONFIG["dt"]

    dt = float(dt)

    if (
        not np.isfinite(dt)
        or dt <= 0.0
    ):
        raise ValueError(
            "dt must be finite and > 0"
        )

    if not uav.active:
        return 0.0

    speed = float(
        np.linalg.norm(
            uav.velocity
        )
    )

    commanded_accel = float(
        np.linalg.norm(
            hybrid_action.movement
        )
        * float(
            CONFIG["max_accel"]
        )
    )

    if (
        uav.position[2] <= 0.0
        and speed <= 1e-12
        and commanded_accel <= 1e-12
    ):
        power_w = float(
            CONFIG[
                "energy_idle_power_w"
            ]
        )
    else:
        power_w = (
            float(
                CONFIG[
                    "energy_hover_power_w"
                ]
            )
            + float(
                CONFIG[
                    "energy_speed_sq_coeff"
                ]
            )
            * speed
            * speed
            + float(
                CONFIG[
                    "energy_accel_sq_coeff"
                ]
            )
            * commanded_accel
            * commanded_accel
        )

    if (
        not np.isfinite(power_w)
        or power_w < 0.0
    ):
        raise ValueError(
            "computed propulsion power "
            "must be finite and >= 0"
        )

    return float(
        power_w * dt
    )


def apply_uav_energy_budget(
    uavs,
    hybrid_actions,
    communication_energy_j,
    dt=None,
):
    if dt is None:
        dt = CONFIG["dt"]

    if len(uavs) != len(
        hybrid_actions
    ):
        raise ValueError(
            "uavs and hybrid_actions "
            "must have the same length"
        )

    communication_energy_j = (
        np.asarray(
            communication_energy_j,
            dtype=np.float64,
        )
    )

    if communication_energy_j.shape != (
        len(uavs),
    ):
        raise ValueError(
            "communication_energy_j has "
            "the wrong shape"
        )

    if (
        not np.all(
            np.isfinite(
                communication_energy_j
            )
        )
        or np.any(
            communication_energy_j < 0.0
        )
    ):
        raise ValueError(
            "communication energy must be "
            "finite and >= 0"
        )

    propulsion = np.zeros(
        len(uavs),
        dtype=np.float64,
    )

    total = np.zeros(
        len(uavs),
        dtype=np.float64,
    )

    depleted_ids = []

    for index, (
        uav,
        hybrid_action,
    ) in enumerate(
        zip(
            uavs,
            hybrid_actions,
        )
    ):
        propulsion[index] = (
            propulsion_energy_j(
                uav,
                hybrid_action,
                dt=dt,
            )
        )

        total[index] = (
            propulsion[index]
            + communication_energy_j[
                index
            ]
        )

        uav.battery_j = max(
            0.0,
            float(uav.battery_j)
            - float(total[index]),
        )

        if (
            uav.active
            and uav.battery_j <= 0.0
        ):
            uav.active = False
            uav.velocity = np.zeros(
                3,
                dtype=np.float64,
            )
            depleted_ids.append(
                int(uav.id)
            )

    return {
        "propulsion_j": propulsion,
        "communication_j": (
            communication_energy_j.copy()
        ),
        "total_j": total,
        "depleted_uav_ids": (
            depleted_ids
        ),
    }


In [ ]:
def _normalized_relative_vector(
    source_position,
    destination_position,
):
    source_position = np.asarray(
        source_position,
        dtype=np.float64,
    )
    destination_position = np.asarray(
        destination_position,
        dtype=np.float64,
    )

    if (
        source_position.shape != (3,)
        or destination_position.shape
        != (3,)
    ):
        raise ValueError(
            "positions must have shape (3,)"
        )

    altitude_span = max(
        1e-9,
        float(CONFIG["altitude_max"])
        - float(CONFIG["altitude_min"]),
    )

    scale = np.array(
        [
            float(CONFIG["map_size"]),
            float(CONFIG["map_size"]),
            altitude_span,
        ],
        dtype=np.float64,
    )

    return np.clip(
        (
            destination_position
            - source_position
        )
        / scale,
        -1.0,
        1.0,
    ).astype(
        np.float32
    )


def extract_belief_patch(
    belief_map,
    position,
):
    patch_cells = int(
        CONFIG["belief_patch_cells"]
    )

    if (
        patch_cells < 1
        or patch_cells % 2 == 0
    ):
        raise ValueError(
            "belief_patch_cells must be "
            "a positive odd integer"
        )

    belief_map = np.asarray(
        belief_map,
        dtype=np.float64,
    )

    if belief_map.ndim != 2:
        raise ValueError(
            "belief_map must be 2-D"
        )

    gx, gy = world_to_grid(
        position
    )

    radius = (
        patch_cells // 2
    )

    patch = np.full(
        (
            patch_cells,
            patch_cells,
        ),
        float(
            CONFIG["belief_prior"]
        ),
        dtype=np.float32,
    )

    height, width = (
        belief_map.shape
    )

    for patch_y in range(
        patch_cells
    ):
        map_y = (
            gy
            + patch_y
            - radius
        )

        if not 0 <= map_y < height:
            continue

        for patch_x in range(
            patch_cells
        ):
            map_x = (
                gx
                + patch_x
                - radius
            )

            if not 0 <= map_x < width:
                continue

            patch[
                patch_y,
                patch_x,
            ] = float(
                belief_map[
                    map_y,
                    map_x,
                ]
            )

    return patch


def build_destination_mask(
    sender_id,
    uavs,
):
    choices = communication_choices(
        sender_id,
        len(uavs),
    )

    mask = np.zeros(
        len(choices),
        dtype=np.int8,
    )

    # Silent is always structurally valid.
    mask[0] = 1

    sender = uavs[sender_id]

    if not sender.active:
        return mask

    gcs_position = np.asarray(
        CONFIG["gcs_position"],
        dtype=np.float64,
    )

    for index, destination in enumerate(
        choices[1:],
        start=1,
    ):
        if destination == GCS_NODE:
            distance = float(
                np.linalg.norm(
                    sender.position
                    - gcs_position
                )
            )
            mask[index] = int(
                distance
                <= float(
                    CONFIG[
                        "gcs_contact_range_m"
                    ]
                )
            )
            continue

        peer = uavs[destination]

        if not peer.active:
            continue

        distance = float(
            np.linalg.norm(
                sender.position
                - peer.position
            )
        )

        mask[index] = int(
            distance
            <= float(
                CONFIG[
                    "peer_contact_range_m"
                ]
            )
        )

    return mask


def build_agent_observation(
    uav_id,
    uavs,
    belief_maps,
    report_buffers,
    pending_reports,
    current_step,
):
    uav = uavs[uav_id]

    map_size = float(
        CONFIG["map_size"]
    )
    altitude_min = float(
        CONFIG["altitude_min"]
    )
    altitude_span = max(
        1e-9,
        float(CONFIG["altitude_max"])
        - altitude_min,
    )
    max_speed = max(
        1e-9,
        float(CONFIG["max_speed"]),
    )
    battery_capacity = max(
        1e-9,
        float(CONFIG["battery_j"]),
    )

    self_state = np.array(
        [
            np.clip(
                uav.position[0]
                / map_size,
                0.0,
                1.0,
            ),
            np.clip(
                uav.position[1]
                / map_size,
                0.0,
                1.0,
            ),
            np.clip(
                (
                    uav.position[2]
                    - altitude_min
                )
                / altitude_span,
                0.0,
                1.0,
            ),
            np.clip(
                uav.velocity[0]
                / max_speed,
                -1.0,
                1.0,
            ),
            np.clip(
                uav.velocity[1]
                / max_speed,
                -1.0,
                1.0,
            ),
            np.clip(
                uav.velocity[2]
                / max_speed,
                -1.0,
                1.0,
            ),
            np.clip(
                uav.battery_j
                / battery_capacity,
                0.0,
                1.0,
            ),
            float(uav.active),
        ],
        dtype=np.float32,
    )

    gcs_relative = (
        _normalized_relative_vector(
            uav.position,
            np.asarray(
                CONFIG["gcs_position"],
                dtype=np.float64,
            ),
        )
    )

    peer_rows = []

    for peer_id, peer in enumerate(
        uavs
    ):
        if peer_id == uav_id:
            continue

        row = np.zeros(
            5,
            dtype=np.float32,
        )

        distance = float(
            np.linalg.norm(
                peer.position
                - uav.position
            )
        )

        visible = bool(
            uav.active
            and peer.active
            and distance
            <= float(
                CONFIG[
                    "peer_contact_range_m"
                ]
            )
        )

        if visible:
            row[:3] = (
                _normalized_relative_vector(
                    uav.position,
                    peer.position,
                )
            )
            row[3] = float(
                np.clip(
                    distance
                    / max(
                        1e-9,
                        float(
                            CONFIG[
                                "peer_contact_range_m"
                            ]
                        ),
                    ),
                    0.0,
                    1.0,
                )
            )
            row[4] = 1.0

        peer_rows.append(
            row
        )

    neighbors = np.stack(
        peer_rows,
        axis=0,
    ).astype(
        np.float32
    )

    buffer = report_buffers[
        uav_id
    ]

    max_buffer_reports = max(
        1.0,
        float(
            CONFIG["buffer_bytes"]
        )
        / max(
            1.0,
            float(
                CONFIG["report_bytes"]
            ),
        ),
    )

    used_fraction = np.clip(
        buffer_used_bytes(buffer)
        / max(
            1.0,
            float(
                CONFIG["buffer_bytes"]
            ),
        ),
        0.0,
        1.0,
    )

    report_fraction = np.clip(
        len(buffer)
        / max_buffer_reports,
        0.0,
        1.0,
    )

    oldest_age_fraction = 0.0

    for report in buffer:
        if report.ttl_s <= 0.0:
            continue

        oldest_age_fraction = max(
            oldest_age_fraction,
            float(
                np.clip(
                    report_age_s(
                        report,
                        current_step,
                    )
                    / report.ttl_s,
                    0.0,
                    1.0,
                )
            ),
        )

    source_pending = sum(
        int(
            report.source_uav
            == uav_id
        )
        for report
        in pending_reports
    )

    pending_fraction = np.clip(
        source_pending
        / max_buffer_reports,
        0.0,
        1.0,
    )

    buffer_state = np.array(
        [
            used_fraction,
            report_fraction,
            oldest_age_fraction,
            pending_fraction,
        ],
        dtype=np.float32,
    )

    return {
        "self_state": self_state,
        "gcs_relative": gcs_relative,
        "neighbors": neighbors,
        "belief_patch": (
            extract_belief_patch(
                belief_maps[uav_id],
                uav.position,
            )
        ),
        "buffer": buffer_state,
        "destination_mask": (
            build_destination_mask(
                uav_id,
                uavs,
            )
        ),
    }


def build_global_state(
    uavs,
    targets,
    report_buffers,
    pending_reports,
    gcs_received_target_ids,
):
    values = []

    map_size = float(
        CONFIG["map_size"]
    )
    altitude_min = float(
        CONFIG["altitude_min"]
    )
    altitude_span = max(
        1e-9,
        float(CONFIG["altitude_max"])
        - altitude_min,
    )
    max_speed = max(
        1e-9,
        float(CONFIG["max_speed"]),
    )
    battery_capacity = max(
        1e-9,
        float(CONFIG["battery_j"]),
    )

    for uav in uavs:
        values.extend(
            [
                np.clip(
                    uav.position[0]
                    / map_size,
                    0.0,
                    1.0,
                ),
                np.clip(
                    uav.position[1]
                    / map_size,
                    0.0,
                    1.0,
                ),
                np.clip(
                    (
                        uav.position[2]
                        - altitude_min
                    )
                    / altitude_span,
                    0.0,
                    1.0,
                ),
                np.clip(
                    uav.velocity[0]
                    / max_speed,
                    -1.0,
                    1.0,
                ),
                np.clip(
                    uav.velocity[1]
                    / max_speed,
                    -1.0,
                    1.0,
                ),
                np.clip(
                    uav.velocity[2]
                    / max_speed,
                    -1.0,
                    1.0,
                ),
                np.clip(
                    uav.battery_j
                    / battery_capacity,
                    0.0,
                    1.0,
                ),
                float(uav.active),
            ]
        )

    for target in targets:
        values.extend(
            [
                np.clip(
                    target.position[0]
                    / map_size,
                    0.0,
                    1.0,
                ),
                np.clip(
                    target.position[1]
                    / map_size,
                    0.0,
                    1.0,
                ),
                float(
                    target.confirmed
                ),
                float(
                    target.id
                    in gcs_received_target_ids
                ),
            ]
        )

    max_buffer_reports = max(
        1.0,
        float(
            CONFIG["buffer_bytes"]
        )
        / max(
            1.0,
            float(
                CONFIG["report_bytes"]
            ),
        ),
    )

    for buffer in report_buffers:
        values.extend(
            [
                np.clip(
                    buffer_used_bytes(
                        buffer
                    )
                    / max(
                        1.0,
                        float(
                            CONFIG[
                                "buffer_bytes"
                            ]
                        ),
                    ),
                    0.0,
                    1.0,
                ),
                np.clip(
                    len(buffer)
                    / max_buffer_reports,
                    0.0,
                    1.0,
                ),
            ]
        )

    values.append(
        np.clip(
            len(pending_reports)
            / max(
                1.0,
                float(
                    CONFIG["num_targets"]
                ),
            ),
            0.0,
            1.0,
        )
    )

    return np.asarray(
        values,
        dtype=np.float32,
    )


In [ ]:
class UAVSearchEnv(gym.Env):
    """Cooperative multi-UAV search environment with structured MARL actions."""

    metadata: ClassVar[dict] = {
        "render_modes": [],
    }

    def __init__(
        self,
        backend_name=None,
        network_backend_kwargs=None,
    ):
        super().__init__()

        self.num_uavs = int(
            CONFIG["num_uavs"]
        )
        self.num_targets = int(
            CONFIG["num_targets"]
        )

        if self.num_uavs < 1:
            raise ValueError(
                "num_uavs must be >= 1"
            )

        patch_cells = int(
            CONFIG["belief_patch_cells"]
        )

        if (
            patch_cells < 1
            or patch_cells % 2 == 0
        ):
            raise ValueError(
                "belief_patch_cells must be "
                "a positive odd integer"
            )

        self.agent_ids = tuple(
            f"uav_{uav_id}"
            for uav_id
            in range(self.num_uavs)
        )

        self.backend_name = (
            CONFIG.get(
                "network_backend",
                "simple",
            )
            if backend_name is None
            else backend_name
        )

        self.network_backend_kwargs = dict(
            network_backend_kwargs
            or {}
        )

        destination_count = (
            self.num_uavs + 1
        )

        self.action_space = (
            gym.spaces.Dict(
                {
                    agent_id: (
                        gym.spaces.Dict(
                            {
                                "motion": (
                                    gym.spaces.Box(
                                        low=-1.0,
                                        high=1.0,
                                        shape=(3,),
                                        dtype=np.float32,
                                    )
                                ),
                                "destination": (
                                    gym.spaces.Discrete(
                                        destination_count
                                    )
                                ),
                                "power": (
                                    gym.spaces.Box(
                                        low=-1.0,
                                        high=1.0,
                                        shape=(1,),
                                        dtype=np.float32,
                                    )
                                ),
                            }
                        )
                    )
                    for agent_id
                    in self.agent_ids
                }
            )
        )

        self.observation_space = (
            gym.spaces.Dict(
                {
                    agent_id: (
                        gym.spaces.Dict(
                            {
                                "self_state": (
                                    gym.spaces.Box(
                                        low=-1.0,
                                        high=1.0,
                                        shape=(8,),
                                        dtype=np.float32,
                                    )
                                ),
                                "gcs_relative": (
                                    gym.spaces.Box(
                                        low=-1.0,
                                        high=1.0,
                                        shape=(3,),
                                        dtype=np.float32,
                                    )
                                ),
                                "neighbors": (
                                    gym.spaces.Box(
                                        low=-1.0,
                                        high=1.0,
                                        shape=(
                                            self.num_uavs
                                            - 1,
                                            5,
                                        ),
                                        dtype=np.float32,
                                    )
                                ),
                                "belief_patch": (
                                    gym.spaces.Box(
                                        low=0.0,
                                        high=1.0,
                                        shape=(
                                            patch_cells,
                                            patch_cells,
                                        ),
                                        dtype=np.float32,
                                    )
                                ),
                                "buffer": (
                                    gym.spaces.Box(
                                        low=0.0,
                                        high=1.0,
                                        shape=(4,),
                                        dtype=np.float32,
                                    )
                                ),
                                "destination_mask": (
                                    gym.spaces.MultiBinary(
                                        destination_count
                                    )
                                ),
                            }
                        )
                    )
                    for agent_id
                    in self.agent_ids
                }
            )
        )

        state_size = (
            self.num_uavs * 8
            + self.num_targets * 4
            + self.num_uavs * 2
            + 1
        )

        self.state_space = (
            gym.spaces.Box(
                low=-1.0,
                high=1.0,
                shape=(state_size,),
                dtype=np.float32,
            )
        )

        self.current_step = 0
        self.episode_seed = None
        self.rng = None
        self.uavs = None
        self.targets = None
        self.obstacles = None
        self.belief_maps = None
        self.report_buffers = None
        self.pending_reports = None
        self.gcs_received_target_ids = None
        self.network_backend = None
        self._episode_done = False

    def _make_network_backend(
        self,
        seed,
    ):
        kwargs = dict(
            self.network_backend_kwargs
        )

        normalized = str(
            self.backend_name
        ).strip().lower()

        if normalized in {
            "uavnetsim",
            "uav_net_sim",
        }:
            kwargs.setdefault(
                "seed",
                int(seed),
            )

        return create_network_backend(
            self.backend_name,
            **kwargs,
        )

    def _canonical_actions(
        self,
        actions,
    ):
        if not isinstance(
            actions,
            dict,
        ):
            raise TypeError(
                "actions must be a dict "
                "keyed by UAV agent id"
            )

        if set(actions) != set(
            self.agent_ids
        ):
            raise ValueError(
                "actions must contain exactly "
                "one entry for every UAV"
            )

        canonical = {}

        for agent_id in self.agent_ids:
            action = actions[
                agent_id
            ]

            if not isinstance(
                action,
                dict,
            ):
                raise TypeError(
                    f"{agent_id} action "
                    "must be a dict"
                )

            if set(action) != {
                "motion",
                "destination",
                "power",
            }:
                raise ValueError(
                    f"{agent_id} action must "
                    "contain exactly motion, "
                    "destination, power"
                )

            destination = action[
                "destination"
            ]

            if isinstance(
                destination,
                (bool, np.bool_),
            ):
                raise TypeError(
                    "destination must be "
                    "an integer"
                )

            if not isinstance(
                destination,
                (int, np.integer),
            ):
                raise TypeError(
                    "destination must be "
                    "an integer"
                )

            motion = np.asarray(
                action["motion"],
                dtype=np.float32,
            )

            power = np.asarray(
                action["power"],
                dtype=np.float32,
            )

            if power.shape == ():
                power = power.reshape(
                    1
                )

            candidate = {
                "motion": motion,
                "destination": int(
                    destination
                ),
                "power": power,
            }

            subspace = (
                self.action_space.spaces[
                    agent_id
                ]
            )

            if not subspace.contains(
                candidate
            ):
                raise ValueError(
                    f"{agent_id} action is "
                    "outside action_space"
                )

            canonical[
                agent_id
            ] = candidate

        return canonical

    def _decode_actions(
        self,
        actions,
    ):
        return [
            decode_hybrid_action(
                uav_id,
                actions[
                    self.agent_ids[
                        uav_id
                    ]
                ],
                self.num_uavs,
            )
            for uav_id
            in range(self.num_uavs)
        ]

    def _observations(self):
        observations = {
            agent_id: (
                build_agent_observation(
                    uav_id,
                    self.uavs,
                    self.belief_maps,
                    self.report_buffers,
                    self.pending_reports,
                    self.current_step,
                )
            )
            for uav_id, agent_id
            in enumerate(
                self.agent_ids
            )
        }

        if not self.observation_space.contains(
            observations
        ):
            raise RuntimeError(
                "generated observation is "
                "outside observation_space"
            )

        return observations

    def global_state(self):
        if self.uavs is None:
            raise RuntimeError(
                "reset must be called before "
                "global_state"
            )

        state = build_global_state(
            self.uavs,
            self.targets,
            self.report_buffers,
            self.pending_reports,
            self.gcs_received_target_ids,
        )

        if not self.state_space.contains(
            state
        ):
            raise RuntimeError(
                "generated global state is "
                "outside state_space"
            )

        return state

    def reset(
        self,
        *,
        seed=None,
        options=None,
    ):
        del options

        if seed is None:
            seed = int(
                CONFIG.get(
                    "seed",
                    44,
                )
            )

        seed = int(seed)

        super().reset(
            seed=seed
        )

        if self.network_backend is not None:
            close_method = getattr(
                self.network_backend,
                "close",
                None,
            )

            if callable(
                close_method
            ):
                close_method()

        set_seed(seed)

        (
            self.rng,
            self.uavs,
            self.targets,
            self.obstacles,
        ) = create_world(seed)

        self.belief_maps = (
            create_belief_maps()
        )
        self.report_buffers = (
            create_report_buffers()
        )
        self.pending_reports = (
            create_pending_reports()
        )
        self.gcs_received_target_ids = (
            create_gcs_received_target_ids()
        )

        self.current_step = 0
        self.episode_seed = seed
        self._episode_done = False

        self.network_backend = (
            self._make_network_backend(
                seed
            )
        )

        self.network_backend.reset(
            uavs=self.uavs,
            gcs_position=CONFIG[
                "gcs_position"
            ],
            obstacles=self.obstacles,
            report_buffers=(
                self.report_buffers
            ),
            pending_reports=(
                self.pending_reports
            ),
            gcs_received_target_ids=(
                self.gcs_received_target_ids
            ),
        )

        observations = (
            self._observations()
        )

        info = {
            "step": 0,
            "seed": seed,
            "backend": str(
                self.backend_name
            ),
            "state": (
                self.global_state()
            ),
        }

        return observations, info

    def step(
        self,
        actions,
    ):
        if self.uavs is None:
            raise RuntimeError(
                "reset must be called before step"
            )

        if self._episode_done:
            raise RuntimeError(
                "episode is done; call reset "
                "before stepping again"
            )

        canonical_actions = (
            self._canonical_actions(
                actions
            )
        )
        hybrid_actions = (
            self._decode_actions(
                canonical_actions
            )
        )

        next_step = (
            self.current_step + 1
        )

        expired_target_ids = set()

        for buffer in self.report_buffers:
            expired_target_ids.update(
                report.target_id
                for report
                in remove_expired_reports(
                    buffer,
                    next_step,
                )
            )

        expired_target_ids.update(
            report.target_id
            for report
            in remove_expired_reports(
                self.pending_reports,
                next_step,
            )
        )

        flush_before = (
            flush_pending_reports(
                self.pending_reports,
                self.report_buffers,
                next_step,
                self.gcs_received_target_ids,
                self.uavs,
            )
        )

        expired_target_ids.update(
            flush_before[
                "expired_target_ids"
            ]
        )

        confirmed_before = {
            target.id
            for target in self.targets
            if target.confirmed
        }

        delivered_before = set(
            self.gcs_received_target_ids
        )

        motion_actions = np.stack(
            [
                action.movement
                for action
                in hybrid_actions
            ],
            axis=0,
        )

        motion_result = (
            apply_swarm_motion(
                self.uavs,
                motion_actions,
                obstacles=self.obstacles,
                dt=CONFIG["dt"],
            )
        )

        sensing_record_count = 0
        information_gain_bits = 0.0
        confirmation_events = []

        for uav_id, uav in enumerate(
            self.uavs
        ):
            records = sense_and_update(
                uav,
                self.belief_maps[
                    uav_id
                ],
                self.targets,
                self.rng,
            )

            sensing_record_count += len(
                records
            )
            information_gain_bits += (
                sensing_information_gain_bits(
                    records
                )
            )

            for record in records:
                confirmation_events.extend(
                    create_confirmation_events(
                        uav,
                        record,
                        next_step,
                    )
                )

        _, event_log = (
            process_confirmation_events(
                confirmation_events,
                self.targets,
                self.report_buffers,
                self.pending_reports,
                self.gcs_received_target_ids,
            )
        )

        flush_after_confirmation = (
            flush_pending_reports(
                self.pending_reports,
                self.report_buffers,
                next_step,
                self.gcs_received_target_ids,
                self.uavs,
            )
        )

        expired_target_ids.update(
            flush_after_confirmation[
                "expired_target_ids"
            ]
        )

        intents = []

        peer_transfer_states = (
            self.network_backend
            .peer_transfer_states
        )

        for uav_id, (
            uav,
            hybrid_action,
        ) in enumerate(
            zip(
                self.uavs,
                hybrid_actions,
            )
        ):
            if not uav.active:
                continue

            intent = (
                build_transmission_intent(
                    uav_id,
                    hybrid_action,
                    self.report_buffers[
                        uav_id
                    ],
                    next_step,
                    peer_transfer_states=(
                        peer_transfer_states
                    ),
                )
            )

            if intent is not None:
                intents.append(
                    intent
                )

        self.network_backend.sync_positions(
            self.uavs
        )

        network_results = (
            self.network_backend.step(
                intents,
                dt=CONFIG["dt"],
                current_step=next_step,
            )
        )

        flush_after_network = (
            flush_pending_reports(
                self.pending_reports,
                self.report_buffers,
                next_step,
                self.gcs_received_target_ids,
                self.uavs,
            )
        )

        expired_target_ids.update(
            flush_after_network[
                "expired_target_ids"
            ]
        )

        communication_energy = (
            self.network_backend
            .last_step_communication_energy_by_uav()
        )

        energy_result = (
            apply_uav_energy_budget(
                self.uavs,
                hybrid_actions,
                communication_energy,
                dt=CONFIG["dt"],
            )
        )

        confirmed_after = {
            target.id
            for target in self.targets
            if target.confirmed
        }

        delivered_after = set(
            self.gcs_received_target_ids
        )

        newly_confirmed = sorted(
            confirmed_after
            - confirmed_before
        )
        newly_delivered = sorted(
            delivered_after
            - delivered_before
        )

        false_confirmation_count = sum(
            event.confirmation_type
            == "false_confirmation"
            for event in event_log
        )

        blocked_count = int(
            np.count_nonzero(
                motion_result[
                    "blocked"
                ]
            )
        )
        boundary_count = int(
            np.count_nonzero(
                motion_result[
                    "boundary_clipped"
                ]
            )
        )

        success = (
            len(
                self.gcs_received_target_ids
            )
            == self.num_targets
        )

        all_inactive = not any(
            uav.active
            for uav in self.uavs
        )

        reward_components = {
            "information_gain": (
                float(
                    CONFIG[
                        "reward_info_gain"
                    ]
                )
                * information_gain_bits
            ),
            "confirmation": (
                float(
                    CONFIG[
                        "reward_confirmation"
                    ]
                )
                * len(
                    newly_confirmed
                )
            ),
            "delivery": (
                float(
                    CONFIG[
                        "reward_delivery"
                    ]
                )
                * len(
                    newly_delivered
                )
            ),
            "false_confirmation": (
                -float(
                    CONFIG[
                        "reward_false_confirmation"
                    ]
                )
                * false_confirmation_count
            ),
            "blocked_motion": (
                -float(
                    CONFIG[
                        "reward_blocked"
                    ]
                )
                * blocked_count
            ),
            "boundary": (
                -float(
                    CONFIG[
                        "reward_boundary"
                    ]
                )
                * boundary_count
            ),
            "expired_report": (
                -float(
                    CONFIG[
                        "reward_expired_report"
                    ]
                )
                * len(
                    expired_target_ids
                )
            ),
            "energy": (
                -float(
                    CONFIG[
                        "reward_energy_per_kj"
                    ]
                )
                * float(
                    np.sum(
                        energy_result[
                            "total_j"
                        ]
                    )
                )
                / 1000.0
            ),
            "step": (
                -float(
                    CONFIG[
                        "reward_step_penalty"
                    ]
                )
            ),
            "success_bonus": (
                float(
                    CONFIG[
                        "reward_all_delivered_bonus"
                    ]
                )
                if success
                else 0.0
            ),
        }

        reward = float(
            sum(
                reward_components.values()
            )
        )

        self.current_step = next_step

        terminated = bool(
            success
            or all_inactive
        )
        truncated = bool(
            (
                self.current_step
                >= int(
                    CONFIG[
                        "max_steps"
                    ]
                )
            )
            and not terminated
        )

        self._episode_done = bool(
            terminated
            or truncated
        )

        observations = (
            self._observations()
        )
        state = self.global_state()

        agent_rewards = {
            agent_id: reward
            for agent_id
            in self.agent_ids
        }

        info = {
            "step": self.current_step,
            "backend": str(
                self.backend_name
            ),
            "state": state,
            "agent_rewards": (
                agent_rewards
            ),
            "reward_components": (
                reward_components
            ),
            "motion": motion_result,
            "network_results": (
                network_results
            ),
            "network_metrics": (
                self.network_backend
                .metrics()
            ),
            "sensing_record_count": (
                sensing_record_count
            ),
            "information_gain_bits": (
                information_gain_bits
            ),
            "newly_confirmed_target_ids": (
                newly_confirmed
            ),
            "newly_delivered_target_ids": (
                newly_delivered
            ),
            "false_confirmation_count": (
                int(
                    false_confirmation_count
                )
            ),
            "expired_target_ids": sorted(
                expired_target_ids
            ),
            "energy": energy_result,
            "success": bool(success),
            "all_uavs_inactive": bool(
                all_inactive
            ),
        }

        return (
            observations,
            reward,
            terminated,
            truncated,
            info,
        )

    def close(self):
        if self.network_backend is None:
            return

        close_method = getattr(
            self.network_backend,
            "close",
            None,
        )

        if callable(close_method):
            close_method()

        self.network_backend = None


In [ ]:
AGENT_OBSERVATION_VECTOR_KEYS = (
    "self_state",
    "gcs_relative",
    "neighbors",
    "belief_patch",
    "buffer",
    "destination_mask",
)


def flatten_agent_observation(
    observation,
):
    if not isinstance(
        observation,
        dict,
    ):
        raise TypeError(
            "observation must be a dict"
        )

    if set(observation) != set(
        AGENT_OBSERVATION_VECTOR_KEYS
    ):
        raise ValueError(
            "observation keys do not match "
            "the MASAC observation contract"
        )

    parts = []

    for key in (
        AGENT_OBSERVATION_VECTOR_KEYS
    ):
        values = np.asarray(
            observation[key],
            dtype=np.float32,
        ).reshape(-1)

        if not np.all(
            np.isfinite(values)
        ):
            raise ValueError(
                f"{key} contains non-finite values"
            )

        parts.append(
            values
        )

    return np.concatenate(
        parts,
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


def observations_to_masac_arrays(
    observations,
    agent_ids,
):
    if set(observations) != set(
        agent_ids
    ):
        raise ValueError(
            "observations must contain exactly "
            "the configured agent ids"
        )

    observation_vectors = np.stack(
        [
            flatten_agent_observation(
                observations[agent_id]
            )
            for agent_id
            in agent_ids
        ],
        axis=0,
    ).astype(
        np.float32
    )

    destination_masks = np.stack(
        [
            np.asarray(
                observations[
                    agent_id
                ][
                    "destination_mask"
                ],
                dtype=np.float32,
            )
            for agent_id
            in agent_ids
        ],
        axis=0,
    )

    if (
        destination_masks.ndim != 2
        or np.any(
            destination_masks.sum(
                axis=-1
            )
            < 1.0
        )
    ):
        raise ValueError(
            "every agent needs at least "
            "one valid destination"
        )

    return (
        observation_vectors,
        destination_masks,
    )


def env_actions_to_masac_arrays(
    actions,
    agent_ids,
):
    continuous = []
    destination_indices = []

    for agent_id in agent_ids:
        action = actions[
            agent_id
        ]

        motion = np.asarray(
            action["motion"],
            dtype=np.float32,
        )

        power = np.asarray(
            action["power"],
            dtype=np.float32,
        ).reshape(-1)

        if (
            motion.shape != (3,)
            or power.shape != (1,)
        ):
            raise ValueError(
                "invalid hybrid action shape"
            )

        continuous.append(
            np.concatenate(
                [
                    motion,
                    power,
                ],
                axis=0,
            )
        )
        destination_indices.append(
            int(
                action["destination"]
            )
        )

    return (
        np.stack(
            continuous,
            axis=0,
        ).astype(
            np.float32
        ),
        np.asarray(
            destination_indices,
            dtype=np.int64,
        ),
    )


def masac_arrays_to_env_actions(
    continuous_actions,
    destination_indices,
    agent_ids,
):
    continuous_actions = np.asarray(
        continuous_actions,
        dtype=np.float32,
    )
    destination_indices = np.asarray(
        destination_indices,
        dtype=np.int64,
    )

    if continuous_actions.shape != (
        len(agent_ids),
        4,
    ):
        raise ValueError(
            "continuous_actions must have "
            "shape (num_agents, 4)"
        )

    if destination_indices.shape != (
        len(agent_ids),
    ):
        raise ValueError(
            "destination_indices must have "
            "shape (num_agents,)"
        )

    return {
        agent_id: {
            "motion": (
                continuous_actions[
                    index,
                    :3,
                ].copy()
            ),
            "destination": int(
                destination_indices[
                    index
                ]
            ),
            "power": np.asarray(
                [
                    continuous_actions[
                        index,
                        3,
                    ]
                ],
                dtype=np.float32,
            ),
        }
        for index, agent_id
        in enumerate(agent_ids)
    }


def sample_valid_random_actions(
    observations,
    agent_ids,
    rng,
):
    actions = {}

    for agent_id in agent_ids:
        mask = np.asarray(
            observations[
                agent_id
            ][
                "destination_mask"
            ],
            dtype=np.int8,
        )

        valid = np.flatnonzero(
            mask
        )

        if valid.size == 0:
            raise RuntimeError(
                "destination mask has "
                "no valid action"
            )

        actions[
            agent_id
        ] = {
            "motion": rng.uniform(
                -1.0,
                1.0,
                size=3,
            ).astype(
                np.float32
            ),
            "destination": int(
                rng.choice(
                    valid
                )
            ),
            "power": rng.uniform(
                -1.0,
                1.0,
                size=1,
            ).astype(
                np.float32
            ),
        }

    return actions


In [ ]:
def build_mlp(
    input_dim,
    hidden_dims,
    output_dim,
):
    dims = (
        int(input_dim),
        *tuple(
            int(value)
            for value
            in hidden_dims
        ),
        int(output_dim),
    )

    layers = []

    for index in range(
        len(dims) - 2
    ):
        layers.extend(
            [
                nn.Linear(
                    dims[index],
                    dims[index + 1],
                ),
                nn.ReLU(),
            ]
        )

    layers.append(
        nn.Linear(
            dims[-2],
            dims[-1],
        )
    )

    return nn.Sequential(
        *layers
    )


def masked_categorical_logits(
    logits,
    destination_mask,
):
    if logits.shape != (
        destination_mask.shape
    ):
        raise ValueError(
            "logits and destination_mask "
            "must have the same shape"
        )

    mask = (
        destination_mask > 0.5
    )

    if torch.any(
        mask.sum(
            dim=-1
        )
        < 1
    ):
        raise ValueError(
            "each categorical row needs "
            "at least one valid action"
        )

    negative_large = torch.finfo(
        logits.dtype
    ).min

    return logits.masked_fill(
        ~mask,
        negative_large,
    )


def straight_through_categorical_sample(
    logits,
    destination_mask,
    temperature,
    deterministic=False,
):
    temperature = float(
        temperature
    )

    if temperature <= 0.0:
        raise ValueError(
            "temperature must be > 0"
        )

    masked_logits = (
        masked_categorical_logits(
            logits,
            destination_mask,
        )
    )

    log_probabilities = (
        F.log_softmax(
            masked_logits,
            dim=-1,
        )
    )
    probabilities = torch.exp(
        log_probabilities
    )

    if deterministic:
        indices = torch.argmax(
            masked_logits,
            dim=-1,
        )
        hard = F.one_hot(
            indices,
            num_classes=(
                masked_logits.shape[-1]
            ),
        ).to(
            masked_logits.dtype
        )
        action = hard
    else:
        uniform = torch.rand_like(
            masked_logits
        ).clamp_(
            1e-6,
            1.0 - 1e-6,
        )

        gumbel_noise = -torch.log(
            -torch.log(
                uniform
            )
        )

        soft = F.softmax(
            (
                masked_logits
                + gumbel_noise
            )
            / temperature,
            dim=-1,
        )

        indices = torch.argmax(
            soft,
            dim=-1,
        )
        hard = F.one_hot(
            indices,
            num_classes=(
                masked_logits.shape[-1]
            ),
        ).to(
            masked_logits.dtype
        )

        # Straight-through estimator:
        # forward value is hard one-hot,
        # backward gradient follows soft.
        action = (
            hard
            - soft.detach()
            + soft
        )

    selected_log_probability = (
        hard
        * log_probabilities
    ).sum(
        dim=-1,
        keepdim=True,
    )

    entropy = -(
        probabilities
        * log_probabilities
    ).sum(
        dim=-1,
        keepdim=True,
    )

    return {
        "action": action,
        "hard_action": hard,
        "indices": indices,
        "log_probability": (
            selected_log_probability
        ),
        "entropy": entropy,
        "probabilities": probabilities,
        "log_probabilities": (
            log_probabilities
        ),
    }


class HybridMASACActor(nn.Module):
    def __init__(
        self,
        observation_dim,
        continuous_dim,
        discrete_dim,
        hidden_dims,
    ):
        super().__init__()

        hidden_dims = tuple(
            int(value)
            for value
            in hidden_dims
        )

        if not hidden_dims:
            raise ValueError(
                "hidden_dims must not be empty"
            )

        self.observation_dim = int(
            observation_dim
        )
        self.continuous_dim = int(
            continuous_dim
        )
        self.discrete_dim = int(
            discrete_dim
        )

        self.encoder = build_mlp(
            self.observation_dim,
            hidden_dims[:-1],
            hidden_dims[-1],
        )

        feature_dim = (
            hidden_dims[-1]
        )

        self.mean_head = nn.Linear(
            feature_dim,
            self.continuous_dim,
        )
        self.log_std_head = nn.Linear(
            feature_dim,
            self.continuous_dim,
        )
        self.discrete_head = nn.Linear(
            feature_dim,
            self.discrete_dim,
        )

    def forward(
        self,
        observations,
    ):
        features = self.encoder(
            observations
        )

        mean = self.mean_head(
            features
        )

        log_std = (
            self.log_std_head(
                features
            ).clamp(
                min=float(
                    CONFIG[
                        "masac_log_std_min"
                    ]
                ),
                max=float(
                    CONFIG[
                        "masac_log_std_max"
                    ]
                ),
            )
        )

        logits = self.discrete_head(
            features
        )

        return (
            mean,
            log_std,
            logits,
        )

    def sample(
        self,
        observations,
        destination_mask,
        deterministic=False,
    ):
        (
            mean,
            log_std,
            logits,
        ) = self(
            observations
        )

        std = torch.exp(
            log_std
        )
        distribution = Normal(
            mean,
            std,
        )

        if deterministic:
            pre_tanh = mean
        else:
            pre_tanh = (
                distribution.rsample()
            )

        continuous_action = (
            torch.tanh(
                pre_tanh
            )
        )

        log_probability = (
            distribution.log_prob(
                pre_tanh
            )
            - torch.log(
                1.0
                - continuous_action.pow(
                    2
                )
                + 1e-6
            )
        ).sum(
            dim=-1,
            keepdim=True,
        )

        discrete = (
            straight_through_categorical_sample(
                logits,
                destination_mask,
                temperature=CONFIG[
                    "masac_gumbel_temperature"
                ],
                deterministic=deterministic,
            )
        )

        return {
            "continuous": (
                continuous_action
            ),
            "continuous_log_probability": (
                log_probability
            ),
            "destination_one_hot": (
                discrete["action"]
            ),
            "destination_hard_one_hot": (
                discrete[
                    "hard_action"
                ]
            ),
            "destination_index": (
                discrete["indices"]
            ),
            "destination_log_probability": (
                discrete[
                    "log_probability"
                ]
            ),
            "destination_entropy": (
                discrete["entropy"]
            ),
        }


class CentralizedQNetwork(nn.Module):
    def __init__(
        self,
        state_dim,
        joint_action_dim,
        hidden_dims,
    ):
        super().__init__()

        self.state_dim = int(
            state_dim
        )
        self.joint_action_dim = int(
            joint_action_dim
        )

        self.network = build_mlp(
            self.state_dim
            + self.joint_action_dim,
            hidden_dims,
            1,
        )

    def forward(
        self,
        state,
        joint_action,
    ):
        if state.ndim != 2:
            raise ValueError(
                "state must be rank 2"
            )

        if joint_action.ndim != 2:
            raise ValueError(
                "joint_action must be rank 2"
            )

        values = torch.cat(
            [
                state,
                joint_action,
            ],
            dim=-1,
        )

        return self.network(
            values
        )


In [ ]:
class HybridReplayBuffer:
    def __init__(
        self,
        capacity,
        num_agents,
        observation_dim,
        state_dim,
        continuous_dim,
        discrete_dim,
        seed,
    ):
        self.capacity = int(
            capacity
        )
        self.num_agents = int(
            num_agents
        )
        self.observation_dim = int(
            observation_dim
        )
        self.state_dim = int(
            state_dim
        )
        self.continuous_dim = int(
            continuous_dim
        )
        self.discrete_dim = int(
            discrete_dim
        )

        if self.capacity < 1:
            raise ValueError(
                "capacity must be >= 1"
            )

        self.rng = (
            np.random.default_rng(
                int(seed)
            )
        )

        self.observations = np.zeros(
            (
                self.capacity,
                self.num_agents,
                self.observation_dim,
            ),
            dtype=np.float32,
        )
        self.next_observations = (
            np.zeros_like(
                self.observations
            )
        )
        self.states = np.zeros(
            (
                self.capacity,
                self.state_dim,
            ),
            dtype=np.float32,
        )
        self.next_states = (
            np.zeros_like(
                self.states
            )
        )
        self.continuous_actions = np.zeros(
            (
                self.capacity,
                self.num_agents,
                self.continuous_dim,
            ),
            dtype=np.float32,
        )
        self.destination_indices = np.zeros(
            (
                self.capacity,
                self.num_agents,
            ),
            dtype=np.int64,
        )
        self.destination_masks = np.zeros(
            (
                self.capacity,
                self.num_agents,
                self.discrete_dim,
            ),
            dtype=np.float32,
        )
        self.next_destination_masks = (
            np.zeros_like(
                self.destination_masks
            )
        )
        self.rewards = np.zeros(
            (
                self.capacity,
                1,
            ),
            dtype=np.float32,
        )
        self.terminated = np.zeros(
            (
                self.capacity,
                1,
            ),
            dtype=np.float32,
        )
        self.truncated = np.zeros(
            (
                self.capacity,
                1,
            ),
            dtype=np.float32,
        )

        self.position = 0
        self.size = 0

    def __len__(self):
        return int(
            self.size
        )

    def add(
        self,
        observations,
        state,
        continuous_actions,
        destination_indices,
        destination_masks,
        reward,
        next_observations,
        next_state,
        next_destination_masks,
        terminated,
        truncated,
    ):
        observations = np.asarray(
            observations,
            dtype=np.float32,
        )
        next_observations = np.asarray(
            next_observations,
            dtype=np.float32,
        )
        state = np.asarray(
            state,
            dtype=np.float32,
        )
        next_state = np.asarray(
            next_state,
            dtype=np.float32,
        )
        continuous_actions = np.asarray(
            continuous_actions,
            dtype=np.float32,
        )
        destination_indices = np.asarray(
            destination_indices,
            dtype=np.int64,
        )
        destination_masks = np.asarray(
            destination_masks,
            dtype=np.float32,
        )
        next_destination_masks = np.asarray(
            next_destination_masks,
            dtype=np.float32,
        )

        expected_observation_shape = (
            self.num_agents,
            self.observation_dim,
        )
        expected_action_shape = (
            self.num_agents,
            self.continuous_dim,
        )
        expected_mask_shape = (
            self.num_agents,
            self.discrete_dim,
        )

        if observations.shape != (
            expected_observation_shape
        ):
            raise ValueError(
                "observations have wrong shape"
            )

        if next_observations.shape != (
            expected_observation_shape
        ):
            raise ValueError(
                "next_observations have wrong shape"
            )

        if state.shape != (
            self.state_dim,
        ):
            raise ValueError(
                "state has wrong shape"
            )

        if next_state.shape != (
            self.state_dim,
        ):
            raise ValueError(
                "next_state has wrong shape"
            )

        if continuous_actions.shape != (
            expected_action_shape
        ):
            raise ValueError(
                "continuous_actions have wrong shape"
            )

        if destination_indices.shape != (
            self.num_agents,
        ):
            raise ValueError(
                "destination_indices have wrong shape"
            )

        if destination_masks.shape != (
            expected_mask_shape
        ):
            raise ValueError(
                "destination_masks have wrong shape"
            )

        if next_destination_masks.shape != (
            expected_mask_shape
        ):
            raise ValueError(
                "next_destination_masks have wrong shape"
            )

        if np.any(
            destination_indices < 0
        ) or np.any(
            destination_indices
            >= self.discrete_dim
        ):
            raise ValueError(
                "destination index out of range"
            )

        slot = int(
            self.position
        )

        self.observations[
            slot
        ] = observations
        self.next_observations[
            slot
        ] = next_observations
        self.states[
            slot
        ] = state
        self.next_states[
            slot
        ] = next_state
        self.continuous_actions[
            slot
        ] = continuous_actions
        self.destination_indices[
            slot
        ] = destination_indices
        self.destination_masks[
            slot
        ] = destination_masks
        self.next_destination_masks[
            slot
        ] = next_destination_masks
        self.rewards[
            slot,
            0,
        ] = float(
            reward
        )
        self.terminated[
            slot,
            0,
        ] = float(
            bool(
                terminated
            )
        )
        self.truncated[
            slot,
            0,
        ] = float(
            bool(
                truncated
            )
        )

        self.position = (
            slot + 1
        ) % self.capacity
        self.size = min(
            self.size + 1,
            self.capacity,
        )

    def state_dict(self):
        size = int(
            self.size
        )

        def packed_tensor(
            array,
        ):
            return torch.from_numpy(
                array[:size].copy()
            )

        return {
            "format_version": 1,
            "capacity": int(
                self.capacity
            ),
            "num_agents": int(
                self.num_agents
            ),
            "observation_dim": int(
                self.observation_dim
            ),
            "state_dim": int(
                self.state_dim
            ),
            "continuous_dim": int(
                self.continuous_dim
            ),
            "discrete_dim": int(
                self.discrete_dim
            ),
            "position": int(
                self.position
            ),
            "size": size,
            "rng_state_json": json.dumps(
                self.rng
                .bit_generator
                .state
            ),
            "observations": packed_tensor(
                self.observations
            ),
            "next_observations": (
                packed_tensor(
                    self.next_observations
                )
            ),
            "states": packed_tensor(
                self.states
            ),
            "next_states": packed_tensor(
                self.next_states
            ),
            "continuous_actions": (
                packed_tensor(
                    self.continuous_actions
                )
            ),
            "destination_indices": (
                packed_tensor(
                    self.destination_indices
                )
            ),
            "destination_masks": (
                packed_tensor(
                    self.destination_masks
                )
            ),
            "next_destination_masks": (
                packed_tensor(
                    self.next_destination_masks
                )
            ),
            "rewards": packed_tensor(
                self.rewards
            ),
            "terminated": packed_tensor(
                self.terminated
            ),
            "truncated": packed_tensor(
                self.truncated
            ),
        }

    def load_state_dict(
        self,
        state,
    ):
        if not isinstance(
            state,
            dict,
        ):
            raise TypeError(
                "replay state must be a dict"
            )

        expected = {
            "capacity": self.capacity,
            "num_agents": self.num_agents,
            "observation_dim": (
                self.observation_dim
            ),
            "state_dim": self.state_dim,
            "continuous_dim": (
                self.continuous_dim
            ),
            "discrete_dim": (
                self.discrete_dim
            ),
        }

        for key, expected_value in (
            expected.items()
        ):
            actual = int(
                state[key]
            )

            if actual != int(
                expected_value
            ):
                raise ValueError(
                    f"replay {key} mismatch: "
                    f"{actual} != "
                    f"{expected_value}"
                )

        size = int(
            state["size"]
        )
        position = int(
            state["position"]
        )

        if not 0 <= size <= self.capacity:
            raise ValueError(
                "invalid replay size"
            )

        if not 0 <= position < self.capacity:
            raise ValueError(
                "invalid replay position"
            )

        if (
            size < self.capacity
            and position != size
        ):
            raise ValueError(
                "partial replay position "
                "must equal size"
            )

        tensor_keys = (
            "observations",
            "next_observations",
            "states",
            "next_states",
            "continuous_actions",
            "destination_indices",
            "destination_masks",
            "next_destination_masks",
            "rewards",
            "terminated",
            "truncated",
        )

        destination_arrays = {
            "observations": (
                self.observations
            ),
            "next_observations": (
                self.next_observations
            ),
            "states": self.states,
            "next_states": (
                self.next_states
            ),
            "continuous_actions": (
                self.continuous_actions
            ),
            "destination_indices": (
                self.destination_indices
            ),
            "destination_masks": (
                self.destination_masks
            ),
            "next_destination_masks": (
                self.next_destination_masks
            ),
            "rewards": self.rewards,
            "terminated": (
                self.terminated
            ),
            "truncated": (
                self.truncated
            ),
        }

        for key in tensor_keys:
            destination = (
                destination_arrays[
                    key
                ]
            )
            destination.fill(
                0
            )

            source_tensor = (
                state[key]
            )

            if not isinstance(
                source_tensor,
                torch.Tensor,
            ):
                raise TypeError(
                    f"replay {key} must "
                    "be a tensor"
                )

            source = (
                source_tensor
                .detach()
                .cpu()
                .numpy()
            )

            expected_shape = (
                destination[:size]
                .shape
            )

            if source.shape != (
                expected_shape
            ):
                raise ValueError(
                    f"replay {key} has "
                    "wrong shape"
                )

            destination[
                :size
            ] = source.astype(
                destination.dtype,
                copy=False,
            )

        self.position = position
        self.size = size

        self.rng.bit_generator.state = (
            json.loads(
                state[
                    "rng_state_json"
                ]
            )
        )

    def sample(
        self,
        batch_size,
        device,
    ):
        batch_size = int(
            batch_size
        )

        if batch_size < 1:
            raise ValueError(
                "batch_size must be >= 1"
            )

        if self.size < batch_size:
            raise ValueError(
                "not enough replay samples"
            )

        indices = self.rng.integers(
            0,
            self.size,
            size=batch_size,
        )

        def tensor(
            array,
            dtype=torch.float32,
        ):
            return torch.as_tensor(
                array[indices],
                dtype=dtype,
                device=device,
            )

        return {
            "observations": tensor(
                self.observations
            ),
            "next_observations": tensor(
                self.next_observations
            ),
            "states": tensor(
                self.states
            ),
            "next_states": tensor(
                self.next_states
            ),
            "continuous_actions": tensor(
                self.continuous_actions
            ),
            "destination_indices": tensor(
                self.destination_indices,
                dtype=torch.long,
            ),
            "destination_masks": tensor(
                self.destination_masks
            ),
            "next_destination_masks": tensor(
                self.next_destination_masks
            ),
            "rewards": tensor(
                self.rewards
            ),
            "terminated": tensor(
                self.terminated
            ),
            "truncated": tensor(
                self.truncated
            ),
        }


In [ ]:
class HybridMASAC:
    def __init__(
        self,
        env,
        device=None,
    ):
        if not isinstance(
            env,
            UAVSearchEnv,
        ):
            raise TypeError(
                "env must be UAVSearchEnv"
            )

        self.agent_ids = tuple(
            env.agent_ids
        )
        self.num_agents = len(
            self.agent_ids
        )

        if device is None:
            device = (
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )

        self.device = torch.device(
            device
        )

        agent_space = (
            env.observation_space
            .spaces[
                self.agent_ids[0]
            ]
        )

        self.observation_dim = int(
            sum(
                np.prod(
                    agent_space.spaces[
                        key
                    ].shape,
                    dtype=np.int64,
                )
                for key
                in AGENT_OBSERVATION_VECTOR_KEYS
            )
        )
        self.state_dim = int(
            np.prod(
                env.state_space.shape,
                dtype=np.int64,
            )
        )
        self.continuous_dim = 4
        self.discrete_dim = int(
            env.action_space
            .spaces[
                self.agent_ids[0]
            ]
            .spaces[
                "destination"
            ]
            .n
        )

        self.joint_action_dim = (
            self.num_agents
            * (
                self.continuous_dim
                + self.discrete_dim
            )
        )

        hidden_dims = tuple(
            CONFIG[
                "masac_hidden_dims"
            ]
        )

        self.actor = (
            HybridMASACActor(
                self.observation_dim,
                self.continuous_dim,
                self.discrete_dim,
                hidden_dims,
            ).to(
                self.device
            )
        )

        self.critic_1 = (
            CentralizedQNetwork(
                self.state_dim,
                self.joint_action_dim,
                hidden_dims,
            ).to(
                self.device
            )
        )
        self.critic_2 = (
            CentralizedQNetwork(
                self.state_dim,
                self.joint_action_dim,
                hidden_dims,
            ).to(
                self.device
            )
        )

        self.target_critic_1 = (
            copy.deepcopy(
                self.critic_1
            ).to(
                self.device
            )
        )
        self.target_critic_2 = (
            copy.deepcopy(
                self.critic_2
            ).to(
                self.device
            )
        )

        for network in (
            self.target_critic_1,
            self.target_critic_2,
        ):
            network.eval()

            for parameter in (
                network.parameters()
            ):
                parameter.requires_grad_(
                    False
                )

        self.actor_optimizer = (
            torch.optim.Adam(
                self.actor.parameters(),
                lr=float(
                    CONFIG[
                        "masac_actor_lr"
                    ]
                ),
            )
        )

        self.critic_optimizer = (
            torch.optim.Adam(
                list(
                    self.critic_1.parameters()
                )
                + list(
                    self.critic_2.parameters()
                ),
                lr=float(
                    CONFIG[
                        "masac_critic_lr"
                    ]
                ),
            )
        )

        initial_alpha_continuous = float(
            CONFIG[
                "masac_initial_alpha_continuous"
            ]
        )
        initial_alpha_discrete = float(
            CONFIG[
                "masac_initial_alpha_discrete"
            ]
        )

        if (
            initial_alpha_continuous <= 0.0
            or initial_alpha_discrete <= 0.0
        ):
            raise ValueError(
                "initial alpha values must be > 0"
            )

        self.log_alpha_continuous = (
            torch.tensor(
                np.log(
                    initial_alpha_continuous
                ),
                dtype=torch.float32,
                device=self.device,
                requires_grad=True,
            )
        )
        self.log_alpha_discrete = (
            torch.tensor(
                np.log(
                    initial_alpha_discrete
                ),
                dtype=torch.float32,
                device=self.device,
                requires_grad=True,
            )
        )

        self.alpha_continuous_optimizer = (
            torch.optim.Adam(
                [
                    self.log_alpha_continuous
                ],
                lr=float(
                    CONFIG[
                        "masac_alpha_lr"
                    ]
                ),
            )
        )
        self.alpha_discrete_optimizer = (
            torch.optim.Adam(
                [
                    self.log_alpha_discrete
                ],
                lr=float(
                    CONFIG[
                        "masac_alpha_lr"
                    ]
                ),
            )
        )

        self.gamma = float(
            CONFIG["masac_gamma"]
        )
        self.tau = float(
            CONFIG["masac_tau"]
        )
        self.gradient_clip_norm = float(
            CONFIG[
                "masac_gradient_clip_norm"
            ]
        )

        self.continuous_target_entropy = (
            float(
                CONFIG[
                    "masac_continuous_target_entropy_scale"
                ]
            )
            * self.num_agents
            * self.continuous_dim
        )

        self.discrete_target_entropy_ratio = float(
            CONFIG[
                "masac_discrete_target_entropy_ratio"
            ]
        )

        if not (
            0.0
            <= self.discrete_target_entropy_ratio
            <= 1.0
        ):
            raise ValueError(
                "discrete target entropy ratio "
                "must be within [0, 1]"
            )

        self.update_count = 0

    @property
    def alpha_continuous(self):
        return torch.exp(
            self.log_alpha_continuous
        )

    @property
    def alpha_discrete(self):
        return torch.exp(
            self.log_alpha_discrete
        )

    def make_replay_buffer(
        self,
        seed=None,
    ):
        if seed is None:
            seed = int(
                CONFIG["seed"]
            )

        return HybridReplayBuffer(
            capacity=CONFIG[
                "masac_replay_capacity"
            ],
            num_agents=(
                self.num_agents
            ),
            observation_dim=(
                self.observation_dim
            ),
            state_dim=(
                self.state_dim
            ),
            continuous_dim=(
                self.continuous_dim
            ),
            discrete_dim=(
                self.discrete_dim
            ),
            seed=seed,
        )

    def _joint_action_tensor(
        self,
        continuous_actions,
        destination_one_hot,
    ):
        if continuous_actions.ndim != 3:
            raise ValueError(
                "continuous_actions must "
                "have rank 3"
            )

        if destination_one_hot.ndim != 3:
            raise ValueError(
                "destination_one_hot must "
                "have rank 3"
            )

        combined = torch.cat(
            [
                continuous_actions,
                destination_one_hot,
            ],
            dim=-1,
        )

        return combined.reshape(
            combined.shape[0],
            -1,
        )

    def _sample_joint_policy(
        self,
        observations,
        destination_masks,
        deterministic=False,
    ):
        if observations.ndim != 3:
            raise ValueError(
                "observations must have rank 3"
            )

        batch_size = (
            observations.shape[0]
        )

        flat_observations = (
            observations.reshape(
                batch_size
                * self.num_agents,
                self.observation_dim,
            )
        )
        flat_masks = (
            destination_masks.reshape(
                batch_size
                * self.num_agents,
                self.discrete_dim,
            )
        )

        sampled = self.actor.sample(
            flat_observations,
            flat_masks,
            deterministic=deterministic,
        )

        continuous = sampled[
            "continuous"
        ].reshape(
            batch_size,
            self.num_agents,
            self.continuous_dim,
        )
        destination_one_hot = (
            sampled[
                "destination_one_hot"
            ].reshape(
                batch_size,
                self.num_agents,
                self.discrete_dim,
            )
        )
        destination_indices = (
            sampled[
                "destination_index"
            ].reshape(
                batch_size,
                self.num_agents,
            )
        )

        continuous_log_probability = (
            sampled[
                "continuous_log_probability"
            ].reshape(
                batch_size,
                self.num_agents,
                1,
            ).sum(
                dim=1
            )
        )
        destination_log_probability = (
            sampled[
                "destination_log_probability"
            ].reshape(
                batch_size,
                self.num_agents,
                1,
            ).sum(
                dim=1
            )
        )
        destination_entropy = (
            sampled[
                "destination_entropy"
            ].reshape(
                batch_size,
                self.num_agents,
                1,
            ).sum(
                dim=1
            )
        )

        return {
            "continuous": continuous,
            "destination_one_hot": (
                destination_one_hot
            ),
            "destination_indices": (
                destination_indices
            ),
            "continuous_log_probability": (
                continuous_log_probability
            ),
            "destination_log_probability": (
                destination_log_probability
            ),
            "destination_entropy": (
                destination_entropy
            ),
        }

    def select_actions(
        self,
        observations,
        deterministic=False,
    ):
        (
            observation_vectors,
            destination_masks,
        ) = observations_to_masac_arrays(
            observations,
            self.agent_ids,
        )

        observation_tensor = (
            torch.as_tensor(
                observation_vectors,
                dtype=torch.float32,
                device=self.device,
            ).unsqueeze(
                0
            )
        )
        mask_tensor = torch.as_tensor(
            destination_masks,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(
            0
        )

        with torch.no_grad():
            sampled = (
                self._sample_joint_policy(
                    observation_tensor,
                    mask_tensor,
                    deterministic=(
                        deterministic
                    ),
                )
            )

        continuous = (
            sampled["continuous"][
                0
            ]
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )
        destination_indices = (
            sampled[
                "destination_indices"
            ][
                0
            ]
            .cpu()
            .numpy()
            .astype(
                np.int64
            )
        )

        return masac_arrays_to_env_actions(
            continuous,
            destination_indices,
            self.agent_ids,
        )

    def _soft_update_targets(
        self,
    ):
        with torch.no_grad():
            for target, source in (
                (
                    self.target_critic_1,
                    self.critic_1,
                ),
                (
                    self.target_critic_2,
                    self.critic_2,
                ),
            ):
                for (
                    target_parameter,
                    source_parameter,
                ) in zip(
                    target.parameters(),
                    source.parameters(),
                ):
                    target_parameter.mul_(
                        1.0 - self.tau
                    )
                    target_parameter.add_(
                        self.tau
                        * source_parameter
                    )

    def update(
        self,
        replay_buffer,
        batch_size=None,
    ):
        if not isinstance(
            replay_buffer,
            HybridReplayBuffer,
        ):
            raise TypeError(
                "replay_buffer must be "
                "HybridReplayBuffer"
            )

        if batch_size is None:
            batch_size = int(
                CONFIG[
                    "masac_batch_size"
                ]
            )

        batch = replay_buffer.sample(
            batch_size,
            self.device,
        )

        destination_one_hot = (
            F.one_hot(
                batch[
                    "destination_indices"
                ],
                num_classes=(
                    self.discrete_dim
                ),
            ).to(
                torch.float32
            )
        )

        replay_joint_action = (
            self._joint_action_tensor(
                batch[
                    "continuous_actions"
                ],
                destination_one_hot,
            )
        )

        with torch.no_grad():
            next_policy = (
                self._sample_joint_policy(
                    batch[
                        "next_observations"
                    ],
                    batch[
                        "next_destination_masks"
                    ],
                    deterministic=False,
                )
            )

            next_joint_action = (
                self._joint_action_tensor(
                    next_policy[
                        "continuous"
                    ],
                    next_policy[
                        "destination_one_hot"
                    ],
                )
            )

            target_q = torch.minimum(
                self.target_critic_1(
                    batch[
                        "next_states"
                    ],
                    next_joint_action,
                ),
                self.target_critic_2(
                    batch[
                        "next_states"
                    ],
                    next_joint_action,
                ),
            )

            entropy_adjusted_target = (
                target_q
                - self.alpha_continuous.detach()
                * next_policy[
                    "continuous_log_probability"
                ]
                - self.alpha_discrete.detach()
                * next_policy[
                    "destination_log_probability"
                ]
            )

            # Gymnasium truncation is a time-limit event,
            # so only true termination blocks bootstrapping.
            critic_target = (
                batch["rewards"]
                + self.gamma
                * (
                    1.0
                    - batch[
                        "terminated"
                    ]
                )
                * entropy_adjusted_target
            )

        critic_1_value = (
            self.critic_1(
                batch["states"],
                replay_joint_action,
            )
        )
        critic_2_value = (
            self.critic_2(
                batch["states"],
                replay_joint_action,
            )
        )

        critic_loss = (
            F.mse_loss(
                critic_1_value,
                critic_target,
            )
            + F.mse_loss(
                critic_2_value,
                critic_target,
            )
        )

        self.critic_optimizer.zero_grad(
            set_to_none=True
        )
        critic_loss.backward()

        critic_grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                list(
                    self.critic_1.parameters()
                )
                + list(
                    self.critic_2.parameters()
                ),
                self.gradient_clip_norm,
            )
        )

        self.critic_optimizer.step()

        for critic in (
            self.critic_1,
            self.critic_2,
        ):
            for parameter in (
                critic.parameters()
            ):
                parameter.requires_grad_(
                    False
                )

        policy = self._sample_joint_policy(
            batch["observations"],
            batch[
                "destination_masks"
            ],
            deterministic=False,
        )

        policy_joint_action = (
            self._joint_action_tensor(
                policy[
                    "continuous"
                ],
                policy[
                    "destination_one_hot"
                ],
            )
        )

        policy_q = torch.minimum(
            self.critic_1(
                batch["states"],
                policy_joint_action,
            ),
            self.critic_2(
                batch["states"],
                policy_joint_action,
            ),
        )

        actor_loss = (
            self.alpha_continuous.detach()
            * policy[
                "continuous_log_probability"
            ]
            + self.alpha_discrete.detach()
            * policy[
                "destination_log_probability"
            ]
            - policy_q
        ).mean()

        self.actor_optimizer.zero_grad(
            set_to_none=True
        )
        actor_loss.backward()

        actor_grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                self.actor.parameters(),
                self.gradient_clip_norm,
            )
        )

        self.actor_optimizer.step()

        for critic in (
            self.critic_1,
            self.critic_2,
        ):
            for parameter in (
                critic.parameters()
            ):
                parameter.requires_grad_(
                    True
                )

        continuous_entropy = -policy[
            "continuous_log_probability"
        ].detach()

        continuous_target = torch.full_like(
            continuous_entropy,
            float(
                self.continuous_target_entropy
            ),
        )

        valid_counts = (
            batch[
                "destination_masks"
            ].sum(
                dim=-1
            ).clamp(
                min=1.0
            )
        )

        discrete_target = (
            self.discrete_target_entropy_ratio
            * torch.log(
                valid_counts
            ).sum(
                dim=1,
                keepdim=True,
            )
        )

        discrete_entropy = policy[
            "destination_entropy"
        ].detach()

        alpha_continuous_loss = (
            self.log_alpha_continuous
            * (
                continuous_entropy
                - continuous_target
            )
        ).mean()

        alpha_discrete_loss = (
            self.log_alpha_discrete
            * (
                discrete_entropy
                - discrete_target
            )
        ).mean()

        self.alpha_continuous_optimizer.zero_grad(
            set_to_none=True
        )
        alpha_continuous_loss.backward()
        self.alpha_continuous_optimizer.step()

        self.alpha_discrete_optimizer.zero_grad(
            set_to_none=True
        )
        alpha_discrete_loss.backward()
        self.alpha_discrete_optimizer.step()

        self._soft_update_targets()

        self.update_count += 1

        metrics = {
            "critic_loss": float(
                critic_loss.detach().cpu()
            ),
            "actor_loss": float(
                actor_loss.detach().cpu()
            ),
            "alpha_continuous_loss": float(
                alpha_continuous_loss
                .detach()
                .cpu()
            ),
            "alpha_discrete_loss": float(
                alpha_discrete_loss
                .detach()
                .cpu()
            ),
            "alpha_continuous": float(
                self.alpha_continuous
                .detach()
                .cpu()
            ),
            "alpha_discrete": float(
                self.alpha_discrete
                .detach()
                .cpu()
            ),
            "continuous_entropy": float(
                continuous_entropy
                .mean()
                .cpu()
            ),
            "discrete_entropy": float(
                discrete_entropy
                .mean()
                .cpu()
            ),
            "target_q_mean": float(
                critic_target
                .mean()
                .detach()
                .cpu()
            ),
            "critic_grad_norm": float(
                torch.as_tensor(
                    critic_grad_norm
                )
                .detach()
                .cpu()
            ),
            "actor_grad_norm": float(
                torch.as_tensor(
                    actor_grad_norm
                )
                .detach()
                .cpu()
            ),
            "update_count": int(
                self.update_count
            ),
        }

        if not all(
            np.isfinite(value)
            for key, value
            in metrics.items()
            if key
            != "update_count"
        ):
            raise FloatingPointError(
                "non-finite MASAC metric"
            )

        return metrics

    @staticmethod
    def _move_optimizer_state_to_device(
        optimizer,
        device,
    ):
        for state in optimizer.state.values():
            for key, value in list(
                state.items()
            ):
                if isinstance(
                    value,
                    torch.Tensor,
                ):
                    state[key] = value.to(
                        device
                    )

    def state_dict(self):
        return {
            "actor": (
                self.actor.state_dict()
            ),
            "critic_1": (
                self.critic_1.state_dict()
            ),
            "critic_2": (
                self.critic_2.state_dict()
            ),
            "target_critic_1": (
                self.target_critic_1
                .state_dict()
            ),
            "target_critic_2": (
                self.target_critic_2
                .state_dict()
            ),
            "actor_optimizer": (
                self.actor_optimizer
                .state_dict()
            ),
            "critic_optimizer": (
                self.critic_optimizer
                .state_dict()
            ),
            "log_alpha_continuous": (
                self.log_alpha_continuous
                .detach()
                .cpu()
            ),
            "log_alpha_discrete": (
                self.log_alpha_discrete
                .detach()
                .cpu()
            ),
            "alpha_continuous_optimizer": (
                self.alpha_continuous_optimizer
                .state_dict()
            ),
            "alpha_discrete_optimizer": (
                self.alpha_discrete_optimizer
                .state_dict()
            ),
            "update_count": int(
                self.update_count
            ),
        }

    def load_state_dict(
        self,
        state,
    ):
        self.actor.load_state_dict(
            state["actor"]
        )
        self.critic_1.load_state_dict(
            state["critic_1"]
        )
        self.critic_2.load_state_dict(
            state["critic_2"]
        )
        self.target_critic_1.load_state_dict(
            state[
                "target_critic_1"
            ]
        )
        self.target_critic_2.load_state_dict(
            state[
                "target_critic_2"
            ]
        )

        self.actor_optimizer.load_state_dict(
            state[
                "actor_optimizer"
            ]
        )
        self.critic_optimizer.load_state_dict(
            state[
                "critic_optimizer"
            ]
        )

        with torch.no_grad():
            self.log_alpha_continuous.copy_(
                state[
                    "log_alpha_continuous"
                ].to(
                    self.device
                )
            )
            self.log_alpha_discrete.copy_(
                state[
                    "log_alpha_discrete"
                ].to(
                    self.device
                )
            )

        self.alpha_continuous_optimizer.load_state_dict(
            state[
                "alpha_continuous_optimizer"
            ]
        )
        self.alpha_discrete_optimizer.load_state_dict(
            state[
                "alpha_discrete_optimizer"
            ]
        )

        for optimizer in (
            self.actor_optimizer,
            self.critic_optimizer,
            self.alpha_continuous_optimizer,
            self.alpha_discrete_optimizer,
        ):
            self._move_optimizer_state_to_device(
                optimizer,
                self.device,
            )

        self.update_count = int(
            state.get(
                "update_count",
                0,
            )
        )


In [ ]:
def add_environment_transition_to_replay(
    replay_buffer,
    observations,
    state,
    actions,
    reward,
    next_observations,
    next_state,
    terminated,
    truncated,
    agent_ids,
):
    (
        observation_vectors,
        destination_masks,
    ) = observations_to_masac_arrays(
        observations,
        agent_ids,
    )

    (
        next_observation_vectors,
        next_destination_masks,
    ) = observations_to_masac_arrays(
        next_observations,
        agent_ids,
    )

    (
        continuous_actions,
        destination_indices,
    ) = env_actions_to_masac_arrays(
        actions,
        agent_ids,
    )

    replay_buffer.add(
        observations=(
            observation_vectors
        ),
        state=state,
        continuous_actions=(
            continuous_actions
        ),
        destination_indices=(
            destination_indices
        ),
        destination_masks=(
            destination_masks
        ),
        reward=reward,
        next_observations=(
            next_observation_vectors
        ),
        next_state=next_state,
        next_destination_masks=(
            next_destination_masks
        ),
        terminated=terminated,
        truncated=truncated,
    )


def train_masac(
    env,
    total_steps,
    seed=None,
    trainer=None,
    replay_buffer=None,
):
    if not isinstance(
        env,
        UAVSearchEnv,
    ):
        raise TypeError(
            "env must be UAVSearchEnv"
        )

    total_steps = int(
        total_steps
    )

    if total_steps < 1:
        raise ValueError(
            "total_steps must be >= 1"
        )

    if seed is None:
        seed = int(
            CONFIG["seed"]
        )

    seed = int(
        seed
    )

    set_seed(
        seed
    )

    if trainer is None:
        trainer = HybridMASAC(
            env
        )

    if replay_buffer is None:
        replay_buffer = (
            trainer.make_replay_buffer(
                seed=seed,
            )
        )

    rng = np.random.default_rng(
        seed
    )

    observations, info = env.reset(
        seed=seed
    )
    state = info[
        "state"
    ]

    episode_return = 0.0
    episode_length = 0
    episode_index = 0
    completed_episodes = []
    latest_update_metrics = None

    learning_starts = int(
        CONFIG[
            "masac_learning_starts"
        ]
    )
    updates_per_step = int(
        CONFIG[
            "masac_updates_per_step"
        ]
    )
    batch_size = int(
        CONFIG[
            "masac_batch_size"
        ]
    )

    for global_step in range(
        1,
        total_steps + 1,
    ):
        if global_step <= learning_starts:
            actions = (
                sample_valid_random_actions(
                    observations,
                    env.agent_ids,
                    rng,
                )
            )
        else:
            actions = (
                trainer.select_actions(
                    observations,
                    deterministic=False,
                )
            )

        (
            next_observations,
            reward,
            terminated,
            truncated,
            next_info,
        ) = env.step(
            actions
        )

        next_state = next_info[
            "state"
        ]

        add_environment_transition_to_replay(
            replay_buffer,
            observations,
            state,
            actions,
            reward,
            next_observations,
            next_state,
            terminated,
            truncated,
            env.agent_ids,
        )

        episode_return += float(
            reward
        )
        episode_length += 1

        if (
            global_step > learning_starts
            and len(
                replay_buffer
            )
            >= batch_size
        ):
            for _ in range(
                updates_per_step
            ):
                latest_update_metrics = (
                    trainer.update(
                        replay_buffer,
                        batch_size=(
                            batch_size
                        ),
                    )
                )

        observations = (
            next_observations
        )
        state = next_state

        if terminated or truncated:
            completed_episodes.append(
                {
                    "episode": (
                        episode_index
                    ),
                    "return": float(
                        episode_return
                    ),
                    "length": int(
                        episode_length
                    ),
                    "terminated": bool(
                        terminated
                    ),
                    "truncated": bool(
                        truncated
                    ),
                    "success": bool(
                        next_info[
                            "success"
                        ]
                    ),
                }
            )

            episode_index += 1
            episode_return = 0.0
            episode_length = 0

            if global_step < total_steps:
                observations, info = (
                    env.reset(
                        seed=(
                            seed
                            + episode_index
                        )
                    )
                )
                state = info[
                    "state"
                ]

    return {
        "trainer": trainer,
        "replay_buffer": replay_buffer,
        "completed_episodes": (
            completed_episodes
        ),
        "latest_update_metrics": (
            latest_update_metrics
        ),
        "total_steps": total_steps,
    }


def evaluate_masac(
    env,
    trainer,
    episodes=3,
    seed=None,
):
    if seed is None:
        seed = int(
            CONFIG["seed"]
        ) + 10_000

    episodes = int(
        episodes
    )

    if episodes < 1:
        raise ValueError(
            "episodes must be >= 1"
        )

    results = []

    for episode_index in range(
        episodes
    ):
        observations, _ = env.reset(
            seed=(
                int(seed)
                + episode_index
            )
        )

        episode_return = 0.0
        diagnostics = (
            new_episode_diagnostics()
        )

        while True:
            actions = trainer.select_actions(
                observations,
                deterministic=True,
            )

            (
                observations,
                reward,
                terminated,
                truncated,
                info,
            ) = env.step(
                actions
            )

            episode_return += float(
                reward
            )

            update_episode_diagnostics(
                diagnostics,
                info,
            )

            if (
                terminated
                or truncated
            ):
                episode_metrics = (
                    finalize_episode_diagnostics(
                        diagnostics,
                        env,
                        episode_return=(
                            episode_return
                        ),
                        episode_length=(
                            info["step"]
                        ),
                        success=(
                            info["success"]
                        ),
                    )
                )

                results.append(
                    {
                        "episode": (
                            episode_index
                        ),
                        **episode_metrics,
                        "terminated": bool(
                            terminated
                        ),
                        "truncated": bool(
                            truncated
                        ),
                    }
                )
                break

    return results


In [ ]:
def _checkpoint_config_snapshot():
    snapshot = {}

    for key, value in CONFIG.items():
        if isinstance(
            value,
            tuple,
        ):
            snapshot[key] = list(
                value
            )
        elif isinstance(
            value,
            np.ndarray,
        ):
            snapshot[key] = (
                value.tolist()
            )
        elif isinstance(
            value,
            np.generic,
        ):
            snapshot[key] = (
                value.item()
            )
        else:
            snapshot[key] = value

    return snapshot


def save_masac_checkpoint(
    path,
    trainer,
    replay_buffer=None,
    training_state=None,
    include_replay=None,
):
    if not isinstance(
        trainer,
        HybridMASAC,
    ):
        raise TypeError(
            "trainer must be HybridMASAC"
        )

    if include_replay is None:
        include_replay = bool(
            CONFIG[
                "training_checkpoint_include_replay"
            ]
        )

    include_replay = bool(
        include_replay
    )

    if (
        include_replay
        and replay_buffer is None
    ):
        raise ValueError(
            "replay_buffer is required "
            "when include_replay=True"
        )

    if (
        replay_buffer is not None
        and not isinstance(
            replay_buffer,
            HybridReplayBuffer,
        )
    ):
        raise TypeError(
            "replay_buffer must be "
            "HybridReplayBuffer"
        )

    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if training_state is None:
        training_state = {}

    if not isinstance(
        training_state,
        dict,
    ):
        raise TypeError(
            "training_state must be a dict"
        )

    payload = {
        "format_version": 1,
        "trainer_state": (
            trainer.state_dict()
        ),
        "training_state": (
            copy.deepcopy(
                training_state
            )
        ),
        "config_snapshot": (
            _checkpoint_config_snapshot()
        ),
        "replay_state": (
            replay_buffer.state_dict()
            if include_replay
            else None
        ),
        "torch_rng_state": (
            torch.get_rng_state()
        ),
        "cuda_rng_states": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else []
        ),
    }

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    torch.save(
        payload,
        temporary_path,
    )

    temporary_path.replace(
        path
    )

    return path


def load_masac_checkpoint(
    path,
    env,
    device=None,
    load_replay=True,
    restore_torch_rng=True,
):
    if not isinstance(
        env,
        UAVSearchEnv,
    ):
        raise TypeError(
            "env must be UAVSearchEnv"
        )

    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            str(path)
        )

    # weights_only=True is sufficient because this project checkpoint
    # contains tensors and basic Python containers, not serialized model
    # objects.
    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=True,
    )

    if int(
        payload.get(
            "format_version",
            -1,
        )
    ) != 1:
        raise ValueError(
            "unsupported checkpoint "
            "format_version"
        )

    trainer = HybridMASAC(
        env,
        device=device,
    )
    trainer.load_state_dict(
        payload[
            "trainer_state"
        ]
    )

    replay_buffer = None

    replay_state = payload.get(
        "replay_state"
    )

    if (
        load_replay
        and replay_state is not None
    ):
        training_state = payload.get(
            "training_state",
            {},
        )
        replay_seed = int(
            training_state.get(
                "seed",
                CONFIG["seed"],
            )
        )

        replay_buffer = (
            trainer.make_replay_buffer(
                seed=replay_seed,
            )
        )
        replay_buffer.load_state_dict(
            replay_state
        )

    if restore_torch_rng:
        torch.set_rng_state(
            payload[
                "torch_rng_state"
            ]
        )

        cuda_states = payload.get(
            "cuda_rng_states",
            [],
        )

        if (
            torch.cuda.is_available()
            and cuda_states
        ):
            torch.cuda.set_rng_state_all(
                cuda_states
            )

    return {
        "trainer": trainer,
        "replay_buffer": replay_buffer,
        "training_state": (
            copy.deepcopy(
                payload.get(
                    "training_state",
                    {},
                )
            )
        ),
        "config_snapshot": (
            copy.deepcopy(
                payload.get(
                    "config_snapshot",
                    {},
                )
            )
        ),
        "path": path,
    }


In [ ]:
def git_experiment_metadata(
    repo_path=".",
):
    repo_path = Path(
        repo_path
    ).resolve()

    def git_output(
        *args,
    ):
        result = subprocess.run(
            [
                "git",
                "-C",
                str(repo_path),
                *args,
            ],
            capture_output=True,
            text=True,
            check=False,
        )

        if result.returncode != 0:
            return None

        value = result.stdout.strip()

        return (
            value
            if value
            else None
        )

    return {
        "git_commit": git_output(
            "rev-parse",
            "HEAD",
        ),
        "git_branch": git_output(
            "branch",
            "--show-current",
        ),
        "git_remote": git_output(
            "remote",
            "get-url",
            "origin",
        ),
    }


def new_episode_diagnostics():
    return {
        "information_gain_bits": 0.0,
        "false_confirmations": 0,
        "expired_reports": 0,
        "blocked_motion": 0,
        "boundary_clips": 0,
        "communication_energy_j": 0.0,
        "total_energy_j": 0.0,
        "network_snapshot": {},
    }


def update_episode_diagnostics(
    diagnostics,
    info,
):
    diagnostics[
        "information_gain_bits"
    ] += float(
        info[
            "information_gain_bits"
        ]
    )
    diagnostics[
        "false_confirmations"
    ] += int(
        info[
            "false_confirmation_count"
        ]
    )
    diagnostics[
        "expired_reports"
    ] += len(
        info[
            "expired_target_ids"
        ]
    )

    motion = info[
        "motion"
    ]

    diagnostics[
        "blocked_motion"
    ] += int(
        np.count_nonzero(
            motion[
                "blocked"
            ]
        )
    )
    diagnostics[
        "boundary_clips"
    ] += int(
        np.count_nonzero(
            motion[
                "boundary_clipped"
            ]
        )
    )

    energy = info[
        "energy"
    ]

    diagnostics[
        "communication_energy_j"
    ] += float(
        np.sum(
            energy[
                "communication_j"
            ]
        )
    )
    diagnostics[
        "total_energy_j"
    ] += float(
        np.sum(
            energy[
                "total_j"
            ]
        )
    )

    diagnostics[
        "network_snapshot"
    ] = copy.deepcopy(
        info[
            "network_metrics"
        ]
    )


def extract_network_kpis(
    snapshot,
):
    metrics = {}

    nested = snapshot.get(
        "network_metrics",
        {},
    )

    key_map = {
        "pdr_percent": (
            "network_pdr_percent"
        ),
        "e2e_delay_ms": (
            "network_e2e_delay_ms"
        ),
        "throughput_kbps": (
            "network_throughput_kbps"
        ),
        "average_hops": (
            "network_average_hops"
        ),
        "phy_success_percent": (
            "network_phy_success_percent"
        ),
        "collisions": (
            "network_collisions"
        ),
    }

    for (
        source_key,
        metric_key,
    ) in key_map.items():
        value = nested.get(
            source_key
        )

        if isinstance(
            value,
            (bool, int, float, np.generic),
        ):
            metrics[
                metric_key
            ] = float(value)

    injected = snapshot.get(
        "payload_bytes_injected"
    )
    delivered = snapshot.get(
        "payload_bytes_delivered"
    )

    if (
        isinstance(
            injected,
            (int, float, np.generic),
        )
        and isinstance(
            delivered,
            (int, float, np.generic),
        )
        and float(injected) > 0.0
    ):
        metrics[
            "network_payload_delivery_ratio"
        ] = float(
            delivered
        ) / float(
            injected
        )

    return metrics


def finalize_episode_diagnostics(
    diagnostics,
    env,
    episode_return,
    episode_length,
    success,
):
    delivered_targets = len(
        env.gcs_received_target_ids
    )
    confirmed_targets = sum(
        target.confirmed
        for target
        in env.targets
    )

    target_count = max(
        1,
        int(
            env.num_targets
        ),
    )
    possible_agent_steps = max(
        1,
        int(
            episode_length
        )
        * int(
            env.num_uavs
        ),
    )

    metrics = {
        "return": float(
            episode_return
        ),
        "length": int(
            episode_length
        ),
        "success": float(
            bool(success)
        ),
        "delivery_rate": float(
            delivered_targets
            / target_count
        ),
        "confirmation_rate": float(
            confirmed_targets
            / target_count
        ),
        "false_confirmations": int(
            diagnostics[
                "false_confirmations"
            ]
        ),
        "expired_reports": int(
            diagnostics[
                "expired_reports"
            ]
        ),
        "blocked_motion_rate": float(
            diagnostics[
                "blocked_motion"
            ]
            / possible_agent_steps
        ),
        "boundary_clip_rate": float(
            diagnostics[
                "boundary_clips"
            ]
            / possible_agent_steps
        ),
        "total_energy_j": float(
            diagnostics[
                "total_energy_j"
            ]
        ),
        "communication_energy_j": float(
            diagnostics[
                "communication_energy_j"
            ]
        ),
        "information_gain_bits": float(
            diagnostics[
                "information_gain_bits"
            ]
        ),
    }

    metrics.update(
        extract_network_kpis(
            diagnostics[
                "network_snapshot"
            ]
        )
    )

    return metrics


def filter_training_metrics_for_wandb(
    algorithm,
    metrics,
):
    normalized = str(
        algorithm
    ).strip().lower()

    if normalized == "masac":
        keep = {
            "critic_loss",
            "actor_loss",
            "alpha_continuous",
            "alpha_discrete",
            "continuous_entropy",
            "discrete_entropy",
        }
    elif normalized == "matd3":
        keep = {
            "critic_loss",
            "actor_loss",
        }
    else:
        raise ValueError(
            "algorithm must be "
            "'masac' or 'matd3'"
        )

    return {
        key: value
        for key, value
        in metrics.items()
        if key in keep
        and value is not None
    }


In [ ]:
class MASACExperimentLogger:
    def __init__(
        self,
        run_dir,
        run_name,
        enable_csv=None,
        enable_tensorboard=None,
        enable_wandb=None,
        wandb_entity=None,
        wandb_project=None,
        wandb_mode=None,
        algorithm=None,
        seed=None,
        backend_name=None,
        config_snapshot=None,
    ):
        self.run_dir = Path(
            run_dir
        )
        self.run_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        self.run_name = str(
            run_name
        )
        self.algorithm = (
            None
            if algorithm is None
            else str(
                algorithm
            ).lower()
        )
        self.seed = (
            None
            if seed is None
            else int(seed)
        )
        self.backend_name = (
            None
            if backend_name is None
            else str(
                backend_name
            )
        )

        if enable_csv is None:
            enable_csv = CONFIG[
                "training_enable_csv"
            ]

        if enable_tensorboard is None:
            enable_tensorboard = CONFIG[
                "training_enable_tensorboard"
            ]

        if enable_wandb is None:
            enable_wandb = CONFIG[
                "training_enable_wandb"
            ]

        if wandb_entity is None:
            wandb_entity = CONFIG.get(
                "training_wandb_entity"
            )

        if wandb_project is None:
            wandb_project = CONFIG[
                "training_wandb_project"
            ]

        if wandb_mode is None:
            wandb_mode = CONFIG[
                "training_wandb_mode"
            ]

        self.enable_csv = bool(
            enable_csv
        )
        self.enable_tensorboard = bool(
            enable_tensorboard
        )
        self.enable_wandb = bool(
            enable_wandb
        )

        self.csv_path = (
            self.run_dir
            / "metrics.csv"
        )
        self._csv_file = None
        self._csv_writer = None
        self._tensorboard_writer = None
        self._wandb_run = None

        self.tensorboard_error = None
        self.wandb_error = None
        self.wandb_run_id = None
        self.wandb_run_url = None

        if config_snapshot is None:
            config_snapshot = (
                _checkpoint_config_snapshot()
            )

        self.config_snapshot = (
            copy.deepcopy(
                config_snapshot
            )
        )

        git_metadata = (
            git_experiment_metadata()
        )

        self.config_snapshot.update(
            {
                "monitor/algorithm": (
                    self.algorithm
                ),
                "monitor/seed": self.seed,
                "monitor/backend": (
                    self.backend_name
                ),
                **{
                    (
                        f"monitor/{key}"
                    ): value
                    for key, value
                    in git_metadata.items()
                },
            }
        )

        if self.enable_csv:
            file_exists = (
                self.csv_path.is_file()
                and self.csv_path.stat().st_size
                > 0
            )

            self._csv_file = open(  # noqa: SIM115 - kept open for the run
                self.csv_path,
                "a",
                newline="",
                encoding="utf-8",
            )
            self._csv_writer = (
                csv.writer(
                    self._csv_file
                )
            )

            if not file_exists:
                self._csv_writer.writerow(
                    [
                        "step",
                        "episode",
                        "metric",
                        "value",
                    ]
                )
                self._csv_file.flush()

        if self.enable_tensorboard:
            try:
                from torch.utils.tensorboard import (
                    SummaryWriter,
                )

                self._tensorboard_writer = (
                    SummaryWriter(
                        log_dir=str(
                            self.run_dir
                            / "tensorboard"
                        )
                    )
                )
            except Exception as exc:  # noqa: BLE001 - optional logger fallback
                self.tensorboard_error = (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                )

        if self.enable_wandb:
            try:
                wandb = (
                    importlib.import_module(
                        "wandb"
                    )
                )

                tags = [
                    value
                    for value in (
                        self.algorithm,
                        self.backend_name,
                        (
                            None
                            if self.seed is None
                            else f"seed-{self.seed}"
                        ),
                    )
                    if value
                ]

                group = "-".join(
                    value
                    for value in (
                        self.algorithm,
                        self.backend_name,
                    )
                    if value
                ) or None

                self._wandb_run = (
                    wandb.init(
                        entity=(
                            wandb_entity
                        ),
                        project=str(
                            wandb_project
                        ),
                        name=self.run_name,
                        group=group,
                        job_type="training",
                        tags=tags,
                        config=(
                            self.config_snapshot
                        ),
                        mode=str(
                            wandb_mode
                        ),
                        dir=str(
                            self.run_dir
                        ),
                        reinit="finish_previous",
                    )
                )

                self.wandb_run_id = (
                    self._wandb_run.id
                )
                self.wandb_run_url = (
                    self._wandb_run.url
                )

                self._configure_wandb_metrics()
            except Exception as exc:  # noqa: BLE001 - optional logger fallback
                self.wandb_error = (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                )

    def _configure_wandb_metrics(
        self,
    ):
        if self._wandb_run is None:
            return

        self._wandb_run.define_metric(
            "global_step"
        )

        for pattern in (
            "train/*",
            "episode/*",
            "evaluation/*",
        ):
            self._wandb_run.define_metric(
                pattern,
                step_metric="global_step",
            )

        for metric in (
            "evaluation/success_rate",
            "evaluation/delivery_rate",
            "evaluation/confirmation_rate",
            "evaluation/return",
            "evaluation/network_pdr_percent",
            "evaluation/network_throughput_kbps",
            "evaluation/network_phy_success_percent",
        ):
            self._wandb_run.define_metric(
                metric,
                summary="max",
            )

        for metric in (
            "evaluation/total_energy_j",
            "evaluation/communication_energy_j",
            "evaluation/false_confirmations",
            "evaluation/expired_reports",
            "evaluation/blocked_motion_rate",
            "evaluation/boundary_clip_rate",
            "evaluation/network_e2e_delay_ms",
            "evaluation/network_collisions",
        ):
            self._wandb_run.define_metric(
                metric,
                summary="min",
            )

        self._wandb_run.define_metric(
            "train/critic_loss",
            summary="min",
        )

    @staticmethod
    def _scalar_metrics(
        metrics,
    ):
        if not isinstance(
            metrics,
            dict,
        ):
            raise TypeError(
                "metrics must be a dict"
            )

        scalar_metrics = {}

        for key, value in (
            metrics.items()
        ):
            if isinstance(
                value,
                torch.Tensor,
            ):
                if value.numel() != 1:
                    continue

                value = float(
                    value
                    .detach()
                    .cpu()
                    .item()
                )
            elif isinstance(
                value,
                np.generic,
            ):
                value = value.item()

            if isinstance(
                value,
                (bool, int, float),
            ):
                value = float(value)

                if np.isfinite(
                    value
                ):
                    scalar_metrics[
                        str(key)
                    ] = value

        return scalar_metrics

    def log_metrics(
        self,
        prefix,
        metrics,
        step,
        episode=None,
    ):
        prefix = str(
            prefix
        ).strip(
            "/"
        )
        step = int(
            step
        )

        scalar_metrics = (
            self._scalar_metrics(
                metrics
            )
        )

        tagged = {
            (
                f"{prefix}/{key}"
                if prefix
                else key
            ): value
            for key, value
            in scalar_metrics.items()
        }

        if (
            self._csv_writer
            is not None
        ):
            for key, value in (
                tagged.items()
            ):
                self._csv_writer.writerow(
                    [
                        step,
                        (
                            ""
                            if episode is None
                            else int(
                                episode
                            )
                        ),
                        key,
                        value,
                    ]
                )

            self._csv_file.flush()

        if (
            self._tensorboard_writer
            is not None
        ):
            for key, value in (
                tagged.items()
            ):
                self._tensorboard_writer.add_scalar(
                    key,
                    value,
                    step,
                )

        if self._wandb_run is not None:
            self._wandb_run.log(
                {
                    "global_step": step,
                    **tagged,
                }
            )

        return tagged

    def update_summary(
        self,
        metrics,
        prefix="summary",
    ):
        scalar_metrics = (
            self._scalar_metrics(
                metrics
            )
        )

        if self._wandb_run is not None:
            for key, value in (
                scalar_metrics.items()
            ):
                self._wandb_run.summary[
                    f"{prefix}/{key}"
                ] = value

        return scalar_metrics

    def log_model_artifact(
        self,
        path,
        aliases,
        metadata=None,
    ):
        if self._wandb_run is None:
            return None

        path = Path(path)

        if not path.is_file():
            raise FileNotFoundError(
                str(path)
            )

        wandb = importlib.import_module(
            "wandb"
        )

        artifact_name = (
            f"{self.run_name}-"
            f"{self.algorithm or 'marl'}-model"
        )

        artifact = wandb.Artifact(
            name=artifact_name,
            type="model",
            metadata=(
                {}
                if metadata is None
                else copy.deepcopy(
                    metadata
                )
            ),
        )
        artifact.add_file(
            str(path),
            name=path.name,
        )

        self._wandb_run.log_artifact(
            artifact,
            aliases=list(
                aliases
            ),
        )

        return artifact_name

    def status(self):
        return {
            "csv": bool(
                self._csv_writer
                is not None
            ),
            "tensorboard": bool(
                self._tensorboard_writer
                is not None
            ),
            "wandb": bool(
                self._wandb_run
                is not None
            ),
            "wandb_run_id": (
                self.wandb_run_id
            ),
            "wandb_run_url": (
                self.wandb_run_url
            ),
            "algorithm": self.algorithm,
            "seed": self.seed,
            "backend": (
                self.backend_name
            ),
            "tensorboard_error": (
                self.tensorboard_error
            ),
            "wandb_error": (
                self.wandb_error
            ),
        }

    def close(self):
        if (
            self._tensorboard_writer
            is not None
        ):
            self._tensorboard_writer.flush()
            self._tensorboard_writer.close()
            self._tensorboard_writer = None

        if self._wandb_run is not None:
            self._wandb_run.finish()
            self._wandb_run = None

        if self._csv_file is not None:
            self._csv_file.flush()
            self._csv_file.close()
            self._csv_file = None
            self._csv_writer = None


def summarize_evaluation_results(
    results,
):
    if not results:
        raise ValueError(
            "evaluation results must "
            "not be empty"
        )

    summary = {
        "episodes": len(
            results
        ),
        "success_rate": float(
            np.mean(
                np.asarray(
                    [
                        item[
                            "success"
                        ]
                        for item
                        in results
                    ],
                    dtype=np.float64,
                )
            )
        ),
        "return": float(
            np.mean(
                np.asarray(
                    [
                        item[
                            "return"
                        ]
                        for item
                        in results
                    ],
                    dtype=np.float64,
                )
            )
        ),
        "return_std": float(
            np.std(
                np.asarray(
                    [
                        item[
                            "return"
                        ]
                        for item
                        in results
                    ],
                    dtype=np.float64,
                )
            )
        ),
        "length": float(
            np.mean(
                np.asarray(
                    [
                        item[
                            "length"
                        ]
                        for item
                        in results
                    ],
                    dtype=np.float64,
                )
            )
        ),
    }

    keys_to_average = (
        "delivery_rate",
        "confirmation_rate",
        "false_confirmations",
        "expired_reports",
        "blocked_motion_rate",
        "boundary_clip_rate",
        "total_energy_j",
        "communication_energy_j",
        "information_gain_bits",
        "network_pdr_percent",
        "network_e2e_delay_ms",
        "network_throughput_kbps",
        "network_average_hops",
        "network_phy_success_percent",
        "network_collisions",
        "network_payload_delivery_ratio",
    )

    for key in keys_to_average:
        values = [
            item[key]
            for item in results
            if key in item
        ]

        if len(values) != len(
            results
        ):
            continue

        summary[key] = float(
            np.mean(
                np.asarray(
                    values,
                    dtype=np.float64,
                )
            )
        )

    return summary


def evaluation_is_better(
    candidate,
    best,
):
    if best is None:
        return True

    candidate_key = (
        float(
            candidate[
                "success_rate"
            ]
        ),
        float(
            candidate.get(
                "delivery_rate",
                0.0,
            )
        ),
        float(
            candidate.get(
                "confirmation_rate",
                0.0,
            )
        ),
        float(
            candidate[
                "return"
            ]
        ),
    )
    best_key = (
        float(
            best[
                "success_rate"
            ]
        ),
        float(
            best.get(
                "delivery_rate",
                0.0,
            )
        ),
        float(
            best.get(
                "confirmation_rate",
                0.0,
            )
        ),
        float(
            best[
                "return"
            ]
        ),
    )

    return bool(
        candidate_key > best_key
    )



In [ ]:
def train_masac_experiment(
    env,
    total_steps,
    seed=None,
    run_dir=None,
    run_name=None,
    resume_checkpoint=None,
    device=None,
    enable_csv=None,
    enable_tensorboard=None,
    enable_wandb=None,
):
    if not isinstance(
        env,
        UAVSearchEnv,
    ):
        raise TypeError(
            "env must be UAVSearchEnv"
        )

    total_steps = int(
        total_steps
    )

    if total_steps < 1:
        raise ValueError(
            "total_steps must be >= 1"
        )

    if seed is None:
        seed = int(
            CONFIG["seed"]
        )

    seed = int(
        seed
    )

    if run_name is None:
        run_name = (
            make_experiment_run_name(
                "masac",
                seed,
            )
        )

    if run_dir is None:
        if resume_checkpoint is not None:
            run_dir = (
                Path(
                    resume_checkpoint
                )
                .resolve()
                .parent
                .parent
            )
        else:
            run_dir = (
                Path(
                    CONFIG[
                        "training_output_dir"
                    ]
                )
                / run_name
            )

    run_dir = Path(
        run_dir
    )
    checkpoint_dir = (
        run_dir / "checkpoints"
    )
    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    set_seed(
        seed
    )

    if resume_checkpoint is None:
        trainer = HybridMASAC(
            env,
            device=device,
        )
        replay_buffer = (
            trainer.make_replay_buffer(
                seed=seed,
            )
        )
        training_state = {
            "seed": seed,
            "global_step": 0,
            "episode_index": 0,
            "best_evaluation": None,
            "last_eval_step": 0,
            "last_checkpoint_step": 0,
            "training_rng_state_json": None,
            "resume_safe": True,
        }
    else:
        restored = (
            load_masac_checkpoint(
                resume_checkpoint,
                env,
                device=device,
                load_replay=True,
                restore_torch_rng=True,
            )
        )
        trainer = restored[
            "trainer"
        ]
        replay_buffer = restored[
            "replay_buffer"
        ]
        training_state = restored[
            "training_state"
        ]

        if replay_buffer is None:
            raise ValueError(
                "resume checkpoint does not "
                "contain replay state; set "
                "training_checkpoint_include_replay=True "
                "when creating resumable checkpoints"
            )

        if not bool(
            training_state.get(
                "resume_safe",
                False,
            )
        ):
            raise ValueError(
                "checkpoint is not marked "
                "resume_safe"
            )

        saved_seed = int(
            training_state.get(
                "seed",
                seed,
            )
        )

        if saved_seed != seed:
            raise ValueError(
                "resume seed does not match "
                "checkpoint seed"
            )

    global_step = int(
        training_state.get(
            "global_step",
            0,
        )
    )
    episode_index = int(
        training_state.get(
            "episode_index",
            0,
        )
    )
    best_evaluation = copy.deepcopy(
        training_state.get(
            "best_evaluation"
        )
    )
    last_eval_step = int(
        training_state.get(
            "last_eval_step",
            0,
        )
    )
    last_checkpoint_step = int(
        training_state.get(
            "last_checkpoint_step",
            0,
        )
    )

    if global_step > total_steps:
        raise ValueError(
            "total_steps is smaller than "
            "checkpoint global_step"
        )

    rng = np.random.default_rng(
        seed
    )

    saved_rng_state = (
        training_state.get(
            "training_rng_state_json"
        )
    )

    if saved_rng_state:
        rng.bit_generator.state = (
            json.loads(
                saved_rng_state
            )
        )

    logger = MASACExperimentLogger(
        run_dir=run_dir,
        run_name=run_name,
        enable_csv=enable_csv,
        enable_tensorboard=(
            enable_tensorboard
        ),
        enable_wandb=enable_wandb,
        wandb_entity=CONFIG.get(
            "training_wandb_entity"
        ),
        wandb_project=CONFIG[
            "training_wandb_project"
        ],
        algorithm="masac",
        seed=seed,
        backend_name=(
            env.backend_name
        ),
        config_snapshot=(
            _checkpoint_config_snapshot()
        ),
    )

    eval_env = UAVSearchEnv(
        backend_name=env.backend_name,
        network_backend_kwargs=(
            env.network_backend_kwargs
        ),
    )

    observations, info = env.reset(
        seed=(
            seed + episode_index
        )
    )
    state = info[
        "state"
    ]

    episode_return = 0.0
    episode_length = 0
    episode_diagnostics = (
        new_episode_diagnostics()
    )
    completed_episodes = []
    latest_update_metrics = None
    latest_evaluation = None

    learning_starts = int(
        CONFIG[
            "masac_learning_starts"
        ]
    )
    batch_size = int(
        CONFIG[
            "masac_batch_size"
        ]
    )
    updates_per_step = int(
        CONFIG[
            "masac_updates_per_step"
        ]
    )
    log_interval = max(
        1,
        int(
            CONFIG[
                "training_log_interval_steps"
            ]
        ),
    )
    eval_interval = max(
        1,
        int(
            CONFIG[
                "training_eval_interval_steps"
            ]
        ),
    )
    checkpoint_interval = max(
        1,
        int(
            CONFIG[
                "training_checkpoint_interval_steps"
            ]
        ),
    )
    eval_episodes = max(
        1,
        int(
            CONFIG[
                "training_eval_episodes"
            ]
        ),
    )

    try:
        while global_step < total_steps:
            next_global_step = (
                global_step + 1
            )

            if (
                next_global_step
                <= learning_starts
            ):
                actions = (
                    sample_valid_random_actions(
                        observations,
                        env.agent_ids,
                        rng,
                    )
                )
            else:
                actions = (
                    trainer.select_actions(
                        observations,
                        deterministic=False,
                    )
                )

            (
                next_observations,
                reward,
                terminated,
                truncated,
                next_info,
            ) = env.step(
                actions
            )

            next_state = next_info[
                "state"
            ]

            add_environment_transition_to_replay(
                replay_buffer,
                observations,
                state,
                actions,
                reward,
                next_observations,
                next_state,
                terminated,
                truncated,
                env.agent_ids,
            )

            global_step = (
                next_global_step
            )
            episode_return += float(
                reward
            )
            episode_length += 1

            update_episode_diagnostics(
                episode_diagnostics,
                next_info,
            )

            if (
                global_step
                > learning_starts
                and len(
                    replay_buffer
                )
                >= batch_size
            ):
                for _ in range(
                    updates_per_step
                ):
                    latest_update_metrics = (
                        trainer.update(
                            replay_buffer,
                            batch_size=(
                                batch_size
                            ),
                        )
                    )

            if (
                latest_update_metrics
                is not None
                and global_step
                % log_interval
                == 0
            ):
                logger.log_metrics(
                    "train",
                    filter_training_metrics_for_wandb(
                        "masac",
                        latest_update_metrics,
                    ),
                    step=global_step,
                    episode=episode_index,
                )

            observations = (
                next_observations
            )
            state = next_state

            if not (
                terminated
                or truncated
            ):
                continue

            episode_metrics = (
                finalize_episode_diagnostics(
                    episode_diagnostics,
                    env,
                    episode_return=(
                        episode_return
                    ),
                    episode_length=(
                        episode_length
                    ),
                    success=(
                        next_info[
                            "success"
                        ]
                    ),
                )
            )

            logger.log_metrics(
                "episode",
                episode_metrics,
                step=global_step,
                episode=episode_index,
            )

            completed_episodes.append(
                {
                    "episode": (
                        episode_index
                    ),
                    **episode_metrics,
                    "terminated": bool(
                        terminated
                    ),
                    "truncated": bool(
                        truncated
                    ),
                }
            )

            episode_index += 1

            should_evaluate = (
                global_step
                - last_eval_step
                >= eval_interval
                or global_step
                >= total_steps
            )

            if should_evaluate:
                evaluation_results = (
                    evaluate_masac(
                        eval_env,
                        trainer,
                        episodes=(
                            eval_episodes
                        ),
                        seed=(
                            seed
                            + int(
                                CONFIG[
                                    "training_eval_seed_offset"
                                ]
                            )
                        ),
                    )
                )
                latest_evaluation = (
                    summarize_evaluation_results(
                        evaluation_results
                    )
                )

                logger.log_metrics(
                    "evaluation",
                    latest_evaluation,
                    step=global_step,
                    episode=(
                        episode_index
                        - 1
                    ),
                )
                logger.update_summary(
                    latest_evaluation,
                    prefix=(
                        "evaluation_latest"
                    ),
                )

                last_eval_step = (
                    global_step
                )

                if evaluation_is_better(
                    latest_evaluation,
                    best_evaluation,
                ):
                    best_evaluation = (
                        copy.deepcopy(
                            latest_evaluation
                        )
                    )

                    best_state = {
                        "seed": seed,
                        "global_step": (
                            global_step
                        ),
                        "episode_index": (
                            episode_index
                        ),
                        "best_evaluation": (
                            best_evaluation
                        ),
                        "last_eval_step": (
                            last_eval_step
                        ),
                        "last_checkpoint_step": (
                            last_checkpoint_step
                        ),
                        "training_rng_state_json": (
                            json.dumps(
                                rng
                                .bit_generator
                                .state
                            )
                        ),
                        "resume_safe": False,
                    }

                    best_checkpoint_path = (
                        save_masac_checkpoint(
                            checkpoint_dir
                            / "best.pt",
                            trainer,
                            replay_buffer=(
                                replay_buffer
                            ),
                            training_state=(
                                best_state
                            ),
                            include_replay=False,
                        )
                    )
                    logger.update_summary(
                        best_evaluation,
                        prefix="best",
                    )
                    logger.log_model_artifact(
                        best_checkpoint_path,
                        aliases=[
                            "best",
                        ],
                        metadata=(
                            best_evaluation
                        ),
                    )

            should_checkpoint = (
                global_step
                - last_checkpoint_step
                >= checkpoint_interval
                or global_step
                >= total_steps
            )

            if should_checkpoint:
                last_checkpoint_step = (
                    global_step
                )

                include_replay = bool(
                    CONFIG[
                        "training_checkpoint_include_replay"
                    ]
                )

                checkpoint_state = {
                    "seed": seed,
                    "global_step": (
                        global_step
                    ),
                    "episode_index": (
                        episode_index
                    ),
                    "best_evaluation": (
                        copy.deepcopy(
                            best_evaluation
                        )
                    ),
                    "last_eval_step": (
                        last_eval_step
                    ),
                    "last_checkpoint_step": (
                        last_checkpoint_step
                    ),
                    "training_rng_state_json": (
                        json.dumps(
                            rng
                            .bit_generator
                            .state
                        )
                    ),
                    "resume_safe": (
                        include_replay
                    ),
                }

                save_masac_checkpoint(
                    checkpoint_dir
                    / "latest.pt",
                    trainer,
                    replay_buffer=(
                        replay_buffer
                    ),
                    training_state=(
                        checkpoint_state
                    ),
                    include_replay=(
                        include_replay
                    ),
                )

            episode_return = 0.0
            episode_length = 0
            episode_diagnostics = (
                new_episode_diagnostics()
            )

            if global_step < total_steps:
                observations, info = (
                    env.reset(
                        seed=(
                            seed
                            + episode_index
                        )
                    )
                )
                state = info[
                    "state"
                ]

        final_state = {
            "seed": seed,
            "global_step": (
                global_step
            ),
            "episode_index": (
                episode_index
            ),
            "best_evaluation": (
                copy.deepcopy(
                    best_evaluation
                )
            ),
            "last_eval_step": (
                last_eval_step
            ),
            "last_checkpoint_step": (
                last_checkpoint_step
            ),
            "training_rng_state_json": (
                json.dumps(
                    rng
                    .bit_generator
                    .state
                )
            ),
            "resume_safe": False,
        }

        # Always keep a lightweight final model checkpoint. If training
        # stops mid-episode it is valid for evaluation, but resume_safe
        # is false because environment state is intentionally not serialized.
        final_checkpoint_path = (
            save_masac_checkpoint(
                checkpoint_dir
                / "final.pt",
                trainer,
                replay_buffer=(
                    replay_buffer
                ),
                training_state=(
                    final_state
                ),
                include_replay=False,
            )
        )
        logger.log_model_artifact(
            final_checkpoint_path,
            aliases=[
                "final",
            ],
            metadata={
                "global_step": (
                    global_step
                ),
                "episode_index": (
                    episode_index
                ),
            },
        )
        logger.update_summary(
            {
                "global_step": (
                    global_step
                ),
                "episodes_completed": (
                    episode_index
                ),
                "trainer_updates": int(
                    trainer.update_count
                ),
            },
            prefix="training_final",
        )

        return {
            "trainer": trainer,
            "replay_buffer": replay_buffer,
            "completed_episodes": (
                completed_episodes
            ),
            "latest_update_metrics": (
                latest_update_metrics
            ),
            "latest_evaluation": (
                latest_evaluation
            ),
            "best_evaluation": (
                best_evaluation
            ),
            "training_state": (
                final_state
            ),
            "logger_status": (
                logger.status()
            ),
            "run_dir": run_dir,
            "checkpoint_dir": (
                checkpoint_dir
            ),
        }
    finally:
        eval_env.close()
        logger.close()


In [ ]:
def straight_through_masked_argmax(
    logits,
    destination_mask,
    temperature=None,
):
    if temperature is None:
        temperature = CONFIG[
            "matd3_gumbel_temperature"
        ]

    temperature = float(
        temperature
    )

    if temperature <= 0.0:
        raise ValueError(
            "temperature must be > 0"
        )

    masked_logits = (
        masked_categorical_logits(
            logits,
            destination_mask,
        )
    )

    probabilities = F.softmax(
        masked_logits
        / temperature,
        dim=-1,
    )

    indices = torch.argmax(
        masked_logits,
        dim=-1,
    )

    hard = F.one_hot(
        indices,
        num_classes=(
            masked_logits.shape[-1]
        ),
    ).to(
        masked_logits.dtype
    )

    action = (
        hard
        - probabilities.detach()
        + probabilities
    )

    return {
        "action": action,
        "hard_action": hard,
        "indices": indices,
        "probabilities": probabilities,
    }


class HybridMATD3Actor(nn.Module):
    def __init__(
        self,
        observation_dim,
        continuous_dim,
        discrete_dim,
        hidden_dims,
    ):
        super().__init__()

        hidden_dims = tuple(
            int(value)
            for value
            in hidden_dims
        )

        if not hidden_dims:
            raise ValueError(
                "hidden_dims must not be empty"
            )

        self.observation_dim = int(
            observation_dim
        )
        self.continuous_dim = int(
            continuous_dim
        )
        self.discrete_dim = int(
            discrete_dim
        )

        self.encoder = build_mlp(
            self.observation_dim,
            hidden_dims[:-1],
            hidden_dims[-1],
        )

        feature_dim = hidden_dims[-1]

        self.continuous_head = nn.Linear(
            feature_dim,
            self.continuous_dim,
        )
        self.discrete_head = nn.Linear(
            feature_dim,
            self.discrete_dim,
        )

    def forward(
        self,
        observations,
    ):
        features = self.encoder(
            observations
        )

        continuous = torch.tanh(
            self.continuous_head(
                features
            )
        )
        logits = self.discrete_head(
            features
        )

        return (
            continuous,
            logits,
        )

    def actions(
        self,
        observations,
        destination_mask,
        straight_through=True,
    ):
        (
            continuous,
            logits,
        ) = self(
            observations
        )

        discrete = (
            straight_through_masked_argmax(
                logits,
                destination_mask,
            )
        )

        destination_action = (
            discrete["action"]
            if straight_through
            else discrete[
                "hard_action"
            ]
        )

        return {
            "continuous": continuous,
            "destination_one_hot": (
                destination_action
            ),
            "destination_hard_one_hot": (
                discrete[
                    "hard_action"
                ]
            ),
            "destination_indices": (
                discrete["indices"]
            ),
            "destination_probabilities": (
                discrete[
                    "probabilities"
                ]
            ),
        }


In [ ]:
class HybridMATD3:
    def __init__(
        self,
        env,
        device=None,
        seed=None,
    ):
        if not isinstance(
            env,
            UAVSearchEnv,
        ):
            raise TypeError(
                "env must be UAVSearchEnv"
            )

        if seed is None:
            seed = int(
                CONFIG["seed"]
            )

        self.seed = int(seed)
        self.rng = (
            np.random.default_rng(
                self.seed
            )
        )

        self.agent_ids = tuple(
            env.agent_ids
        )
        self.num_agents = len(
            self.agent_ids
        )

        if device is None:
            device = (
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )

        self.device = torch.device(
            device
        )

        agent_space = (
            env.observation_space
            .spaces[
                self.agent_ids[0]
            ]
        )

        self.observation_dim = int(
            sum(
                np.prod(
                    agent_space.spaces[
                        key
                    ].shape,
                    dtype=np.int64,
                )
                for key
                in AGENT_OBSERVATION_VECTOR_KEYS
            )
        )
        self.state_dim = int(
            np.prod(
                env.state_space.shape,
                dtype=np.int64,
            )
        )
        self.continuous_dim = 4
        self.discrete_dim = int(
            env.action_space
            .spaces[
                self.agent_ids[0]
            ]
            .spaces[
                "destination"
            ]
            .n
        )
        self.joint_action_dim = (
            self.num_agents
            * (
                self.continuous_dim
                + self.discrete_dim
            )
        )

        hidden_dims = tuple(
            CONFIG[
                "matd3_hidden_dims"
            ]
        )

        self.actor = HybridMATD3Actor(
            self.observation_dim,
            self.continuous_dim,
            self.discrete_dim,
            hidden_dims,
        ).to(
            self.device
        )

        self.target_actor = copy.deepcopy(
            self.actor
        ).to(
            self.device
        )

        self.critic_1 = (
            CentralizedQNetwork(
                self.state_dim,
                self.joint_action_dim,
                hidden_dims,
            ).to(
                self.device
            )
        )
        self.critic_2 = (
            CentralizedQNetwork(
                self.state_dim,
                self.joint_action_dim,
                hidden_dims,
            ).to(
                self.device
            )
        )

        self.target_critic_1 = (
            copy.deepcopy(
                self.critic_1
            ).to(
                self.device
            )
        )
        self.target_critic_2 = (
            copy.deepcopy(
                self.critic_2
            ).to(
                self.device
            )
        )

        for network in (
            self.target_actor,
            self.target_critic_1,
            self.target_critic_2,
        ):
            network.eval()

            for parameter in (
                network.parameters()
            ):
                parameter.requires_grad_(
                    False
                )

        self.actor_optimizer = (
            torch.optim.Adam(
                self.actor.parameters(),
                lr=float(
                    CONFIG[
                        "matd3_actor_lr"
                    ]
                ),
            )
        )
        self.critic_optimizer = (
            torch.optim.Adam(
                list(
                    self.critic_1.parameters()
                )
                + list(
                    self.critic_2.parameters()
                ),
                lr=float(
                    CONFIG[
                        "matd3_critic_lr"
                    ]
                ),
            )
        )

        self.gamma = float(
            CONFIG["matd3_gamma"]
        )
        self.tau = float(
            CONFIG["matd3_tau"]
        )
        self.policy_delay = int(
            CONFIG[
                "matd3_policy_delay"
            ]
        )
        self.target_policy_noise = float(
            CONFIG[
                "matd3_target_policy_noise"
            ]
        )
        self.target_noise_clip = float(
            CONFIG[
                "matd3_target_noise_clip"
            ]
        )
        self.exploration_noise = float(
            CONFIG[
                "matd3_exploration_noise"
            ]
        )
        self.gradient_clip_norm = float(
            CONFIG[
                "matd3_gradient_clip_norm"
            ]
        )

        if self.policy_delay < 1:
            raise ValueError(
                "matd3_policy_delay "
                "must be >= 1"
            )

        self.update_count = 0
        self.action_count = 0

    def make_replay_buffer(
        self,
        seed=None,
    ):
        if seed is None:
            seed = self.seed

        return HybridReplayBuffer(
            capacity=CONFIG[
                "matd3_replay_capacity"
            ],
            num_agents=(
                self.num_agents
            ),
            observation_dim=(
                self.observation_dim
            ),
            state_dim=(
                self.state_dim
            ),
            continuous_dim=(
                self.continuous_dim
            ),
            discrete_dim=(
                self.discrete_dim
            ),
            seed=seed,
        )

    def _joint_action_tensor(
        self,
        continuous_actions,
        destination_one_hot,
    ):
        if continuous_actions.ndim != 3:
            raise ValueError(
                "continuous_actions must "
                "have rank 3"
            )

        if destination_one_hot.ndim != 3:
            raise ValueError(
                "destination_one_hot must "
                "have rank 3"
            )

        combined = torch.cat(
            [
                continuous_actions,
                destination_one_hot,
            ],
            dim=-1,
        )

        return combined.reshape(
            combined.shape[0],
            -1,
        )

    def _actor_joint_actions(
        self,
        observations,
        destination_masks,
        straight_through=True,
        target_actor=False,
    ):
        if observations.ndim != 3:
            raise ValueError(
                "observations must "
                "have rank 3"
            )

        batch_size = int(
            observations.shape[0]
        )

        flat_observations = (
            observations.reshape(
                batch_size
                * self.num_agents,
                self.observation_dim,
            )
        )
        flat_masks = (
            destination_masks.reshape(
                batch_size
                * self.num_agents,
                self.discrete_dim,
            )
        )

        actor = (
            self.target_actor
            if target_actor
            else self.actor
        )

        sampled = actor.actions(
            flat_observations,
            flat_masks,
            straight_through=(
                straight_through
            ),
        )

        continuous = sampled[
            "continuous"
        ].reshape(
            batch_size,
            self.num_agents,
            self.continuous_dim,
        )
        destination_one_hot = (
            sampled[
                "destination_one_hot"
            ].reshape(
                batch_size,
                self.num_agents,
                self.discrete_dim,
            )
        )
        destination_indices = (
            sampled[
                "destination_indices"
            ].reshape(
                batch_size,
                self.num_agents,
            )
        )

        return {
            "continuous": continuous,
            "destination_one_hot": (
                destination_one_hot
            ),
            "destination_indices": (
                destination_indices
            ),
        }

    def _discrete_epsilon(
        self,
        step,
    ):
        start = float(
            CONFIG[
                "matd3_discrete_epsilon_start"
            ]
        )
        end = float(
            CONFIG[
                "matd3_discrete_epsilon_end"
            ]
        )
        decay_steps = max(
            1,
            int(
                CONFIG[
                    "matd3_discrete_epsilon_decay_steps"
                ]
            ),
        )

        fraction = min(
            1.0,
            max(
                0.0,
                float(step)
                / decay_steps,
            ),
        )

        return float(
            start
            + fraction
            * (
                end - start
            )
        )

    def select_actions(
        self,
        observations,
        deterministic=False,
        exploration_step=None,
    ):
        (
            observation_vectors,
            destination_masks,
        ) = observations_to_masac_arrays(
            observations,
            self.agent_ids,
        )

        observation_tensor = (
            torch.as_tensor(
                observation_vectors,
                dtype=torch.float32,
                device=self.device,
            )
        )
        mask_tensor = torch.as_tensor(
            destination_masks,
            dtype=torch.float32,
            device=self.device,
        )

        with torch.no_grad():
            sampled = self.actor.actions(
                observation_tensor,
                mask_tensor,
                straight_through=False,
            )

        continuous = (
            sampled[
                "continuous"
            ]
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )
        destination_indices = (
            sampled[
                "destination_indices"
            ]
            .cpu()
            .numpy()
            .astype(
                np.int64
            )
        )

        if not deterministic:
            noise = self.rng.normal(
                loc=0.0,
                scale=self.exploration_noise,
                size=continuous.shape,
            ).astype(
                np.float32
            )

            continuous = np.clip(
                continuous + noise,
                -1.0,
                1.0,
            ).astype(
                np.float32
            )

            if exploration_step is None:
                exploration_step = (
                    self.action_count
                )

            epsilon = (
                self._discrete_epsilon(
                    exploration_step
                )
            )

            for agent_index in range(
                self.num_agents
            ):
                if (
                    self.rng.random()
                    >= epsilon
                ):
                    continue

                valid = np.flatnonzero(
                    destination_masks[
                        agent_index
                    ]
                    > 0.5
                )

                destination_indices[
                    agent_index
                ] = int(
                    self.rng.choice(
                        valid
                    )
                )

            self.action_count += 1

        return masac_arrays_to_env_actions(
            continuous,
            destination_indices,
            self.agent_ids,
        )

    def _soft_update(
        self,
        target,
        source,
    ):
        with torch.no_grad():
            for (
                target_parameter,
                source_parameter,
            ) in zip(
                target.parameters(),
                source.parameters(),
            ):
                target_parameter.mul_(
                    1.0 - self.tau
                )
                target_parameter.add_(
                    self.tau
                    * source_parameter
                )

    def update(
        self,
        replay_buffer,
        batch_size=None,
    ):
        if not isinstance(
            replay_buffer,
            HybridReplayBuffer,
        ):
            raise TypeError(
                "replay_buffer must be "
                "HybridReplayBuffer"
            )

        if batch_size is None:
            batch_size = int(
                CONFIG[
                    "matd3_batch_size"
                ]
            )

        batch = replay_buffer.sample(
            batch_size,
            self.device,
        )

        replay_destination_one_hot = (
            F.one_hot(
                batch[
                    "destination_indices"
                ],
                num_classes=(
                    self.discrete_dim
                ),
            ).to(
                torch.float32
            )
        )

        replay_joint_action = (
            self._joint_action_tensor(
                batch[
                    "continuous_actions"
                ],
                replay_destination_one_hot,
            )
        )

        with torch.no_grad():
            target_policy = (
                self._actor_joint_actions(
                    batch[
                        "next_observations"
                    ],
                    batch[
                        "next_destination_masks"
                    ],
                    straight_through=False,
                    target_actor=True,
                )
            )

            noise = (
                torch.randn_like(
                    target_policy[
                        "continuous"
                    ]
                )
                * self.target_policy_noise
            ).clamp(
                -self.target_noise_clip,
                self.target_noise_clip,
            )

            target_continuous = (
                target_policy[
                    "continuous"
                ]
                + noise
            ).clamp(
                -1.0,
                1.0,
            )

            target_joint_action = (
                self._joint_action_tensor(
                    target_continuous,
                    target_policy[
                        "destination_one_hot"
                    ],
                )
            )

            target_q = torch.minimum(
                self.target_critic_1(
                    batch[
                        "next_states"
                    ],
                    target_joint_action,
                ),
                self.target_critic_2(
                    batch[
                        "next_states"
                    ],
                    target_joint_action,
                ),
            )

            critic_target = (
                batch["rewards"]
                + self.gamma
                * (
                    1.0
                    - batch[
                        "terminated"
                    ]
                )
                * target_q
            )

        critic_1_value = (
            self.critic_1(
                batch["states"],
                replay_joint_action,
            )
        )
        critic_2_value = (
            self.critic_2(
                batch["states"],
                replay_joint_action,
            )
        )

        critic_loss = (
            F.mse_loss(
                critic_1_value,
                critic_target,
            )
            + F.mse_loss(
                critic_2_value,
                critic_target,
            )
        )

        self.critic_optimizer.zero_grad(
            set_to_none=True
        )
        critic_loss.backward()

        critic_grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                list(
                    self.critic_1.parameters()
                )
                + list(
                    self.critic_2.parameters()
                ),
                self.gradient_clip_norm,
            )
        )

        self.critic_optimizer.step()

        self.update_count += 1

        actor_updated = bool(
            self.update_count
            % self.policy_delay
            == 0
        )
        actor_loss_value = None
        actor_grad_norm_value = None

        if actor_updated:
            for critic in (
                self.critic_1,
                self.critic_2,
            ):
                for parameter in (
                    critic.parameters()
                ):
                    parameter.requires_grad_(
                        False
                    )

            policy = (
                self._actor_joint_actions(
                    batch[
                        "observations"
                    ],
                    batch[
                        "destination_masks"
                    ],
                    straight_through=True,
                    target_actor=False,
                )
            )

            policy_joint_action = (
                self._joint_action_tensor(
                    policy[
                        "continuous"
                    ],
                    policy[
                        "destination_one_hot"
                    ],
                )
            )

            actor_loss = -self.critic_1(
                batch["states"],
                policy_joint_action,
            ).mean()

            self.actor_optimizer.zero_grad(
                set_to_none=True
            )
            actor_loss.backward()

            actor_grad_norm = (
                torch.nn.utils.clip_grad_norm_(
                    self.actor.parameters(),
                    self.gradient_clip_norm,
                )
            )

            self.actor_optimizer.step()

            for critic in (
                self.critic_1,
                self.critic_2,
            ):
                for parameter in (
                    critic.parameters()
                ):
                    parameter.requires_grad_(
                        True
                    )

            self._soft_update(
                self.target_actor,
                self.actor,
            )
            self._soft_update(
                self.target_critic_1,
                self.critic_1,
            )
            self._soft_update(
                self.target_critic_2,
                self.critic_2,
            )

            actor_loss_value = float(
                actor_loss
                .detach()
                .cpu()
            )
            actor_grad_norm_value = float(
                torch.as_tensor(
                    actor_grad_norm
                )
                .detach()
                .cpu()
            )

        metrics = {
            "critic_loss": float(
                critic_loss
                .detach()
                .cpu()
            ),
            "critic_1_q_mean": float(
                critic_1_value
                .mean()
                .detach()
                .cpu()
            ),
            "critic_2_q_mean": float(
                critic_2_value
                .mean()
                .detach()
                .cpu()
            ),
            "target_q_mean": float(
                critic_target
                .mean()
                .detach()
                .cpu()
            ),
            "critic_grad_norm": float(
                torch.as_tensor(
                    critic_grad_norm
                )
                .detach()
                .cpu()
            ),
            "actor_updated": (
                actor_updated
            ),
            "actor_loss": (
                actor_loss_value
            ),
            "actor_grad_norm": (
                actor_grad_norm_value
            ),
            "update_count": int(
                self.update_count
            ),
        }

        for key, value in (
            metrics.items()
        ):
            if value is None:
                continue

            if isinstance(
                value,
                bool,
            ):
                continue

            if not np.isfinite(
                value
            ):
                raise FloatingPointError(
                    f"non-finite MATD3 metric: "
                    f"{key}"
                )

        return metrics

    @staticmethod
    def _move_optimizer_state_to_device(
        optimizer,
        device,
    ):
        for state in optimizer.state.values():
            for key, value in list(
                state.items()
            ):
                if isinstance(
                    value,
                    torch.Tensor,
                ):
                    state[key] = value.to(
                        device
                    )

    def state_dict(self):
        return {
            "actor": (
                self.actor.state_dict()
            ),
            "target_actor": (
                self.target_actor
                .state_dict()
            ),
            "critic_1": (
                self.critic_1.state_dict()
            ),
            "critic_2": (
                self.critic_2.state_dict()
            ),
            "target_critic_1": (
                self.target_critic_1
                .state_dict()
            ),
            "target_critic_2": (
                self.target_critic_2
                .state_dict()
            ),
            "actor_optimizer": (
                self.actor_optimizer
                .state_dict()
            ),
            "critic_optimizer": (
                self.critic_optimizer
                .state_dict()
            ),
            "update_count": int(
                self.update_count
            ),
            "action_count": int(
                self.action_count
            ),
            "rng_state_json": json.dumps(
                self.rng
                .bit_generator
                .state
            ),
        }

    def load_state_dict(
        self,
        state,
    ):
        self.actor.load_state_dict(
            state["actor"]
        )
        self.target_actor.load_state_dict(
            state["target_actor"]
        )
        self.critic_1.load_state_dict(
            state["critic_1"]
        )
        self.critic_2.load_state_dict(
            state["critic_2"]
        )
        self.target_critic_1.load_state_dict(
            state[
                "target_critic_1"
            ]
        )
        self.target_critic_2.load_state_dict(
            state[
                "target_critic_2"
            ]
        )

        self.actor_optimizer.load_state_dict(
            state[
                "actor_optimizer"
            ]
        )
        self.critic_optimizer.load_state_dict(
            state[
                "critic_optimizer"
            ]
        )

        for optimizer in (
            self.actor_optimizer,
            self.critic_optimizer,
        ):
            self._move_optimizer_state_to_device(
                optimizer,
                self.device,
            )

        self.update_count = int(
            state.get(
                "update_count",
                0,
            )
        )
        self.action_count = int(
            state.get(
                "action_count",
                0,
            )
        )

        self.rng.bit_generator.state = (
            json.loads(
                state[
                    "rng_state_json"
                ]
            )
        )


In [ ]:
def train_matd3(
    env,
    total_steps,
    seed=None,
    trainer=None,
    replay_buffer=None,
):
    if not isinstance(
        env,
        UAVSearchEnv,
    ):
        raise TypeError(
            "env must be UAVSearchEnv"
        )

    total_steps = int(
        total_steps
    )

    if total_steps < 1:
        raise ValueError(
            "total_steps must be >= 1"
        )

    if seed is None:
        seed = int(
            CONFIG["seed"]
        )

    seed = int(seed)

    set_seed(seed)

    if trainer is None:
        trainer = HybridMATD3(
            env,
            seed=seed,
        )

    if replay_buffer is None:
        replay_buffer = (
            trainer.make_replay_buffer(
                seed=seed,
            )
        )

    rng = np.random.default_rng(
        seed
    )

    observations, info = env.reset(
        seed=seed
    )
    state = info["state"]

    episode_return = 0.0
    episode_length = 0
    episode_index = 0
    completed_episodes = []
    latest_update_metrics = None

    learning_starts = int(
        CONFIG[
            "matd3_learning_starts"
        ]
    )
    updates_per_step = int(
        CONFIG[
            "matd3_updates_per_step"
        ]
    )
    batch_size = int(
        CONFIG[
            "matd3_batch_size"
        ]
    )

    for global_step in range(
        1,
        total_steps + 1,
    ):
        if global_step <= learning_starts:
            actions = (
                sample_valid_random_actions(
                    observations,
                    env.agent_ids,
                    rng,
                )
            )
        else:
            actions = (
                trainer.select_actions(
                    observations,
                    deterministic=False,
                    exploration_step=(
                        global_step
                    ),
                )
            )

        (
            next_observations,
            reward,
            terminated,
            truncated,
            next_info,
        ) = env.step(
            actions
        )

        next_state = next_info[
            "state"
        ]

        add_environment_transition_to_replay(
            replay_buffer,
            observations,
            state,
            actions,
            reward,
            next_observations,
            next_state,
            terminated,
            truncated,
            env.agent_ids,
        )

        episode_return += float(
            reward
        )
        episode_length += 1

        if (
            global_step > learning_starts
            and len(
                replay_buffer
            )
            >= batch_size
        ):
            for _ in range(
                updates_per_step
            ):
                latest_update_metrics = (
                    trainer.update(
                        replay_buffer,
                        batch_size=(
                            batch_size
                        ),
                    )
                )

        observations = (
            next_observations
        )
        state = next_state

        if terminated or truncated:
            completed_episodes.append(
                {
                    "episode": (
                        episode_index
                    ),
                    "return": float(
                        episode_return
                    ),
                    "length": int(
                        episode_length
                    ),
                    "terminated": bool(
                        terminated
                    ),
                    "truncated": bool(
                        truncated
                    ),
                    "success": bool(
                        next_info[
                            "success"
                        ]
                    ),
                }
            )

            episode_index += 1
            episode_return = 0.0
            episode_length = 0

            if global_step < total_steps:
                observations, info = (
                    env.reset(
                        seed=(
                            seed
                            + episode_index
                        )
                    )
                )
                state = info[
                    "state"
                ]

    return {
        "trainer": trainer,
        "replay_buffer": replay_buffer,
        "completed_episodes": (
            completed_episodes
        ),
        "latest_update_metrics": (
            latest_update_metrics
        ),
        "total_steps": total_steps,
    }


def evaluate_matd3(
    env,
    trainer,
    episodes=3,
    seed=None,
):
    if not isinstance(
        trainer,
        HybridMATD3,
    ):
        raise TypeError(
            "trainer must be HybridMATD3"
        )

    if seed is None:
        seed = int(
            CONFIG["seed"]
        ) + 20_000

    episodes = int(
        episodes
    )

    if episodes < 1:
        raise ValueError(
            "episodes must be >= 1"
        )

    results = []

    for episode_index in range(
        episodes
    ):
        observations, _ = env.reset(
            seed=(
                int(seed)
                + episode_index
            )
        )

        episode_return = 0.0
        diagnostics = (
            new_episode_diagnostics()
        )

        while True:
            actions = trainer.select_actions(
                observations,
                deterministic=True,
            )

            (
                observations,
                reward,
                terminated,
                truncated,
                info,
            ) = env.step(
                actions
            )

            episode_return += float(
                reward
            )

            update_episode_diagnostics(
                diagnostics,
                info,
            )

            if (
                terminated
                or truncated
            ):
                episode_metrics = (
                    finalize_episode_diagnostics(
                        diagnostics,
                        env,
                        episode_return=(
                            episode_return
                        ),
                        episode_length=(
                            info["step"]
                        ),
                        success=(
                            info["success"]
                        ),
                    )
                )

                results.append(
                    {
                        "episode": (
                            episode_index
                        ),
                        **episode_metrics,
                        "terminated": bool(
                            terminated
                        ),
                        "truncated": bool(
                            truncated
                        ),
                    }
                )
                break

    return results


In [ ]:
def save_matd3_checkpoint(
    path,
    trainer,
    replay_buffer=None,
    training_state=None,
    include_replay=None,
):
    if not isinstance(
        trainer,
        HybridMATD3,
    ):
        raise TypeError(
            "trainer must be HybridMATD3"
        )

    if include_replay is None:
        include_replay = bool(
            CONFIG[
                "training_checkpoint_include_replay"
            ]
        )

    include_replay = bool(
        include_replay
    )

    if (
        include_replay
        and replay_buffer is None
    ):
        raise ValueError(
            "replay_buffer is required "
            "when include_replay=True"
        )

    if training_state is None:
        training_state = {}

    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    payload = {
        "format_version": 1,
        "algorithm": "hybrid_matd3",
        "trainer_state": (
            trainer.state_dict()
        ),
        "training_state": (
            copy.deepcopy(
                training_state
            )
        ),
        "config_snapshot": (
            _checkpoint_config_snapshot()
        ),
        "replay_state": (
            replay_buffer.state_dict()
            if include_replay
            else None
        ),
        "torch_rng_state": (
            torch.get_rng_state()
        ),
        "cuda_rng_states": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else []
        ),
    }

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    torch.save(
        payload,
        temporary_path,
    )
    temporary_path.replace(
        path
    )

    return path


def load_matd3_checkpoint(
    path,
    env,
    device=None,
    load_replay=True,
    restore_torch_rng=True,
):
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            str(path)
        )

    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=True,
    )

    if payload.get(
        "algorithm"
    ) != "hybrid_matd3":
        raise ValueError(
            "checkpoint algorithm "
            "is not hybrid_matd3"
        )

    trainer = HybridMATD3(
        env,
        device=device,
        seed=int(
            payload.get(
                "training_state",
                {},
            ).get(
                "seed",
                CONFIG["seed"],
            )
        ),
    )
    trainer.load_state_dict(
        payload[
            "trainer_state"
        ]
    )

    replay_buffer = None
    replay_state = payload.get(
        "replay_state"
    )

    if (
        load_replay
        and replay_state is not None
    ):
        replay_buffer = (
            trainer.make_replay_buffer(
                seed=trainer.seed
            )
        )
        replay_buffer.load_state_dict(
            replay_state
        )

    if restore_torch_rng:
        torch.set_rng_state(
            payload[
                "torch_rng_state"
            ]
        )

        cuda_states = payload.get(
            "cuda_rng_states",
            [],
        )

        if (
            torch.cuda.is_available()
            and cuda_states
        ):
            torch.cuda.set_rng_state_all(
                cuda_states
            )

    return {
        "trainer": trainer,
        "replay_buffer": replay_buffer,
        "training_state": (
            copy.deepcopy(
                payload.get(
                    "training_state",
                    {},
                )
            )
        ),
        "config_snapshot": (
            copy.deepcopy(
                payload.get(
                    "config_snapshot",
                    {},
                )
            )
        ),
        "path": path,
    }


In [ ]:
def train_matd3_experiment(
    env,
    total_steps,
    seed=None,
    run_dir=None,
    run_name=None,
    resume_checkpoint=None,
    device=None,
    enable_csv=None,
    enable_tensorboard=None,
    enable_wandb=None,
):
    if not isinstance(
        env,
        UAVSearchEnv,
    ):
        raise TypeError(
            "env must be UAVSearchEnv"
        )

    total_steps = int(
        total_steps
    )

    if total_steps < 1:
        raise ValueError(
            "total_steps must be >= 1"
        )

    if seed is None:
        seed = int(
            CONFIG["seed"]
        )

    seed = int(seed)

    if run_name is None:
        run_name = (
            make_experiment_run_name(
                "matd3",
                seed,
            )
        )

    if run_dir is None:
        if resume_checkpoint is not None:
            run_dir = (
                Path(
                    resume_checkpoint
                )
                .resolve()
                .parent
                .parent
            )
        else:
            run_dir = (
                Path(
                    CONFIG[
                        "matd3_training_output_dir"
                    ]
                )
                / run_name
            )

    run_dir = Path(
        run_dir
    )
    checkpoint_dir = (
        run_dir / "checkpoints"
    )
    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    set_seed(seed)

    if resume_checkpoint is None:
        trainer = HybridMATD3(
            env,
            device=device,
            seed=seed,
        )
        replay_buffer = (
            trainer.make_replay_buffer(
                seed=seed
            )
        )
        training_state = {
            "seed": seed,
            "global_step": 0,
            "episode_index": 0,
            "best_evaluation": None,
            "last_eval_step": 0,
            "last_checkpoint_step": 0,
            "training_rng_state_json": None,
            "resume_safe": True,
        }
    else:
        restored = (
            load_matd3_checkpoint(
                resume_checkpoint,
                env,
                device=device,
                load_replay=True,
                restore_torch_rng=True,
            )
        )
        trainer = restored[
            "trainer"
        ]
        replay_buffer = restored[
            "replay_buffer"
        ]
        training_state = restored[
            "training_state"
        ]

        if replay_buffer is None:
            raise ValueError(
                "resume checkpoint does not "
                "contain replay state; set "
                "training_checkpoint_include_replay=True"
            )

        if not bool(
            training_state.get(
                "resume_safe",
                False,
            )
        ):
            raise ValueError(
                "checkpoint is not marked "
                "resume_safe"
            )

        if int(
            training_state.get(
                "seed",
                seed,
            )
        ) != seed:
            raise ValueError(
                "resume seed does not match "
                "checkpoint seed"
            )

    global_step = int(
        training_state.get(
            "global_step",
            0,
        )
    )
    episode_index = int(
        training_state.get(
            "episode_index",
            0,
        )
    )
    best_evaluation = copy.deepcopy(
        training_state.get(
            "best_evaluation"
        )
    )
    last_eval_step = int(
        training_state.get(
            "last_eval_step",
            0,
        )
    )
    last_checkpoint_step = int(
        training_state.get(
            "last_checkpoint_step",
            0,
        )
    )

    if global_step > total_steps:
        raise ValueError(
            "total_steps is smaller than "
            "checkpoint global_step"
        )

    rng = np.random.default_rng(
        seed
    )

    saved_rng_state = (
        training_state.get(
            "training_rng_state_json"
        )
    )

    if saved_rng_state:
        rng.bit_generator.state = (
            json.loads(
                saved_rng_state
            )
        )

    logger = MASACExperimentLogger(
        run_dir=run_dir,
        run_name=run_name,
        enable_csv=enable_csv,
        enable_tensorboard=(
            enable_tensorboard
        ),
        enable_wandb=enable_wandb,
        wandb_entity=CONFIG.get(
            "training_wandb_entity"
        ),
        wandb_project=CONFIG[
            "matd3_wandb_project"
        ],
        algorithm="matd3",
        seed=seed,
        backend_name=(
            env.backend_name
        ),
        config_snapshot=(
            _checkpoint_config_snapshot()
        ),
    )

    eval_env = UAVSearchEnv(
        backend_name=env.backend_name,
        network_backend_kwargs=(
            env.network_backend_kwargs
        ),
    )

    observations, info = env.reset(
        seed=(
            seed + episode_index
        )
    )
    state = info[
        "state"
    ]

    episode_return = 0.0
    episode_length = 0
    episode_diagnostics = (
        new_episode_diagnostics()
    )
    completed_episodes = []
    latest_update_metrics = None
    latest_evaluation = None

    learning_starts = int(
        CONFIG[
            "matd3_learning_starts"
        ]
    )
    batch_size = int(
        CONFIG[
            "matd3_batch_size"
        ]
    )
    updates_per_step = int(
        CONFIG[
            "matd3_updates_per_step"
        ]
    )
    log_interval = max(
        1,
        int(
            CONFIG[
                "training_log_interval_steps"
            ]
        ),
    )
    eval_interval = max(
        1,
        int(
            CONFIG[
                "training_eval_interval_steps"
            ]
        ),
    )
    checkpoint_interval = max(
        1,
        int(
            CONFIG[
                "training_checkpoint_interval_steps"
            ]
        ),
    )
    eval_episodes = max(
        1,
        int(
            CONFIG[
                "training_eval_episodes"
            ]
        ),
    )

    try:
        while global_step < total_steps:
            next_global_step = (
                global_step + 1
            )

            if (
                next_global_step
                <= learning_starts
            ):
                actions = (
                    sample_valid_random_actions(
                        observations,
                        env.agent_ids,
                        rng,
                    )
                )
            else:
                actions = (
                    trainer.select_actions(
                        observations,
                        deterministic=False,
                        exploration_step=(
                            next_global_step
                        ),
                    )
                )

            (
                next_observations,
                reward,
                terminated,
                truncated,
                next_info,
            ) = env.step(
                actions
            )

            next_state = next_info[
                "state"
            ]

            add_environment_transition_to_replay(
                replay_buffer,
                observations,
                state,
                actions,
                reward,
                next_observations,
                next_state,
                terminated,
                truncated,
                env.agent_ids,
            )

            global_step = (
                next_global_step
            )
            episode_return += float(
                reward
            )
            episode_length += 1

            update_episode_diagnostics(
                episode_diagnostics,
                next_info,
            )

            if (
                global_step
                > learning_starts
                and len(
                    replay_buffer
                )
                >= batch_size
            ):
                for _ in range(
                    updates_per_step
                ):
                    latest_update_metrics = (
                        trainer.update(
                            replay_buffer,
                            batch_size=(
                                batch_size
                            ),
                        )
                    )

            if (
                latest_update_metrics
                is not None
                and global_step
                % log_interval
                == 0
            ):
                logger.log_metrics(
                    "train",
                    filter_training_metrics_for_wandb(
                        "matd3",
                        latest_update_metrics,
                    ),
                    step=global_step,
                    episode=episode_index,
                )

            observations = (
                next_observations
            )
            state = next_state

            if not (
                terminated
                or truncated
            ):
                continue

            episode_metrics = (
                finalize_episode_diagnostics(
                    episode_diagnostics,
                    env,
                    episode_return=(
                        episode_return
                    ),
                    episode_length=(
                        episode_length
                    ),
                    success=(
                        next_info[
                            "success"
                        ]
                    ),
                )
            )

            logger.log_metrics(
                "episode",
                episode_metrics,
                step=global_step,
                episode=episode_index,
            )

            completed_episodes.append(
                {
                    "episode": (
                        episode_index
                    ),
                    **episode_metrics,
                    "terminated": bool(
                        terminated
                    ),
                    "truncated": bool(
                        truncated
                    ),
                }
            )

            episode_index += 1

            should_evaluate = (
                global_step
                - last_eval_step
                >= eval_interval
                or global_step
                >= total_steps
            )

            if should_evaluate:
                evaluation_results = (
                    evaluate_matd3(
                        eval_env,
                        trainer,
                        episodes=(
                            eval_episodes
                        ),
                        seed=(
                            seed
                            + int(
                                CONFIG[
                                    "training_eval_seed_offset"
                                ]
                            )
                        ),
                    )
                )
                latest_evaluation = (
                    summarize_evaluation_results(
                        evaluation_results
                    )
                )

                logger.log_metrics(
                    "evaluation",
                    latest_evaluation,
                    step=global_step,
                    episode=(
                        episode_index - 1
                    ),
                )
                logger.update_summary(
                    latest_evaluation,
                    prefix=(
                        "evaluation_latest"
                    ),
                )

                last_eval_step = (
                    global_step
                )

                if evaluation_is_better(
                    latest_evaluation,
                    best_evaluation,
                ):
                    best_evaluation = (
                        copy.deepcopy(
                            latest_evaluation
                        )
                    )

                    best_checkpoint_path = (
                        save_matd3_checkpoint(
                            checkpoint_dir
                            / "best.pt",
                            trainer,
                            replay_buffer=(
                                replay_buffer
                            ),
                            training_state={
                                "seed": seed,
                                "global_step": (
                                    global_step
                                ),
                                "episode_index": (
                                    episode_index
                                ),
                                "best_evaluation": (
                                    best_evaluation
                                ),
                                "last_eval_step": (
                                    last_eval_step
                                ),
                                "last_checkpoint_step": (
                                    last_checkpoint_step
                                ),
                                "training_rng_state_json": (
                                    json.dumps(
                                        rng
                                        .bit_generator
                                        .state
                                    )
                                ),
                                "resume_safe": False,
                            },
                            include_replay=False,
                        )
                    )
                    logger.update_summary(
                        best_evaluation,
                        prefix="best",
                    )
                    logger.log_model_artifact(
                        best_checkpoint_path,
                        aliases=[
                            "best",
                        ],
                        metadata=(
                            best_evaluation
                        ),
                    )

            should_checkpoint = (
                global_step
                - last_checkpoint_step
                >= checkpoint_interval
                or global_step
                >= total_steps
            )

            if should_checkpoint:
                last_checkpoint_step = (
                    global_step
                )
                include_replay = bool(
                    CONFIG[
                        "training_checkpoint_include_replay"
                    ]
                )

                save_matd3_checkpoint(
                    checkpoint_dir
                    / "latest.pt",
                    trainer,
                    replay_buffer=(
                        replay_buffer
                    ),
                    training_state={
                        "seed": seed,
                        "global_step": (
                            global_step
                        ),
                        "episode_index": (
                            episode_index
                        ),
                        "best_evaluation": (
                            copy.deepcopy(
                                best_evaluation
                            )
                        ),
                        "last_eval_step": (
                            last_eval_step
                        ),
                        "last_checkpoint_step": (
                            last_checkpoint_step
                        ),
                        "training_rng_state_json": (
                            json.dumps(
                                rng
                                .bit_generator
                                .state
                            )
                        ),
                        "resume_safe": (
                            include_replay
                        ),
                    },
                    include_replay=(
                        include_replay
                    ),
                )

            episode_return = 0.0
            episode_length = 0
            episode_diagnostics = (
                new_episode_diagnostics()
            )

            if global_step < total_steps:
                observations, info = (
                    env.reset(
                        seed=(
                            seed
                            + episode_index
                        )
                    )
                )
                state = info[
                    "state"
                ]

        final_state = {
            "seed": seed,
            "global_step": (
                global_step
            ),
            "episode_index": (
                episode_index
            ),
            "best_evaluation": (
                copy.deepcopy(
                    best_evaluation
                )
            ),
            "last_eval_step": (
                last_eval_step
            ),
            "last_checkpoint_step": (
                last_checkpoint_step
            ),
            "training_rng_state_json": (
                json.dumps(
                    rng
                    .bit_generator
                    .state
                )
            ),
            "resume_safe": False,
        }

        final_checkpoint_path = (
            save_matd3_checkpoint(
                checkpoint_dir
                / "final.pt",
                trainer,
                replay_buffer=(
                    replay_buffer
                ),
                training_state=(
                    final_state
                ),
                include_replay=False,
            )
        )
        logger.log_model_artifact(
            final_checkpoint_path,
            aliases=[
                "final",
            ],
            metadata={
                "global_step": (
                    global_step
                ),
                "episode_index": (
                    episode_index
                ),
            },
        )
        logger.update_summary(
            {
                "global_step": (
                    global_step
                ),
                "episodes_completed": (
                    episode_index
                ),
                "trainer_updates": int(
                    trainer.update_count
                ),
            },
            prefix="training_final",
        )

        return {
            "trainer": trainer,
            "replay_buffer": replay_buffer,
            "completed_episodes": (
                completed_episodes
            ),
            "latest_update_metrics": (
                latest_update_metrics
            ),
            "latest_evaluation": (
                latest_evaluation
            ),
            "best_evaluation": (
                best_evaluation
            ),
            "training_state": (
                final_state
            ),
            "logger_status": (
                logger.status()
            ),
            "run_dir": run_dir,
            "checkpoint_dir": (
                checkpoint_dir
            ),
        }
    finally:
        eval_env.close()
        logger.close()


In [ ]:
def running_on_kaggle():
    return bool(
        os.environ.get(
            "KAGGLE_KERNEL_RUN_TYPE"
        )
        or Path(
            "/kaggle/working"
        ).is_dir()
    )


def kaggle_runtime_status():
    cuda_available = bool(
        torch.cuda.is_available()
    )
    device_count = (
        torch.cuda.device_count()
        if cuda_available
        else 0
    )
    gpu_names = [
        torch.cuda.get_device_name(
            index
        )
        for index in range(
            device_count
        )
    ]

    return {
        "running_on_kaggle": (
            running_on_kaggle()
        ),
        "cuda_available": (
            cuda_available
        ),
        "cuda_device_count": int(
            device_count
        ),
        "device": (
            "cuda"
            if cuda_available
            else "cpu"
        ),
        "gpu_names": gpu_names,
        "wandb_api_key_present": bool(
            os.environ.get(
                "WANDB_API_KEY"
            )
        ),
    }


def configure_kaggle_wandb(
    secret_names=None,
    verify_login=True,
):
    result = {
        "running_on_kaggle": (
            running_on_kaggle()
        ),
        "configured": False,
        "secret_name": None,
        "error": None,
    }

    if not result[
        "running_on_kaggle"
    ]:
        result["error"] = (
            "not running inside a "
            "Kaggle notebook"
        )
        return result

    if secret_names is None:
        secret_names = CONFIG[
            "kaggle_wandb_secret_names"
        ]

    try:
        from kaggle_secrets import (
            UserSecretsClient,
        )

        client = UserSecretsClient()
    except Exception as exc:  # noqa: BLE001 - Kaggle-only optional API
        result["error"] = (
            f"{type(exc).__name__}: "
            f"{exc}"
        )
        return result

    api_key = os.environ.get(
        "WANDB_API_KEY"
    )
    selected_name = None

    if not api_key:
        for secret_name in (
            secret_names
        ):
            try:
                candidate = (
                    client.get_secret(
                        str(
                            secret_name
                        )
                    )
                )
            except Exception:  # noqa: BLE001,S112 - try the next configured label
                continue

            if candidate:
                api_key = candidate
                selected_name = str(
                    secret_name
                )
                break

    if not api_key:
        result["error"] = (
            "no attached W&B secret "
            "was found"
        )
        return result

    os.environ[
        "WANDB_API_KEY"
    ] = api_key

    try:
        wandb = importlib.import_module(
            "wandb"
        )

        if verify_login:
            wandb.login(
                key=api_key,
                relogin=True,
                verify=True,
            )

        result["configured"] = True
        result["secret_name"] = (
            selected_name
        )
    except Exception as exc:  # noqa: BLE001 - optional logger must not crash setup
        result["error"] = (
            f"{type(exc).__name__}: "
            f"{exc}"
        )

    return result


def prepare_kaggle_training(
    algorithm="matd3",
    configure_wandb=True,
):
    normalized = str(
        algorithm
    ).strip().lower()

    if normalized not in {
        "matd3",
        "masac",
    }:
        raise ValueError(
            "algorithm must be "
            "'matd3' or 'masac'"
        )

    status = (
        kaggle_runtime_status()
    )

    if running_on_kaggle():
        output_root = Path(
            "/kaggle/working"
        )

        if normalized == "matd3":
            CONFIG[
                "matd3_training_output_dir"
            ] = str(
                output_root
                / "outputs"
                / "matd3"
            )
        else:
            CONFIG[
                "training_output_dir"
            ] = str(
                output_root
                / "outputs"
                / "masac"
            )

    wandb_status = None

    if configure_wandb:
        wandb_status = (
            configure_kaggle_wandb()
        )

        if (
            wandb_status.get(
                "configured"
            )
        ):
            CONFIG[
                "training_enable_wandb"
            ] = True
            CONFIG[
                "training_wandb_mode"
            ] = "online"

    return {
        **status,
        "algorithm": normalized,
        "wandb": wandb_status,
        "output_dir": (
            CONFIG[
                "matd3_training_output_dir"
            ]
            if normalized == "matd3"
            else CONFIG[
                "training_output_dir"
            ]
        ),
    }


In [ ]:
def run_kaggle_matd3_experiment(
    total_steps=100_000,
    seed=None,
    run_name=None,
    backend_name="simple",
    enable_wandb=True,
):
    if not running_on_kaggle():
        raise RuntimeError(
            "run_kaggle_matd3_experiment "
            "must be called inside a "
            "Kaggle notebook"
        )

    if seed is None:
        seed = int(
            CONFIG["seed"]
        )

    seed = int(seed)

    if run_name is None:
        run_name = (
            make_experiment_run_name(
                "matd3",
                seed,
            )
        )

    preparation = (
        prepare_kaggle_training(
            algorithm="matd3",
            configure_wandb=(
                enable_wandb
            ),
        )
    )

    if (
        enable_wandb
        and not preparation[
            "wandb"
        ].get(
            "configured",
            False,
        )
    ):
        raise RuntimeError(
            "W&B was requested but no "
            "attached Kaggle secret could "
            "be configured. Add a Kaggle "
            "Secret named WANDB_API_KEY "
            "or wandb_key."
        )

    run_dir = (
        Path(
            preparation[
                "output_dir"
            ]
        )
        / run_name
    )

    env = UAVSearchEnv(
        backend_name=backend_name
    )

    try:
        result = (
            train_matd3_experiment(
                env,
                total_steps=(
                    total_steps
                ),
                seed=seed,
                run_dir=run_dir,
                run_name=run_name,
                device=preparation[
                    "device"
                ],
                enable_csv=True,
                enable_tensorboard=(
                    CONFIG[
                        "training_enable_tensorboard"
                    ]
                ),
                enable_wandb=(
                    enable_wandb
                ),
            )
        )

        return {
            **result,
            "kaggle_status": (
                preparation
            ),
        }
    finally:
        env.close()


In [ ]:
def make_experiment_run_name(
    algorithm,
    seed,
    run_suffix=None,
):
    algorithm = str(
        algorithm
    ).strip().upper()

    if algorithm not in {
        "MASAC",
        "MATD3",
    }:
        raise ValueError(
            "algorithm must be MASAC or MATD3"
        )

    seed = int(seed)

    if run_suffix is None:
        # Uses OS randomness, not Python/NumPy/Torch RNG,
        # so display-name uniqueness does not change training determinism.
        run_suffix = secrets.token_hex(
            4
        )

    run_suffix = str(
        run_suffix
    ).strip()

    if not run_suffix:
        raise ValueError(
            "run_suffix must not be empty"
        )

    return (
        f"{algorithm}-"
        f"seed{seed}-"
        f"{run_suffix}"
    )


def run_training_experiment(
    algorithm,
    total_steps,
    seed=44,
    backend_name="simple",
    run_name=None,
    run_dir=None,
    resume_checkpoint=None,
    device=None,
    enable_csv=None,
    enable_tensorboard=None,
    enable_wandb=None,
):
    normalized = str(
        algorithm
    ).strip().lower()

    if normalized not in {
        "masac",
        "matd3",
    }:
        raise ValueError(
            "algorithm must be "
            "'masac' or 'matd3'"
        )

    seed = int(seed)

    if run_name is None:
        run_name = (
            make_experiment_run_name(
                normalized,
                seed,
            )
        )

    env = UAVSearchEnv(
        backend_name=backend_name
    )

    try:
        if normalized == "masac":
            return train_masac_experiment(
                env,
                total_steps=total_steps,
                seed=seed,
                run_dir=run_dir,
                run_name=run_name,
                resume_checkpoint=(
                    resume_checkpoint
                ),
                device=device,
                enable_csv=enable_csv,
                enable_tensorboard=(
                    enable_tensorboard
                ),
                enable_wandb=(
                    enable_wandb
                ),
            )

        return train_matd3_experiment(
            env,
            total_steps=total_steps,
            seed=seed,
            run_dir=run_dir,
            run_name=run_name,
            resume_checkpoint=(
                resume_checkpoint
            ),
            device=device,
            enable_csv=enable_csv,
            enable_tensorboard=(
                enable_tensorboard
            ),
            enable_wandb=(
                enable_wandb
            ),
        )
    finally:
        env.close()


In [ ]:
DEFAULT_WANDB_COMPARISON_METRICS = (
    "evaluation/success_rate",
    "evaluation/delivery_rate",
    "evaluation/confirmation_rate",
    "evaluation/return",
    "evaluation/total_energy_j",
    "evaluation/false_confirmations",
    "evaluation/boundary_clip_rate",
    "evaluation/network_pdr_percent",
    "evaluation/network_e2e_delay_ms",
    "evaluation/network_throughput_kbps",
)


def _select_latest_wandb_training_runs(
    seed,
    algorithms=("masac", "matd3"),
    entity=None,
    project=None,
):
    wandb = importlib.import_module(
        "wandb"
    )

    if entity is None:
        entity = CONFIG.get(
            "training_wandb_entity"
        )

    if project is None:
        project = CONFIG[
            "training_wandb_project"
        ]

    api = wandb.Api(
        timeout=30
    )
    all_runs = list(
        api.runs(
            f"{entity}/{project}",
            order="-created_at",
            per_page=200,
        )
    )

    selected = {}

    for algorithm in algorithms:
        normalized = str(
            algorithm
        ).strip().lower()

        for run in all_runs:
            if run.job_type != "training":
                continue

            config = dict(
                run.config
            )

            if str(
                config.get(
                    "monitor/algorithm",
                    "",
                )
            ).lower() != normalized:
                continue

            if int(
                config.get(
                    "monitor/seed",
                    -1,
                )
            ) != int(seed):
                continue

            if run.state != "finished":
                continue

            selected[
                normalized
            ] = run
            break

    missing = [
        str(algorithm)
        for algorithm in algorithms
        if str(
            algorithm
        ).strip().lower()
        not in selected
    ]

    if missing:
        raise RuntimeError(
            "no finished W&B training run "
            "found for: "
            + ", ".join(
                missing
            )
        )

    return selected


def log_wandb_seed_comparison_plots(
    seed=44,
    algorithms=("masac", "matd3"),
    metrics=None,
    entity=None,
    project=None,
    mode=None,
):
    wandb = importlib.import_module(
        "wandb"
    )

    seed = int(seed)

    if entity is None:
        entity = CONFIG.get(
            "training_wandb_entity"
        )

    if project is None:
        project = CONFIG[
            "training_wandb_project"
        ]

    if mode is None:
        mode = CONFIG[
            "training_wandb_mode"
        ]

    if metrics is None:
        metrics = (
            DEFAULT_WANDB_COMPARISON_METRICS
        )

    selected = (
        _select_latest_wandb_training_runs(
            seed=seed,
            algorithms=algorithms,
            entity=entity,
            project=project,
        )
    )

    analysis_name = (
        make_experiment_run_name(
            "MASAC",
            seed,
            run_suffix=(
                "comparison-"
                + secrets.token_hex(
                    3
                )
            ),
        )
        .replace(
            "MASAC",
            "COMPARE",
            1,
        )
    )

    with wandb.init(
        entity=entity,
        project=str(project),
        name=analysis_name,
        group=f"comparison-seed{seed}",
        job_type="analysis",
        tags=[
            "comparison",
            f"seed-{seed}",
            *[
                str(
                    algorithm
                ).strip().lower()
                for algorithm
                in algorithms
            ],
        ],
        config={
            "seed": seed,
            "selected_runs": {
                algorithm: {
                    "id": run.id,
                    "name": run.name,
                    "url": run.url,
                }
                for algorithm, run
                in selected.items()
            },
            **git_experiment_metadata(),
        },
        mode=str(mode),
        reinit="finish_previous",
    ) as analysis_run:
        logged_metrics = []

        for metric in metrics:
            rows = []

            for (
                algorithm,
                source_run,
            ) in selected.items():
                history = (
                    source_run.history(
                        keys=[
                            "global_step",
                            metric,
                        ],
                        pandas=False,
                    )
                )

                for point in history:
                    step = point.get(
                        "global_step"
                    )
                    value = point.get(
                        metric
                    )

                    if (
                        step is None
                        or value is None
                    ):
                        continue

                    if not np.isfinite(
                        float(value)
                    ):
                        continue

                    rows.append(
                        [
                            float(step),
                            float(value),
                            algorithm.upper(),
                        ]
                    )

            if not rows:
                continue

            table = wandb.Table(
                data=rows,
                columns=[
                    "global_step",
                    "value",
                    "algorithm",
                ],
            )

            key = (
                metric.replace(
                    "/",
                    "_",
                )
            )

            chart = wandb.plot.line(
                table=table,
                x="global_step",
                y="value",
                stroke="algorithm",
                title=(
                    f"{metric} | seed {seed}"
                ),
                split_table=True,
            )

            analysis_run.log(
                {
                    f"comparison/{key}": (
                        chart
                    )
                }
            )
            logged_metrics.append(
                metric
            )

        return {
            "run_id": (
                analysis_run.id
            ),
            "run_url": (
                analysis_run.url
            ),
            "run_name": (
                analysis_run.name
            ),
            "seed": seed,
            "source_runs": {
                algorithm: {
                    "id": run.id,
                    "name": run.name,
                    "url": run.url,
                }
                for algorithm, run
                in selected.items()
            },
            "logged_metrics": (
                logged_metrics
            ),
        }
